<a href="https://colab.research.google.com/github/paolazcastillo/Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System/blob/main/EDA_SINGLE_SESSION_CATEGORIZATION_OF_LENGTHS_IN_THE_RHESUS_MONKEY_(MACACA_MULATTA)_WITH_A_THREE_CATEGORY_SYSTEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Exploratory Data Analysis: single session**

# Categorization of Lengths in the Rhesus Monkey (Macaca Mulatta) with a Three-Category System

Paola Castillo

This notebook analyses **one session** end to end: signal processing and data
quality, feature engineering, PCA and clustering, psychometric and chronometric
functions, sequential effects, error taxonomy, changes of mind and supervised
decoding. Pick the session in 0.1 (`select_session`) and run section 1 top to
bottom.

Pooling several sessions (learning curves, mixed-effects models, the RNN-ready
dataset) lives in the companion notebook
`EDA_MULTI_SESSION_CATEGORIZATION_OF_LENGTHS_IN_THE_RHESUS_MONKEY_(MACACA_MULATTA)_WITH_A_THREE_CATEGORY_SYSTEM.ipynb`.


## 0. GitHub Repository Connection

In [ ]:
# Remove any clone left by an earlier run of this runtime. Without this,
# `git clone` below fails with "destination path already exists" and the
# notebook silently keeps reading the OLD data.
!rm -rf "/content/Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System"

In [ ]:
!git clone https://github.com/paolazcastillo/Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System.git

Cloning into 'Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 222 (delta 108), reused 131 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (222/222), 411.91 KiB | 2.92 MiB/s, done.
Resolving deltas: 100% (108/108), done.


In [ ]:
!ls "/content/Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System"

centerTask_v8.17  fonts  README.md


In [ ]:
from google.colab import files

_uploaded = files.upload()
_names = list(_uploaded.keys())

def _pick(prefix, exts=(".csv",), exclude=None):
    return next((n for n in _names
                if n.startswith(prefix) and n.lower().endswith(exts)
                and (exclude is None or not n.startswith(exclude))), None)

TRAJ_PATH           = _pick("trajectory_movement")
TRAJ_FULL_PATH      = _pick("trajectory_", exclude="trajectory_movement")
TRIAL_DATA_PATH     = _pick("trial_data")
KINEMATICS_PATH     = _pick("trial_kinematics")
FOIL_EVENTS_PATH    = _pick("foil_events")
SESSION_DATA_PATH   = _pick("session_data")
SESSION_REPORT_PATH = _pick("session_report", exts=(".txt",))
PARAMS_SESS_PATH    = _pick("params_sess", exts=(".mat",))
GEOMETRY_SPEC_PATH  = PARAMS_SESS_PATH or _pick("geometry_spec", exts=(".json", ".csv"))

# load_session_params (section 0.1) reads orgParams -- geometry, category
# colours and input source -- straight out of params_sess*.mat. If that
# section hasn't run yet in this Colab runtime, keep going with defaults
# rather than failing the upload.
_load_params_fn = globals().get("load_session_params")
if callable(_load_params_fn):
    SESSION_PARAMS = _load_params_fn(PARAMS_SESS_PATH)
else:
    SESSION_PARAMS = {}
    if PARAMS_SESS_PATH:
        print(f"Uploaded '{PARAMS_SESS_PATH}' but load_session_params() isn't defined yet "
              "(it lives in section 0.1); geometry, category colours and input source will "
              "use module defaults until that cell has run.")

_labeled = [
    ("trajectory_movement", TRAJ_PATH),
    ("trajectory (full)", TRAJ_FULL_PATH),
    ("trial_data", TRIAL_DATA_PATH),
    ("trial_kinematics", KINEMATICS_PATH),
    ("foil_events", FOIL_EVENTS_PATH),
    ("session_data", SESSION_DATA_PATH),
    ("session_report", SESSION_REPORT_PATH),
    ("params_sess", PARAMS_SESS_PATH),
    ("geometry_spec", GEOMETRY_SPEC_PATH),
]


## 0.1. Loading sessions from `outputs/`

The task writes one folder per session under `outputs/`, named after the
runTag it stamps everything with
(`sessROM_<dd-mmm-yyyy>_<HH-MM>`, e.g. `sessROM_26-Aug-2026_12-28`). Each
folder holds that session's `trial_data_*.csv`, both trajectory exports,
`session_data_*.csv` and the printed `session_report_*.txt`.

This cell reads that tree directly, so the manual upload above is only needed
for a file that does not live there.

- **Filtering by date.** Set `SESSION_START_DATE` and `SESSION_END_DATE` to
  anything pandas can parse (`'2026-08-21'`, `'21-Aug-2026'`, a datetime) or
  leave them `None` for no bound. Both ends are inclusive, and a bare end
  date covers that whole day, so passing the same value twice selects one
  day's sessions. The timestamp is parsed from the runTag rather than from
  the `Date` column inside the CSV, which is what lets two sessions recorded
  on the same day still be ordered.

- **What it sets.** `SESSION_INVENTORY` is the table of sessions in the
  window with a path per file kind (`None` where a session predates that
  export). Pooling several sessions is the job of the companion
  multi-session notebook, which reads the same `outputs/` tree.

- **Invalid sessions.** `INVALID_SESSIONS` lists sessions whose timing is not
  usable (the pre-v8.21 fixed-anchor clock made the RZ2 timestamps drift
  against the task clock, so decision windows shrank during the session).
  With `EXCLUDE_INVALID_SESSIONS = True` they are dropped from the inventory
  and the reason is printed; set it to `False` to keep them, flagged in the
  `InvalidReason` column. Entries match a runTag exactly or as a substring;
  an entry that matches no folder is reported, so a short name can be
  replaced by the full runTag.

- **Choosing one session.** `select_session(SESSION_INVENTORY, index)` sets
  `TRIAL_DATA_PATH`, `TRAJ_PATH` and the rest, which is what every
  per-session cell below reads; the default takes the most recent session in
  the window. Switch sessions with one call and re-run section 1.

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUTS_DIR = "outputs"
OUTPUTS_GLOBS = ["outputs", "/content/*/outputs", "../outputs"]
SESSION_START_DATE = None
SESSION_END_DATE = None
RUNTAG_DATE_RE = re.compile(r"_(\d{2}-[A-Za-z]{3}-\d{4})_(\d{2})-(\d{2})$")
SESSION_FILE_KINDS = {
    "trial_data": "trial_data_{tag}.csv",
    "trajectory": "trajectory_{tag}.csv",
    "trajectory_movement": "trajectory_movement_{tag}.csv",
    "session_data": "session_data_{tag}.csv",
    "session_report": "session_report_{tag}.txt",
    "trial_kinematics": "trial_kinematics_{tag}.csv",
    "foil_events": "foil_events_{tag}.csv",
    "params_sess": "params_{tag}.mat",
}

INVALID_SESSIONS = {
    "sessROM_31-Aug-2026_14-20": "pre-v8.21 fixed-anchor clock: RZ2 stamps drifted ~13.6 ms/s "
                                 "from GetSecs, so sample-anchored windows shrank",
    "sessPX-309": "pre-v8.21 fixed-anchor clock (03-Sep): 13.7 ms/s drift, windows had "
                  "already expired from trial 27 on",
}
EXCLUDE_INVALID_SESSIONS = True


def invalid_reason(run_tag, invalid=None):
    invalid = INVALID_SESSIONS if invalid is None else invalid
    tag = str(run_tag)
    for key, why in invalid.items():
        if key == tag or key in tag:
            return why
    return None


def _normalize(td):
    """normalize_trial_schema if the analysis cells have already run, else the
    same canonicalisation inline.

    Duplicated deliberately: this cell is meant to be runnable FIRST, before
    the cells that define the real one, so that the paths it sets are ready
    for them. Keeping the fallback means loading never depends on execution
    order.
    """
    fn = globals().get("normalize_trial_schema")
    if callable(fn):
        return fn(td)
    td = td.copy()
    if "ExecutionTime_s" not in td.columns and "ReactionTime_s" in td.columns:
        td["ExecutionTime_s"] = td["ReactionTime_s"]
        td = td.drop(columns=["ReactionTime_s"])
    if "TotalTime_s" not in td.columns:
        if "ReactionTime_s" in td.columns:
            td["TotalTime_s"] = td["ReactionTime_s"]
        elif {"DecisionTime_s", "ExecutionTime_s"} <= set(td.columns):
            td["TotalTime_s"] = td["DecisionTime_s"] + td["ExecutionTime_s"]
    if "ReactionTime_s" in td.columns:
        td = td.drop(columns=["ReactionTime_s"])
    return td


def resolve_outputs_dir(candidates=OUTPUTS_GLOBS):
    """Find the outputs/ tree the task writes its sessions into.

    Checked in order so the same notebook works from a local checkout and
    from Colab, where the repo lands under /content/<repo>/. Returns None
    rather than raising, so the single-session cells above can still be run
    against a hand-picked file.
    """
    import glob as _g
    for cand in candidates:
        for hit in sorted(_g.glob(cand)):
            if Path(hit).is_dir():
                return Path(hit)
    return None


def session_timestamp(run_tag):
    """Session start time parsed out of the runTag.

    CenterOutTask.m builds it as datestr(now, 'dd-mmm-yyyy_HH-MM'), so
    'sessROM_26-Aug-2026_12-28' carries the real start instant. Parsing the
    tag rather than the Date column inside the CSV is what lets two sessions
    recorded on the SAME day still be ordered, and what makes the date filter
    below work without opening a single file.
    """
    m = RUNTAG_DATE_RE.search(str(run_tag))
    if not m:
        return pd.NaT
    return pd.to_datetime(f"{m.group(1)} {m.group(2)}:{m.group(3)}",
                          format="%d-%b-%Y %H:%M", errors="coerce")


def load_session_params(path):
    """Read one session's params_sess*.mat (orgParams struct) into a plain dict.

    This is the ONE authoritative source for three things every per-session
    cell below used to either guess at or hardcode:
      - which physical device logged the trajectory: orgParams.inputSource
        ('rz2adc' | 'joystick' | 'mouse' -- RZ2 analog rig vs. a standard USB
        joystick/mouse);
      - the screen geometry the task actually rendered that day:
        orgParams.centerRad / targetRad / centerToTargetDist / targetRadius /
        screenViewingDist_mm / screenPixelPitch;
      - the real on-screen category colours: orgParams.color3Cat* /
        color2Cat*, the same hex values ColorCategoryMap.m paints with.
    Returns {} (never None) on any failure, so callers can do
    SESSION_PARAMS.get(...) and quietly fall back to their own module
    defaults instead of branching on None everywhere.
    """
    if not path:
        return {}
    try:
        from scipy.io import loadmat
    except ImportError:
        print(f"WARNING: scipy is not installed (pip install scipy); cannot read '{path}'. "
              "Geometry, category colours and input source will fall back to module defaults.")
        return {}
    try:
        mat = loadmat(path, squeeze_me=True, struct_as_record=False)
        op = mat["orgParams"]
    except Exception as exc:
        print(f"WARNING: could not read orgParams from '{path}' ({exc}); falling back to defaults.")
        return {}

    def _get(name):
        v = getattr(op, name, None)
        if isinstance(v, np.ndarray) and v.size == 0:
            return None
        return v

    raw_input_source = _get("inputSource")
    return {
        "source": str(path),
        "input_source": str(raw_input_source).strip().lower() if raw_input_source is not None else None,
        "center_diameter_px": _get("centerRad"),
        "target_diameter_px": _get("targetRad"),
        "center_to_target_dist_px": _get("centerToTargetDist"),
        "target_reach_radius_px": _get("targetRadius"),
        "screen_viewing_dist_mm": _get("screenViewingDist_mm"),
        "screen_pixel_pitch_mm": _get("screenPixelPitch"),
        "color3cat_hex": {
            "ShortGroup": _get("color3CatShort"),
            "MidGroup": _get("color3CatMid"),
            "LongGroup": _get("color3CatLong"),
        },
        "color2cat_hex": {
            "ShortGroup": _get("color2CatShort"),
            "LongGroup": _get("color2CatLong"),
        },
    }


def discover_sessions(outputs_dir=None, start_date=None, end_date=None, verbose=True,
                      exclude_invalid=None):
    """Inventory of every session folder under outputs/, filtered by date.

    One row per session, one column per file kind, holding the path when that
    file exists and None when it does not -- older sessions carry no
    foil_events or trial_kinematics, and nothing here should have to care.

    start_date / end_date are inclusive and accept anything pandas can parse
    ('2026-08-21', '21-Aug-2026', a datetime). end_date given as a bare date
    covers that whole day, so passing the same value for both selects one
    day's sessions rather than only those recorded at exactly midnight.
    """
    outputs_dir = Path(outputs_dir) if outputs_dir else resolve_outputs_dir()
    if outputs_dir is None or not outputs_dir.exists():
        print(f"No outputs/ directory found (looked in {OUTPUTS_GLOBS}). "
              f"Set OUTPUTS_DIR to its path and re-run.")
        return pd.DataFrame()
    rows = []
    for d in sorted(p for p in outputs_dir.iterdir() if p.is_dir()):
        tag = d.name
        rec = {"Session": tag, "SessionStart": session_timestamp(tag), "folder": str(d)}
        for kind, pattern in SESSION_FILE_KINDS.items():
            f = d / pattern.format(tag=tag)
            rec[kind] = str(f) if f.exists() else None
        if rec["trial_data"] is None:
            hits = sorted(d.glob("trial_data_*.csv"))
            rec["trial_data"] = str(hits[0]) if hits else None
        rows.append(rec)
    inv = pd.DataFrame(rows)
    if inv.empty:
        print(f"{outputs_dir} has no session sub-folders.")
        return inv
    n_all = len(inv)
    all_tags = list(inv["Session"])
    exclude_invalid = EXCLUDE_INVALID_SESSIONS if exclude_invalid is None else exclude_invalid
    if start_date is not None:
        lo = pd.to_datetime(start_date)
        inv = inv[inv["SessionStart"].notna() & (inv["SessionStart"] >= lo)]
    if end_date is not None:
        hi = pd.to_datetime(end_date)
        if hi == hi.normalize():
            hi = hi + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
        inv = inv[inv["SessionStart"].notna() & (inv["SessionStart"] <= hi)]
    inv["InvalidReason"] = inv["Session"].map(invalid_reason)
    flagged = inv[inv["InvalidReason"].notna()]
    if exclude_invalid:
        inv = inv[inv["InvalidReason"].isna()]
    inv = inv.sort_values("SessionStart").reset_index(drop=True)
    inv.insert(0, "index", np.arange(1, len(inv) + 1))
    if verbose:
        window = ""
        if start_date is not None or end_date is not None:
            window = f"  [filter: {start_date or 'earliest'} .. {end_date or 'latest'}]"
        print(f"{outputs_dir}: {len(inv)}/{n_all} sessions{window}\n")
        show = inv[["index", "Session", "SessionStart"]].copy()
        show["files"] = [sum(r[k] is not None for k in SESSION_FILE_KINDS)
                         for _, r in inv.iterrows()]
        print(show.to_string(index=False))
        missing = [k for k in SESSION_FILE_KINDS if inv[k].isna().all()]
        if missing:
            print(f"\n  No session in this window has: {missing} "
                  f"(expected for sessions recorded before those exports existed).")
        if len(flagged):
            state = "excluded" if exclude_invalid else "kept, see the InvalidReason column"
            print(f"\nInvalid sessions in this window ({state}):")
            for _, fr in flagged.iterrows():
                print(f"  {fr['Session']}: {fr['InvalidReason']}")
        unmatched = [k for k in INVALID_SESSIONS if not any(k == s or k in s for s in all_tags)]
        if unmatched:
            print(f"\n  INVALID_SESSIONS entries that match no folder under {outputs_dir}: {unmatched}. "
                  f"If an entry is a short name, replace it with the session's full runTag.")
    return inv


def select_session(inv, index=-1):
    """Point the single-session cells at one row of the inventory.

    index is 1-based to match the printed table; -1 (the default) takes the
    most recent session in the window. Sets TRIAL_DATA_PATH and friends, which
    is what every per-session cell above reads, so switching sessions is one
    call and a re-run rather than an edit in several places.

    Also loads PARAMS_SESS_PATH's orgParams into SESSION_PARAMS (see
    load_session_params): the per-session cells below read that global to get
    the real screen geometry, category colours and input source for THIS
    session instead of the module's fallback defaults. GEOMETRY_SPEC_PATH is
    kept as an alias to the same file for any older cell that still checks it.
    """
    if inv is None or inv.empty:
        print("Empty inventory; nothing to select.")
        return None
    row = inv.iloc[-1] if index == -1 else inv[inv["index"] == index].iloc[0]
    g = globals()
    g["TRIAL_DATA_PATH"] = row["trial_data"]
    g["TRAJ_FULL_PATH"] = row["trajectory"]
    g["TRAJ_PATH"] = row["trajectory_movement"] or row["trajectory"]
    g["KINEMATICS_PATH"] = row["trial_kinematics"]
    g["FOIL_EVENTS_PATH"] = row["foil_events"]
    g["SESSION_DATA_PATH"] = row["session_data"]
    g["SESSION_REPORT_PATH"] = row["session_report"]
    g["PARAMS_SESS_PATH"] = row["params_sess"]
    g["GEOMETRY_SPEC_PATH"] = row["params_sess"]
    g["SESSION_PARAMS"] = load_session_params(row["params_sess"])
    print(f"Selected session {row['index']}: {row['Session']}  ({row['SessionStart']})")
    if invalid_reason(row["Session"]):
        print(f"WARNING: this session is listed in INVALID_SESSIONS: {invalid_reason(row['Session'])}")
    for k in ("TRIAL_DATA_PATH", "TRAJ_PATH", "TRAJ_FULL_PATH", "FOIL_EVENTS_PATH", "PARAMS_SESS_PATH"):
        print(f"  {k:18s} {g[k]}")
    sp = g["SESSION_PARAMS"]
    if sp:
        print(f"  Session params from {Path(sp['source']).name}: "
              f"inputSource={sp['input_source']!r}, "
              f"centerToTargetDist={sp['center_to_target_dist_px']} px, "
              f"color3Cat={sp['color3cat_hex']}")
    else:
        print("  No params_sess*.mat found for this session: geometry, category colours "
              "and input source fall back to module defaults.")
    print("\nRe-run the per-session cells below (1.1 onward) to analyse it.")
    return row


REQUIRED_TRIAL_FIELDS = ["NumCategories", "SessionMode"]


def check_session_fields(inv, fields=REQUIRED_TRIAL_FIELDS):
    """Warn when a trial_data file lacks a field the analyses read, and say
    WHICH file was read: a NaN in NumCategories / SessionMode for a session
    that has them in the repository means the notebook is reading a stale
    copy (typically a Colab runtime whose old clone was not removed)."""
    if inv is None or inv.empty:
        return
    missing = []
    for _, r in inv.iterrows():
        if not r.get("trial_data"):
            continue
        with open(r["trial_data"]) as fh:
            header = fh.readline().strip().split(",")
        lack = [f for f in fields if f not in header]
        if lack:
            missing.append((r["Session"], lack, r["trial_data"]))
    if missing:
        print(f"\nWARNING: {len(missing)} session file(s) lack {fields}:")
        for tag, lack, path in missing:
            print(f"  {tag}: missing {lack}  <- {path}")
        print("  If the repository has these columns, the files being read are stale: "
              "re-run the clone cell in section 0 (it now removes the old clone) "
              "or pull the latest outputs/.")


SESSION_INVENTORY = discover_sessions(OUTPUTS_DIR if Path(OUTPUTS_DIR).exists() else None,
                                      SESSION_START_DATE, SESSION_END_DATE)
check_session_fields(SESSION_INVENTORY)
if not SESSION_INVENTORY.empty:
    select_session(SESSION_INVENTORY)


### 0.2. Movement-onset threshold

`move_takeoff_ms` (used throughout sections 1.1-1.10) is defined **retrospectively**: within each trial's search window the peak speed is located first, and only then is the trace walked *backward* from that peak to the first sample still at or above a fraction of it (see `_takeoff_index` in the signal-processing cells). Two fractions are kept for every trial: `MOVE_SPEED_FRAC` (primary, used everywhere downstream) and `MOVE_SPEED_FRAC_ALT` (drawn only as a secondary reference line in the trajectory figures).

Adjust either one below -- both per-session trajectory cells (1.1's hold + final-place variant and the entry-only trajectory-EDA variant) and every downstream cell that reads `move_takeoff_ms` pick these two globals up instead of a hardcoded literal.

In [ ]:
MOVE_SPEED_FRAC = 0.05       # primary threshold: 5% of the trial's peak speed
MOVE_SPEED_FRAC_ALT = 0.10   # secondary threshold: 10% of the trial's peak speed (reference line only)

print(f"Movement-onset threshold: {MOVE_SPEED_FRAC:.0%} of peak speed "
     f"(alt reference: {MOVE_SPEED_FRAC_ALT:.0%}).")


## Trajectory

Category-agnostic exploratory analysis: the per-trial figures are driven entirely by the CSV columns (`DirectionCorrect` / `IsCorrect` / `ErrorType` / `BarSizeVA_deg`) and make no assumption about how many categories the session used, so the same script handles a 2-category (Short/Long), 3-category (Short/Mid/Long), or any other configuration. The screen-path panel draws all four cardinal target positions as fixed reference circles regardless of how many were actually shown.

`TRAJ_PATH` and `TRIAL_DATA_PATH` (and optionally `KINEMATICS_PATH`) are expected to be defined by the upload cell run before this one. The analysis cell itself does no file input.

---

## Signal processing pipeline

The continuous joystick coordinates $(x_i, y_i)$, sampled at irregular timestamps $t_i$, pass through five stages before any kinematic quantity is computed. The order is deliberate and is stated first, because each stage assumes the previous one already ran:

$$
\text{raw }(t_i, x_i, y_i)
\;\xrightarrow{\text{1. Hampel}}\;
\;\xrightarrow{\text{2. Kalman + RTS}}\;
\;\xrightarrow{\text{3. PCHIP}}\;
\;\xrightarrow{\text{4. Butterworth}}\;
\;\xrightarrow{\text{5. Savitzky-Golay}}\;
\text{kinematics}
$$

Each axis $x$ and $y$ is processed independently unless noted.

---

### 1. Hampel filter (impulsive outlier removal)

**What it is.** For each sample $i$ define a sliding window of half-width $h=3$,

$$
W_i = \{\, x_j : |j - i| \le h \,\}.
$$

Compute the local median and the (scaled) median absolute deviation:

$$
m_i = \operatorname{median}(W_i),
\qquad
\mathrm{MAD}_i = 1.4826 \cdot \operatorname{median}\big(\, |x_j - m_i| : x_j \in W_i \,\big).
$$

Replace the sample only when it lies beyond $n_\sigma = 3$ scaled deviations:

$$
\hat{x}_i =
\begin{cases}
m_i & \text{if } |x_i - m_i| > n_\sigma \, \mathrm{MAD}_i \\[4pt]
x_i & \text{otherwise.}
\end{cases}
$$

**Why the $1.4826$.** For Gaussian data, $\mathbb{E}[\mathrm{MAD}] = \sigma / 1.4826$, so multiplying by $1.4826$ turns the MAD into a consistent estimator of the standard deviation $\sigma$. The threshold $n_\sigma \mathrm{MAD}_i$ is then read in the same units as a $3\sigma$ rule, but robustly.

**Why it is here, and first.** Joystick spikes (single-sample dropouts, quantization glitches) are impulsive: a handful of gross errors. The median and MAD have a breakdown point of $50\%$, so a few spikes do not corrupt the estimate the way a mean and standard deviation would. It runs first because a single un-removed spike would otherwise be treated by the Kalman filter as a real (very high acceleration) measurement and would bleed into the smoothed state.

---

### 2. Kalman filter with constant-jerk model + RTS smoother

**State.** Per axis, the latent state is position, velocity, acceleration:

$$
\mathbf{s}_k = \begin{bmatrix} p_k \\ v_k \\ a_k \end{bmatrix}.
$$

**Dynamics (constant-jerk / white-noise-jerk model).** Jerk (the derivative of acceleration) is modeled as a continuous white-noise process of power spectral density $q = \sigma_\text{jerk}^2$. Over a step $\Delta t_k = t_k - t_{k-1}$ the exact discretization is

$$
\mathbf{s}_k = F_k\,\mathbf{s}_{k-1} + \mathbf{w}_k,
\qquad
F_k = \begin{bmatrix} 1 & \Delta t & \tfrac{\Delta t^2}{2} \\ 0 & 1 & \Delta t \\ 0 & 0 & 1 \end{bmatrix},
$$

$$
Q_k = q \begin{bmatrix}
\tfrac{\Delta t^5}{20} & \tfrac{\Delta t^4}{8} & \tfrac{\Delta t^3}{6} \\[4pt]
\tfrac{\Delta t^4}{8} & \tfrac{\Delta t^3}{3} & \tfrac{\Delta t^2}{2} \\[4pt]
\tfrac{\Delta t^3}{6} & \tfrac{\Delta t^2}{2} & \Delta t
\end{bmatrix}.
$$

**Measurement.** Only position is observed, with variance $R = \sigma_\text{meas}^2$ (here $\sigma_\text{meas} = 2$ px):

$$
z_k = H\,\mathbf{s}_k + \nu_k,
\qquad H = \begin{bmatrix} 1 & 0 & 0 \end{bmatrix},
\qquad \nu_k \sim \mathcal{N}(0, R).
$$

**Forward pass (predict / update).**

$$
\hat{\mathbf{s}}_k^- = F_k \hat{\mathbf{s}}_{k-1},
\qquad
P_k^- = F_k P_{k-1} F_k^\top + Q_k,
$$

$$
y_k = z_k - H\hat{\mathbf{s}}_k^-,
\qquad
S_k = H P_k^- H^\top + R,
\qquad
K_k = \frac{P_k^- H^\top}{S_k},
$$

$$
\hat{\mathbf{s}}_k = \hat{\mathbf{s}}_k^- + K_k\,y_k,
\qquad
P_k = (I - K_k H)\,P_k^-.
$$

An optional $\chi^2$ gate accepts the update only when $y_k^2 \le \gamma^2 S_k$ (parameter `gate_sigma`; default $\gamma = \infty$, i.e. no gating).

**Backward pass (Rauch-Tung-Striebel smoother).** After filtering forward, a backward recursion refines every estimate using future data:

$$
C_k = P_k F_{k+1}^\top \big(P_{k+1}^-\big)^{-1},
\qquad
\hat{\mathbf{s}}_k^s = \hat{\mathbf{s}}_k + C_k\big(\hat{\mathbf{s}}_{k+1}^s - \hat{\mathbf{s}}_{k+1}^-\big).
$$

The smoothed position $\hat{p}_k^s = \hat{\mathbf{s}}_k^s[0]$ is the output.

**Why it is here.** Unlike a fixed-frequency filter, the Kalman filter uses a physical model of the movement (position, velocity, and acceleration are coupled through $F_k$) to denoise while respecting kinematic smoothness. Two properties matter for this data specifically: it handles the non-uniform $\Delta t_k$ of the raw timestamps exactly (each step uses its own $\Delta t$), and the RTS pass is non-causal, so it uses the whole trajectory to produce the best estimate at each instant rather than lagging behind like a causal filter.

---

### 3. PCHIP resampling to a uniform grid

**What it is.** The smoothed, still irregularly-timed positions are interpolated onto a uniform grid $t_g = t_0 + m\,\Delta t$ with $\Delta t = 8$ ms (125 Hz) using a Piecewise Cubic Hermite Interpolating Polynomial. On each interval $[t_i, t_{i+1}]$ PCHIP fits a cubic $P(t)$ matching the endpoint values, with endpoint slopes chosen by the Fritsch-Carlson rule so that $P$ is monotone wherever the data are monotone:

$$
P(t_i) = \hat{p}_i^s,
\quad P(t_{i+1}) = \hat{p}_{i+1}^s,
\quad \text{slopes set to avoid overshoot between samples.}
$$

**Why it is here.** The next two stages (the fixed-$\Delta t$ Butterworth filter and the Savitzky-Golay derivative) both assume uniform spacing. PCHIP is chosen over a natural cubic spline because a spline can overshoot and ring near sharp movement onsets, inventing oscillations that are not in the data. PCHIP's monotonicity constraint prevents that, at the cost of a discontinuous second derivative (acceptable, because acceleration is recomputed later by a separate smoothing differentiator).

---

### 4. Butterworth low-pass filter (zero-phase)

**What it is.** A second-order ($N=2$) Butterworth low-pass, whose analog prototype has a maximally-flat passband:

$$
|H(j\omega)|^2 = \frac{1}{1 + \left(\dfrac{\omega}{\omega_c}\right)^{2N}}.
$$

The design cutoff is $f_c = 6$ Hz (`CUTOFF_HZ`). The digital coefficients come from the bilinear transform with frequency prewarping through $k = \tan\!\big(\pi f_c / f_s\big)$. The filter is applied forward and then backward over the signal (a `filtfilt`-style pass), which squares the magnitude response and cancels phase:

$$
|H_\text{eff}(j\omega)|^2 = |H(j\omega)|^2 \cdot |H(j\omega)|^2,
\qquad
\angle H_\text{eff} = 0.
$$

Reflection padding at both ends and steady-state initial conditions suppress edge transients.

**Why the effective cutoff is 4.8 Hz and not 6 Hz.** $f_c$ is the half-power ($-3$ dB) point of a *single* pass: $|H(j\omega_c)|^2 = 1/2$. Running the filter forward and then backward multiplies the response by itself, so the half-power point of the combined filter is where the *product* of the two passes equals $1/2$. With $N = 2$:

$$
|H_\text{eff}(j\omega)|^2 = \frac{1}{\left(1 + (\omega/\omega_c)^{4}\right)^{2}} = \frac{1}{2}
\iff 1 + \left(\frac{\omega}{\omega_c}\right)^{4} = \sqrt{2}
\iff \frac{\omega}{\omega_c} = \left(\sqrt{2} - 1\right)^{1/4} \approx 0.802 .
$$

For the analog prototype this gives $0.802 \times 6 \approx 4.81$ Hz. The digital filter meets the same ratio on the prewarped frequency axis, $\tan(\pi f / f_s) = 0.802 \tan(\pi f_c / f_s)$, which at $f_s = 125$ Hz gives $f \approx 4.83$ Hz. At the nominal 6 Hz each pass contributes $-3$ dB, so the combined filter is already at $-6$ dB there. In short, 6 Hz is the design parameter and about 4.8 Hz is the frequency the data actually experience as the cutoff, which is the value any description of the filter should quote. The code computes it from `CUTOFF_HZ` and the grid rate as `EFFECTIVE_CUTOFF_HZ`, so the figure titles stay correct if either parameter changes. For $p$ passes of an order-$N$ Butterworth the general ratio is $\left(2^{1/p} - 1\right)^{1/(2N)}$.

**Why it is here.** Voluntary reaching movements have essentially no spectral power above a few Hz, so residual content above $\sim 6$ Hz is sensor and quantization noise. The zero-phase (forward-backward) application matters because the analysis reads timing of kinematic events (movement onset, peak velocity): an ordinary causal filter would shift those events in time, whereas a zero-phase filter attenuates the noise without moving any feature.

---

### 5. Kinematics and the Savitzky-Golay derivative

**Speed.** From the filtered uniform-grid position, speed is the first difference divided by the (constant) step, evaluated at the segment midpoint:

$$
v_k = \frac{\big\| (x_{k+1}, y_{k+1}) - (x_k, y_k) \big\|_2}{\Delta t},
\qquad
t_k^{(v)} = \frac{t_k + t_{k+1}}{2}.
$$

**Acceleration (Savitzky-Golay first derivative).** Rather than differencing $v_k$ again, the tangential acceleration is estimated by fitting, in a sliding window of $w = 11$ samples ($\approx 88$ ms), a degree $p = 3$ polynomial by least squares and taking the analytic derivative of that local fit at the window center:

$$
a_k = \left| \frac{d}{dt}\,\widehat{v}^{\,(p)}(t) \right|_{t = t_k},
\qquad w = 11,\; p = 3.
$$

This is a convolution with fixed differentiating coefficients, so it is a smoothing differentiator.

**Why a smoothing differentiator.** Differentiation amplifies high-frequency noise: a bare finite difference multiplies the noise spectrum by $\omega^2$, so a single-sample jitter dominates the result. The local least-squares polynomial fit suppresses that high-frequency content before differentiating, so the peak and mean acceleration reflect the movement rather than the noise. Reported statistics use robust summaries (a high percentile for the peak, so one spurious spike does not set the value) consistent with this goal.

**Summary window.** Every per-trial summary is taken over the same samples, from the movement takeoff to the moment the cursor enters the target. The speed peak is searched from the start of the loaded epochs up to the last MOVEMENT sample (target entry), so a jitter in the target hold can never be taken as the peak. Walking back from that peak, `move_takeoff_ms` is the first sample still above `MOVE_SPEED_FRAC` (5 %) of it; `move_takeoff_alt_ms` repeats the walk at `MOVE_SPEED_FRAC_ALT` (10 %) and is only drawn in the figures. Peak, mean and median speed, and peak (99th percentile), mean and median acceleration are then computed over [5 % takeoff, target entry]. Trials without a MOVEMENT epoch (early exits) fall back to the end of the loaded epochs. Because the window no longer includes the rest in the centre or the hold, the same trial gives the same summaries in the 1.1 cell (DECISION, MOVEMENT, TARGET_HOLD loaded), the trajectory-EDA variant and the feature table (DECISION, MOVEMENT loaded).

**Takeoff lead.** `takeoff_lead_ms = move_onset_ms - move_takeoff_ms` is the time the hand was already moving when it crossed the centre circle. Both times come from rows of the same trajectory, and the first MOVEMENT row is the row the task used for `t.leaveCenter`, so transport latency cancels in the difference. `TakeoffTime_s = DecisionTime_s - takeoff_lead_ms / 1000` places the takeoff on the origin (go) and the clock of `DecisionTime_s`.

**Duplicate and unindexed rows.** On sessions detected as rz2adc, rows with NaN `RZ2Idx` (the cached rows a frame writes when no new sample arrived, and any un-indexed legacy samples) are removed when the file is read. Before any stage runs, samples are then sorted by time with a stable sort and repeated timestamps are collapsed to their first occurrence. Both counts are printed per session (`n_duplicate_ts` is also kept per trial).

---

## Data quality flags

Beyond the filters, each trial is screened with three flags: **under-resolved** (too few movement samples to differentiate reliably), **hold unstable** (the cursor's maximum excursion during the target-hold exceeds the target radius), and **hold incomplete** (the hold epoch is shorter than $80\%$ of the session-median hold, i.e. the subject left before satisfying the minimum hold time). These do not alter the signal; they mark trials whose kinematics should be read with caution.

## 1. Per session analysis

### 1.1. Signal Processing and Data Quality

- Assess the effectiveness of the multi-stage cleaning pipeline applied to
  the continuous joystick coordinates (columns `X_px`, `Y_px` of
  `trajectory_movement_<runTag>.csv`). The stages run in this order:
  Hampel outlier screening, constant-jerk Kalman smoothing, PCHIP
  resampling onto a uniform 8 ms grid (125 Hz), and a zero-phase
  Butterworth low-pass (design cutoff 6 Hz, effective $-3$ dB point 4.8 Hz
  because of the forward-backward application; the derivation is in the
  Trajectory section above). This differs from a single low-pass step: the
  Butterworth is only the final stage.

- Compare the raw per-sample signal against the cleaned output. The
  per-trial figure overlays, on a shared time axis, the raw trace, the
  Hampel median/MAD band with the flagged outliers, and the fully filtered
  trace, so the effect of each stage is visible directly.

- Evaluate each method against its role:
  - `HampelScreen`: temporal anomaly detection per axis, using a sliding
    median and a MAD-based threshold (scale 1.4826, half-window 3,
    n_sigma 3), replacing flagged spikes with the local median.
  - `KalmanTrajectorySmoother`: a 3-state (position, velocity,
    acceleration) constant-jerk Kalman filter followed by an RTS backward
    smoother, applied independently to X and Y.
  - `ButterworthLowpass`: a second-order low-pass designed at 6 Hz, applied
    forward and backward (zero phase, effective $-3$ dB at 4.8 Hz) with
    reflection padding and steady-state initialization.

- Quantify the kinematics the cleaning enables: instantaneous speed
  |d(x,y)/dt| (cm/s) and tangential acceleration |d|v|/dt| (cm/s^2),
  where the acceleration is estimated with a Savitzky-Golay first
  derivative (11-sample window, about 88 ms, order 3) rather than a raw
  finite difference, so single-sample jitter does not dominate the peak
  and mean.

- Screen data quality with the per-trial flags the cell already computes:
  under-resolved trials (too few samples to differentiate), hold
  instability (`hold_excursion_px` beyond the target radius), and
  incomplete holds (hold duration below 80 percent of the session median).

- Summaries (peak, mean and median of speed and acceleration) are taken from
  the 5 % takeoff to target entry; the figures draw the takeoff at 5 % and at
  10 % of peak speed. See "Summary window" and "Takeoff lead" in the
  Trajectory section.

- Input source. Each trajectory file is classified when it is read:
  `rz2adc` if `RZ2Idx` has indexed rows, otherwise by the median step between
  samples within a trial (below `RZ2_MAX_MEDIAN_STEP_MS` = 3 ms is rz2adc,
  about 8 ms is the USB joystick; mouse sessions also land in `usb`).
  `INPUT_SOURCE_OVERRIDES` (runTag, or part of it, to source) overrides the
  rule. The source picks a `PipelineProfile` from `PIPELINE_PROFILES`: Hampel
  window and threshold, Kalman measurement and jerk noise, and whether rows
  with NaN `RZ2Idx` are dropped (rz2adc only; on USB every row has NaN
  `RZ2Idx`). Both profiles start with the same filter values, so results
  only change once a profile is edited. The source is printed per file and
  shown in each figure title.

- Screen geometry. The hit-test geometry (centre and target diameters,
  centre-to-target distance and its scale) defaults to the constants in the
  cell. If a file named `geometry_spec*.json` or `geometry_spec*.csv` sits
  next to the session's `trial_data_*.csv` (the session folder under
  `outputs/`, or the Colab working directory after an upload), its values are
  used instead and the printed line says which file was read. Accepted
  layouts: a JSON object, a two-column `key,value` CSV, or a one-row CSV with
  the keys as headers. Keys are `CENTER_DIAMETER_PX`, `TARGET_DIAMETER_PX`,
  `CENTER_TO_TARGET_DIST_PX` and `TARGET_DIST_SCALE`; any key missing from the
  file keeps its default. `hold_unstable` uses the resulting target radius.

- Offline clock check. When the full trajectory export carries `RZ2Idx`
  (v8.21 onward) and the session is detected as rz2adc, the cell fits $\tau_i = b + a\,n_i$ by
  ordinary least squares over the indexed rows and prints the implied write
  rate $1000/a$ (and its deviation in ppm from $24414.0625/26$ Hz), the
  residuals of the online `Time_ms` about that line, and the number of
  backward steps of `Time_ms` in index order. A healthy session shows a rate
  within a few hundred ppm of the seed, small residuals and no backward
  steps. The analyses still read `Time_ms`; the fit is a check on it.

In [ ]:
import glob
import json
import zipfile
from dataclasses import dataclass, field
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.signal import lfilter, savgol_filter, find_peaks
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import MultipleLocator
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize

GRID_DT_S = 0.008
CUTOFF_HZ = 6.0
ACCEL_PEAK_PERCENTILE = 99.0
SAVGOL_WINDOW = 11
SAVGOL_POLYORDER = 3
# Movement-onset ("takeoff") threshold. Adjustable from the config cell in
# section 0.2 -- this is only the fallback default for running this cell
# standalone. move_takeoff_ms (used everywhere downstream, including the
# window search below) is retrospective: it is found by first locating the
# peak speed in the search window, then walking BACKWARD from that peak to
# the first sample still above TAKEOFF_SPEED_FRAC_CHOICE's fraction of it
# (see _takeoff_index below). Only one threshold is ever active: whichever
# of "5%" / "10%" is chosen is both the reference line drawn on the
# trajectory figures and the one used for the window -- the two are no
# longer shown or computed side by side.
TAKEOFF_SPEED_FRAC_OPTIONS = {"5%": 0.05, "10%": 0.10}
TAKEOFF_SPEED_FRAC_CHOICE = globals().get("TAKEOFF_SPEED_FRAC_CHOICE", "5%")
MOVE_SPEED_FRAC = TAKEOFF_SPEED_FRAC_OPTIONS[TAKEOFF_SPEED_FRAC_CHOICE]
# Which local speed peak anchors the takeoff walk-back, when a trial has more
# than one (e.g. a correction / change of mind -- see 1.9): "largest" (default,
# original behaviour) anchors to the tallest peak in the window, which can sit
# on a LATER bump; "first" anchors to the earliest peak that clears
# MULTI_PEAK_PROMINENCE_FRAC of the window's own max speed, so takeoff lands on
# the first real movement instead. Single-peak trials are unaffected either way.
TAKEOFF_PEAK_CHOICE = globals().get("TAKEOFF_PEAK_CHOICE", "largest")  # "largest" or "first"
MULTI_PEAK_PROMINENCE_FRAC = globals().get("MULTI_PEAK_PROMINENCE_FRAC", 0.15)
PIXEL_PITCH_MM = 0.3108
SHOW_INLINE = True
OUTPUT_DIR = "figures"
FIX_TIME_AXIS = True
FIX_POS_AXIS = True
TIME_TICK_STEP_MS = 300
MAX_TRAJ_TIME_MS = 4000.0  # cap the shared trajectory time axis at 4s so a few long outlier trials don't stretch it
AXIS_MARGIN = 0.03
SCREEN_HALF_EXTENT_PX = 750

FIX_ACCEL_SCALE = True
ACCEL_SCALE_PERCENTILE = 99.0

DECISION_EPOCH = 6
MOVEMENT_EPOCH = 7
TARGET_HOLD_EPOCH = 8
MOVE_EPOCHS = (DECISION_EPOCH, MOVEMENT_EPOCH, TARGET_HOLD_EPOCH)

_EPOCH_NAME_TO_CODE = {
    "REACTION": DECISION_EPOCH, "DECISION": DECISION_EPOCH,
    "DECISION_TIME": DECISION_EPOCH, "DECISIONTIME": DECISION_EPOCH,
    "MOVEMENT": MOVEMENT_EPOCH, "MOVE": MOVEMENT_EPOCH, "EXECUTION": MOVEMENT_EPOCH,
    "TARGET_HOLD": TARGET_HOLD_EPOCH, "TARGETHOLD": TARGET_HOLD_EPOCH, "HOLD": TARGET_HOLD_EPOCH,
}


def _epoch_to_code(v):
    """Numeric TaskEpoch code from a numeric OR string Epoch cell (NaN if unknown)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    try:
        return int(v)
    except (ValueError, TypeError):
        return _EPOCH_NAME_TO_CODE.get(str(v).strip().upper(), np.nan)

CENTER_TO_TARGET_DIST_PX = 320
TARGET_DIST_SCALE = 1.27
TARGET_DIAMETER_PX = 180
CENTER_DIAMETER_PX = 200
TARGET_DIST_PX = CENTER_TO_TARGET_DIST_PX * TARGET_DIST_SCALE
TARGET_RADIUS_PX = TARGET_DIAMETER_PX / 2.0
CENTER_RADIUS_PX = CENTER_DIAMETER_PX / 2.0
SCREEN_WIDTH_PX = 1920
SCREEN_HEIGHT_PX = 1080
USE_SCREEN_CENTER = True
SCREEN_VIEW_FULL = True

CM_PER_PX = PIXEL_PITCH_MM / 10.0


def effective_cutoff_hz(cutoff_hz=CUTOFF_HZ, fs_hz=1.0 / GRID_DT_S, order=2, n_passes=2):
    ratio = (2.0 ** (1.0 / n_passes) - 1.0) ** (1.0 / (2.0 * order))
    return float(fs_hz / np.pi * np.arctan(ratio * np.tan(np.pi * cutoff_hz / fs_hz)))


EFFECTIVE_CUTOFF_HZ = effective_cutoff_hz()

GEOMETRY_SPEC_GLOBS = ["geometry_spec*.json", "geometry_spec*.csv"]
GEOMETRY_KEYS = {
    "CENTER_TO_TARGET_DIST_PX": "center_to_target_dist_px",
    "TARGET_DIST_SCALE": "target_dist_scale",
    "TARGET_DIAMETER_PX": "target_diameter_px",
    "CENTER_DIAMETER_PX": "center_diameter_px",
}


@dataclass
class ScreenGeometry:
    center_to_target_dist_px: float = CENTER_TO_TARGET_DIST_PX
    target_dist_scale: float = TARGET_DIST_SCALE
    target_diameter_px: float = TARGET_DIAMETER_PX
    center_diameter_px: float = CENTER_DIAMETER_PX
    source: str = "defaults (module constants)"

    @property
    def target_dist_px(self):
        return self.center_to_target_dist_px * self.target_dist_scale

    @property
    def target_radius_px(self):
        return self.target_diameter_px / 2.0

    @property
    def center_radius_px(self):
        return self.center_diameter_px / 2.0

    @staticmethod
    def _read_pairs(path):
        path = Path(path)
        if path.suffix.lower() == ".json":
            with open(path) as fh:
                data = json.load(fh)
            return {str(k).strip().upper(): v for k, v in data.items()}
        raw = pd.read_csv(path, header=None, dtype=str)
        first_col = raw.iloc[:, 0].astype(str).str.strip().str.upper()
        if raw.shape[1] >= 2 and first_col.isin(list(GEOMETRY_KEYS)).any():
            return dict(zip(first_col, raw.iloc[:, 1]))
        if len(raw) < 2:
            return {}
        header = raw.iloc[0].astype(str).str.strip().str.upper()
        return dict(zip(header, raw.iloc[1]))

    @classmethod
    def from_file(cls, path):
        geometry = cls()
        try:
            pairs = cls._read_pairs(path)
        except Exception as exc:
            print(f"WARNING: could not read geometry spec '{path}' ({exc}); using the defaults.")
            return geometry
        missing = []
        for key, attr in GEOMETRY_KEYS.items():
            value = pd.to_numeric(pd.Series([pairs.get(key)]), errors="coerce").iloc[0]
            if pd.notna(value) and np.isfinite(value):
                setattr(geometry, attr, float(value))
            else:
                missing.append(key)
        geometry.source = f"'{Path(path).name}'"
        if missing:
            geometry.source += f" (defaults kept for {missing})"
        return geometry

    @classmethod
    def from_session_params(cls, session_params):
        """Geometry read straight from that session's params_sess*.mat
        (orgParams.centerRad / targetRad / centerToTargetDist), the real
        source of truth for the screen the task actually rendered that day --
        see load_session_params in section 0.1. geometry_spec*.json/csv
        (from_file, above) is kept only as a manual-override path for
        sessions that predate params_sess exports."""
        geometry = cls()
        keys = {
            "center_to_target_dist_px": "center_to_target_dist_px",
            "target_diameter_px": "target_diameter_px",
            "center_diameter_px": "center_diameter_px",
        }
        missing = []
        for src_key, attr in keys.items():
            value = session_params.get(src_key)
            if value is not None and np.isfinite(value):
                setattr(geometry, attr, float(value))
            else:
                missing.append(src_key)
        geometry.source = f"orgParams in '{Path(session_params['source']).name}'"
        if missing:
            geometry.source += f" (defaults kept for {missing})"
        return geometry

    @classmethod
    def resolve(cls, trial_data_path=None, explicit_path=None, patterns=GEOMETRY_SPEC_GLOBS,
               session_params=None):
        session_params = globals().get("SESSION_PARAMS") if session_params is None else session_params
        if session_params and session_params.get("center_to_target_dist_px") is not None:
            return cls.from_session_params(session_params)
        candidates = []
        if trial_data_path:
            folder = Path(trial_data_path).parent
            for pattern in patterns:
                candidates.extend(sorted(folder.glob(pattern)))
        if explicit_path and Path(explicit_path).suffix.lower() != ".mat":
            candidates.append(Path(explicit_path))
        for cand in candidates:
            if Path(cand).is_file():
                return cls.from_file(cand)
        return cls()

    def describe(self):
        return (f"Screen geometry from {self.source}: centre diameter {self.center_diameter_px:g} px, "
                f"target diameter {self.target_diameter_px:g} px, centre-to-target "
                f"{self.center_to_target_dist_px:g} px x {self.target_dist_scale:g} = "
                f"{self.target_dist_px:g} px.")

FONT_CANDIDATES = ["fonts/harding.ttf", "fonts/Harding.ttf"]
FONT_GLOBS = ["/content/*/fonts/harding.ttf", "/content/*/fonts/Harding.ttf"]


def _resolve_font():
    for cand in FONT_CANDIDATES:
        if Path(cand).exists():
            return cand
    for pattern in FONT_GLOBS:
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits[0]
    return None


def _register_font(path):
    try:
        if path and Path(path).exists():
            fm.fontManager.addfont(path)
            plt.rcParams["font.family"] = fm.FontProperties(fname=path).get_name()
            return True
    except Exception:
        pass
    return False


plt.rcParams["axes.unicode_minus"] = False
FONT_PATH = _resolve_font()
if _register_font(FONT_PATH):
    print(f"Using font: {FONT_PATH}")
else:
    print("Harding font not found (looked in fonts/harding.ttf and /content/*/fonts/); using default font.")


FIGURE_DPI = 300
BASE_FONT_SIZE = 10

# Figure ink, storytelling-with-data style: everything that is context is grey,
# colour is spent only on the mark the reader should look at. Category colours
# stay the rig's (ColorCategoryMap.m), because a figure and the screen must agree.
INK = "#333333"          # text, emphasised lines
GRAY = "#8c8c8c"         # context marks, error bars, reference lines
GRAY_LIGHT = "#d9d9d9"   # de-emphasised fills, bands, chance lines
ACCENT = "#31688e"       # the one series the panel is about
ACCENT_2 = "#d44842"     # a second, warm accent (incorrect, a contrasting series)


def apply_figure_style(dpi=FIGURE_DPI, base=BASE_FONT_SIZE):
    """One font, one export resolution and one ink for every figure.

    rcParams are global to the kernel, so calling this once here -- before
    anything is drawn -- is what makes the whole notebook consistent instead
    of each cell setting its own sizes. savefig.dpi is set as well as the
    explicit dpi= arguments, so even a figure saved without one comes out at
    the same resolution. font.family is left alone: it was just set from
    harding.ttf above, and overriding it here would undo that.

    The chrome follows the storytelling-with-data rules: no top/right spines,
    grey hairline axes and ticks, no grid, frameless legends, left-aligned
    titles, so the data ink is the darkest thing on the page.
    """
    plt.rcParams.update({
        "figure.dpi": 110,
        "savefig.dpi": dpi,
        "savefig.bbox": "tight",
        "font.size": base,
        "axes.titlesize": base + 1,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "figure.titlesize": base + 2,
        "axes.unicode_minus": False,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": GRAY,
        "axes.linewidth": 0.8,
        "axes.labelcolor": INK,
        "axes.titlecolor": INK,
        "axes.titlelocation": "left",
        "axes.grid": False,
        "xtick.color": GRAY,
        "ytick.color": GRAY,
        "xtick.labelcolor": INK,
        "ytick.labelcolor": INK,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "text.color": INK,
        "legend.frameon": False,
        "lines.linewidth": 1.6,
        "lines.markersize": 5,
        "errorbar.capsize": 0,
    })


apply_figure_style()


class HampelScreen:
    MAD_SCALE = 1.4826

    def __init__(self, half_window=3, n_sigma=3.0):
        self.half_window = max(1, int(round(half_window)))
        self.n_sigma = float(n_sigma)

    def detect(self, x):
        x = np.atleast_2d(np.asarray(x, dtype=float))
        if x.shape[0] == 1 and x.shape[1] > 1:
            x = x.T
        n, d = x.shape
        cleaned = x.copy()
        median_env = x.copy()
        threshold_env = np.zeros_like(x)
        mask = np.zeros_like(x, dtype=bool)
        if n < 3:
            return cleaned, mask, median_env, threshold_env
        h = self.half_window
        for col in range(d):
            source = x[:, col]
            for i in range(n):
                lo = max(0, i - h)
                hi = min(n, i + h + 1)
                window = source[lo:hi]
                med = np.median(window)
                mad = self.MAD_SCALE * np.median(np.abs(window - med))
                thr = self.n_sigma * mad
                median_env[i, col] = med
                threshold_env[i, col] = thr
                if mad > 0 and abs(source[i] - med) > thr:
                    cleaned[i, col] = med
                    mask[i, col] = True
        return cleaned, mask, median_env, threshold_env


class ButterworthLowpass:
    N_FACT = 6

    def __init__(self, cutoff_hz=20.0):
        self.cutoff_hz = float(cutoff_hz)

    def _design(self, fs):
        k = np.tan(np.pi * self.cutoff_hz / fs)
        norm = 1.0 / (1.0 + np.sqrt(2.0) * k + k ** 2)
        b = np.array([k ** 2, 2.0 * k ** 2, k ** 2]) * norm
        a = np.array([1.0, 2.0 * (k ** 2 - 1.0) * norm, (1.0 - np.sqrt(2.0) * k + k ** 2) * norm])
        return b, a

    @staticmethod
    def _steady_state(b, a):
        companion = np.array([[-a[1], 1.0], [-a[2], 0.0]])
        rhs = np.array([b[1] - a[1] * b[0], b[2] - a[2] * b[0]])
        return np.linalg.solve(np.eye(2) - companion, rhs)

    def __call__(self, x, fs):
        x = np.atleast_2d(np.asarray(x, dtype=float))
        if x.shape[0] == 1 and x.shape[1] > 1:
            x = x.T
        n, d = x.shape
        y = x.copy()
        if n == 0 or fs <= 0 or self.cutoff_hz <= 0 or self.cutoff_hz >= fs / 2.0:
            return y, False
        if n <= self.N_FACT:
            return y, False
        b, a = self._design(fs)
        zi = self._steady_state(b, a)
        nf = self.N_FACT
        for col in range(d):
            series = x[:, col]
            pre = 2.0 * series[0] - series[nf:0:-1]
            post = 2.0 * series[-1] - series[-2:-nf - 2:-1]
            ext = np.concatenate([pre, series, post])
            ext, _ = lfilter(b, a, ext, zi=zi * ext[0])
            ext = ext[::-1]
            ext, _ = lfilter(b, a, ext, zi=zi * ext[0])
            ext = ext[::-1]
            y[:, col] = ext[nf:-nf]
        return y, True


class KalmanTrajectorySmoother:
    def __init__(self, meas_sigma_px=2.0, jerk_sigma_px_s3=1e5, gate_sigma=np.inf):
        self.meas_sigma_px = float(meas_sigma_px)
        self.jerk_sigma_px_s3 = float(jerk_sigma_px_s3)
        self.gate_sigma = float(gate_sigma)

    def __call__(self, t, xy):
        t = np.asarray(t, dtype=float).ravel()
        xy = np.atleast_2d(np.asarray(xy, dtype=float))
        if xy.shape[0] == 1 and xy.shape[1] > 1:
            xy = xy.T
        n = t.size
        smoothed = xy.copy()
        n_gated = 0
        if n < 3 or xy.shape[0] != n:
            return smoothed, n_gated
        r = self.meas_sigma_px ** 2
        q = self.jerk_sigma_px_s3 ** 2
        dt_all = np.diff(t)
        acc_std0 = self.jerk_sigma_px_s3 * 20.0 * np.median(dt_all)
        gate2 = self.gate_sigma ** 2
        for col in range(xy.shape[1]):
            z = xy[:, col]
            xf = np.zeros((3, n))
            pf = np.zeros((3, 3, n))
            xp = np.zeros((3, n))
            pp = np.zeros((3, 3, n))
            fk = np.zeros((3, 3, n))
            dt0 = dt_all[0]
            xp[:, 0] = [z[0], (z[1] - z[0]) / dt0, 0.0]
            pp[:, :, 0] = np.diag([r, 2.0 * r / dt0 ** 2, acc_std0 ** 2])
            fk[:, :, 0] = np.eye(3)
            for k in range(n):
                if k > 0:
                    dt = dt_all[k - 1]
                    f = np.array([[1.0, dt, dt ** 2 / 2.0],
                                  [0.0, 1.0, dt],
                                  [0.0, 0.0, 1.0]])
                    qm = q * np.array([[dt ** 5 / 20.0, dt ** 4 / 8.0, dt ** 3 / 6.0],
                                       [dt ** 4 / 8.0, dt ** 3 / 3.0, dt ** 2 / 2.0],
                                       [dt ** 3 / 6.0, dt ** 2 / 2.0, dt]])
                    fk[:, :, k] = f
                    xp[:, k] = f @ xf[:, k - 1]
                    pp[:, :, k] = f @ pf[:, :, k - 1] @ f.T + qm
                innov = z[k] - xp[0, k]
                s = pp[0, 0, k] + r
                if innov ** 2 <= gate2 * s:
                    gain = pp[:, 0, k] / s
                    xf[:, k] = xp[:, k] + gain * innov
                    pf[:, :, k] = pp[:, :, k] - np.outer(gain, pp[0, :, k])
                    pf[:, :, k] = (pf[:, :, k] + pf[:, :, k].T) / 2.0
                else:
                    xf[:, k] = xp[:, k]
                    pf[:, :, k] = pp[:, :, k]
                    n_gated += 1
            xs = xf.copy()
            for k in range(n - 2, -1, -1):
                f = fk[:, :, k + 1]
                c = pf[:, :, k] @ f.T @ np.linalg.inv(pp[:, :, k + 1])
                xs[:, k] = xf[:, k] + c @ (xs[:, k + 1] - xp[:, k + 1])
            smoothed[:, col] = xs[0, :]
        return smoothed, n_gated


@dataclass
class KinematicSummary:
    peak_vel_cm: float = float("nan")
    mean_vel_cm: float = float("nan")
    median_vel_cm: float = float("nan")
    peak_accel_cm: float = float("nan")
    mean_accel_cm: float = float("nan")
    median_accel_cm: float = float("nan")
    vel_time_ms: np.ndarray = field(default_factory=lambda: np.array([]))
    vel_cm: np.ndarray = field(default_factory=lambda: np.array([]))
    accel_cm: np.ndarray = field(default_factory=lambda: np.array([]))
    move_takeoff_ms: float = float("nan")
    n_speed_peaks: int = 0
    peak_vel_time_ms: float = float("nan")
    window_start_ms: float = float("nan")
    window_end_ms: float = float("nan")


@dataclass
class TrialResult:
    raw_time_ms: np.ndarray
    raw_xy: np.ndarray
    grid_time_ms: np.ndarray
    grid_xy: np.ndarray
    outlier_mask: np.ndarray
    hampel_median: np.ndarray
    hampel_threshold: np.ndarray
    n_replaced: int
    butter_applied: bool
    under_resolved: bool
    peak_vel_cm: float
    mean_vel_cm: float
    median_vel_cm: float
    peak_accel_cm: float
    mean_accel_cm: float
    median_accel_cm: float
    vel_time_ms: np.ndarray
    vel_cm: np.ndarray
    accel_cm: np.ndarray
    move_onset_ms: float = float("nan")
    move_offset_ms: float = float("nan")
    move_takeoff_ms: float = float("nan")
    n_duplicate_ts: int = 0
    n_speed_peaks: int = 0
    peak_vel_time_ms: float = float("nan")
    window_start_ms: float = float("nan")
    window_end_ms: float = float("nan")


class TrajectoryProcessor:
    def __init__(self, grid_dt_s=0.008, cutoff_hz=CUTOFF_HZ, cm_per_px=CM_PER_PX,
                min_move_samples=5, min_move_dur_s=None,
                hampel_half_window=3, hampel_n_sigma=3.0,
                meas_sigma_px=2.0, jerk_sigma_px_s3=1e5, gate_sigma=np.inf):
        self.grid_dt_s = grid_dt_s
        self.cutoff_hz = cutoff_hz
        self.cm_per_px = cm_per_px
        self.min_move_samples = min_move_samples
        self.min_move_dur_s = min_move_dur_s
        self.hampel = HampelScreen(hampel_half_window, hampel_n_sigma)
        self.butter = ButterworthLowpass(cutoff_hz)
        self.kalman = KalmanTrajectorySmoother(meas_sigma_px, jerk_sigma_px_s3, gate_sigma)

    @classmethod
    def for_profile(cls, profile, **kwargs):
        return cls(hampel_half_window=profile.hampel_half_window,
                   hampel_n_sigma=profile.hampel_n_sigma,
                   meas_sigma_px=profile.meas_sigma_px,
                   jerk_sigma_px_s3=profile.jerk_sigma_px_s3,
                   gate_sigma=profile.gate_sigma, **kwargs)

    @staticmethod
    def _savgol_accel(vel_cm, dt_s, window, poly):
        n = int(vel_cm.size)
        if n == 0:
            return np.array([])
        if n == 1:
            return np.array([0.0])
        if n < 5:
            return np.abs(np.gradient(vel_cm, dt_s))
        w = min(int(window), n if n % 2 == 1 else n - 1)
        if w % 2 == 0:
            w -= 1
        w = max(w, 5)
        p = min(int(poly), w - 1)
        return np.abs(savgol_filter(vel_cm, w, p, deriv=1, delta=dt_s))

    def _build_grid(self, t):
        total = t[-1] - t[0]
        if total < self.grid_dt_s:
            return np.array([t[0], t[-1]])
        grid = np.arange(t[0], t[-1] + 1e-12, self.grid_dt_s)
        if t[-1] - grid[-1] > self.grid_dt_s / 4.0:
            grid = np.append(grid, t[-1])
        else:
            grid[-1] = t[-1]
        return grid

    @staticmethod
    def _takeoff_index(vel_cm, pk_idx, frac):
        """Walk backward from an ALREADY-KNOWN peak-speed sample (pk_idx) to
        the first earlier sample still at or above frac * peak.

        This is a retrospective (offline) definition of movement onset: pk_idx
        is the index of the largest speed in the search window, found by
        _kinematics before this is called, so takeoff is only ever set after
        the peak is known. When a trial has more than one local speed peak
        (e.g. a correction / change of mind), this returns the takeoff for
        whichever peak is largest overall, not necessarily the first peak in
        time -- see section 1.9 for trials where that distinction matters.
        """
        thr = frac * float(vel_cm[pk_idx])
        lo_idx = pk_idx
        while lo_idx > 0 and vel_cm[lo_idx - 1] >= thr:
            lo_idx -= 1
        return lo_idx

    def _kinematics(self, grid_time_ms, grid_xy, under_resolved, window_end_ms=None):
        if under_resolved:
            return KinematicSummary()
        t = grid_time_ms / 1000.0
        gdt = np.diff(t)
        if gdt.size < 1 or np.any(gdt <= 0):
            return KinematicSummary()
        vel = np.hypot(np.diff(grid_xy[:, 0]), np.diff(grid_xy[:, 1])) / gdt
        tv_ms = (t[:-1] + gdt / 2.0) * 1000.0
        vel_cm = vel * self.cm_per_px
        accel_cm = self._savgol_accel(vel_cm, float(np.median(gdt)), SAVGOL_WINDOW, SAVGOL_POLYORDER)
        summary = KinematicSummary(vel_time_ms=tv_ms, vel_cm=vel_cm, accel_cm=accel_cm)
        if window_end_ms is not None and np.isfinite(window_end_ms):
            search = tv_ms <= window_end_ms
        else:
            search = np.ones(tv_ms.shape, dtype=bool)
        if not search.any():
            search = np.ones(tv_ms.shape, dtype=bool)
        search_idx = np.where(search)[0]
        seg = vel_cm[search_idx]
        local_peaks = (find_peaks(seg, prominence=MULTI_PEAK_PROMINENCE_FRAC * float(seg.max()))[0]
                      if seg.size >= 3 and seg.max() > 0 else np.array([], dtype=int))
        summary.n_speed_peaks = max(int(local_peaks.size), 1)
        if TAKEOFF_PEAK_CHOICE == "first" and local_peaks.size:
            pk_idx = int(search_idx[local_peaks[0]])
        else:
            pk_idx = int(search_idx[np.argmax(seg)])
        if float(vel[pk_idx]) < 1e-6:
            return summary
        lo_idx = self._takeoff_index(vel_cm, pk_idx, MOVE_SPEED_FRAC)
        summary.move_takeoff_ms = float(tv_ms[lo_idx])
        mask = search & (np.arange(tv_ms.size) >= lo_idx)
        win_time = tv_ms[mask]
        win_vel = vel_cm[mask]
        summary.window_start_ms = float(win_time[0])
        summary.window_end_ms = float(win_time[-1])
        k = int(np.argmax(win_vel))
        summary.peak_vel_cm = float(win_vel[k])
        summary.peak_vel_time_ms = float(win_time[k])
        summary.mean_vel_cm = float(np.mean(win_vel))
        summary.median_vel_cm = float(np.median(win_vel))
        win_acc = accel_cm[mask] if accel_cm.size == vel_cm.size else np.array([])
        win_acc = win_acc[np.isfinite(win_acc)]
        if win_acc.size:
            summary.peak_accel_cm = float(np.percentile(win_acc, ACCEL_PEAK_PERCENTILE))
            summary.mean_accel_cm = float(np.mean(win_acc))
            summary.median_accel_cm = float(np.median(win_acc))
        return summary

    def process(self, time_ms, x, y, move_window_ms=None, hold_start_ms=None):
        t = np.asarray(time_ms, dtype=float) / 1000.0
        order = np.argsort(t, kind="stable")
        t = t[order]
        xy = np.column_stack([np.asarray(x, dtype=float)[order],
                            np.asarray(y, dtype=float)[order]])
        n_raw = t.size
        t_unique, keep = np.unique(t, return_index=True)
        n_duplicate_ts = int(n_raw - t_unique.size)
        t = t_unique
        xy = xy[keep]
        n = t.size
        under_resolved = n < self.min_move_samples
        if self.min_move_dur_s is not None and n >= 2:
            under_resolved = under_resolved or (t[-1] - t[0]) < self.min_move_dur_s
        screened, outlier_mask, median_env, threshold_env = self.hampel.detect(xy)
        n_replaced = int(outlier_mask.sum())
        stage1, _ = self.kalman(t, screened)
        if n < 2:
            grid = t.copy()
            grid_xy = stage1.copy()
            applied = False
        else:
            grid = self._build_grid(t)
            grid_xy = np.column_stack([
                PchipInterpolator(t, stage1[:, 0])(grid),
                PchipInterpolator(t, stage1[:, 1])(grid),
            ])
            grid_fs = 1.0 / self.grid_dt_s
            grid_xy, applied = self.butter(grid_xy, grid_fs)
        grid_time_ms = (grid - t[0]) * 1000.0
        t0_ms = t[0] * 1000.0
        move_window_ms0 = None
        move_onset_ms = float("nan")
        if move_window_ms is not None:
            move_window_ms0 = (move_window_ms[0] - t0_ms, move_window_ms[1] - t0_ms)
            move_onset_ms = move_window_ms0[0]
        move_offset_ms = (hold_start_ms - t0_ms) if hold_start_ms is not None else float("nan")
        window_end_ms = move_window_ms0[1] if move_window_ms0 is not None else None
        kin = self._kinematics(grid_time_ms, grid_xy, under_resolved, window_end_ms)
        return TrialResult(
            raw_time_ms=(t - t[0]) * 1000.0,
            raw_xy=xy,
            grid_time_ms=grid_time_ms,
            grid_xy=grid_xy,
            outlier_mask=outlier_mask,
            hampel_median=median_env,
            hampel_threshold=threshold_env,
            n_replaced=n_replaced,
            butter_applied=applied,
            under_resolved=under_resolved,
            peak_vel_cm=kin.peak_vel_cm,
            mean_vel_cm=kin.mean_vel_cm,
            median_vel_cm=kin.median_vel_cm,
            peak_accel_cm=kin.peak_accel_cm,
            mean_accel_cm=kin.mean_accel_cm,
            median_accel_cm=kin.median_accel_cm,
            vel_time_ms=kin.vel_time_ms,
            vel_cm=kin.vel_cm,
            accel_cm=kin.accel_cm,
            move_onset_ms=move_onset_ms,
            move_offset_ms=move_offset_ms,
            move_takeoff_ms=kin.move_takeoff_ms,
            n_duplicate_ts=n_duplicate_ts,
            n_speed_peaks=kin.n_speed_peaks,
            peak_vel_time_ms=kin.peak_vel_time_ms,
            window_start_ms=kin.window_start_ms,
            window_end_ms=kin.window_end_ms,
        )


INPUT_SOURCE_OVERRIDES = {}
RZ2_MAX_MEDIAN_STEP_MS = 3.0


@dataclass
class PipelineProfile:
    hampel_half_window: int = 3
    hampel_n_sigma: float = 3.0
    meas_sigma_px: float = 2.0
    jerk_sigma_px_s3: float = 1e5
    gate_sigma: float = np.inf
    drop_unindexed_rows: bool = False


PIPELINE_PROFILES = {
    "rz2adc": PipelineProfile(drop_unindexed_rows=True),
    "usb": PipelineProfile(),
    "unknown": PipelineProfile(),
}


class InputSourceDetector:
    RZ2 = "rz2adc"
    USB = "usb"
    UNKNOWN = "unknown"
    GROUP_KEYS = ["Block", "TrialNumInBlock", "Attempt"]
    TAG_PREFIXES = ("trajectory_movement_", "trajectory_", "trial_data_", "trial_kinematics_")

    def __init__(self, overrides=None, max_rz2_step_ms=None):
        self.overrides = INPUT_SOURCE_OVERRIDES if overrides is None else overrides
        self.max_rz2_step_ms = RZ2_MAX_MEDIAN_STEP_MS if max_rz2_step_ms is None else max_rz2_step_ms

    @classmethod
    def run_tag(cls, path):
        name = Path(str(path)).stem
        for prefix in cls.TAG_PREFIXES:
            if name.startswith(prefix):
                return name[len(prefix):]
        return name

    @classmethod
    def median_step_ms(cls, frame):
        if "Time_ms" not in frame.columns:
            return float("nan")
        keys = [k for k in cls.GROUP_KEYS if k in frame.columns]
        d = frame.assign(_t=pd.to_numeric(frame["Time_ms"], errors="coerce")).dropna(subset=["_t"])
        if keys:
            d = d.sort_values(keys + ["_t"], kind="stable")
            steps = d.groupby(keys)["_t"].diff()
        else:
            steps = d["_t"].sort_values(kind="stable").diff()
        steps = steps[steps > 0]
        return float(steps.median()) if len(steps) else float("nan")

    # orgParams.inputSource values (ConfigOrgParams.m) mapped onto this
    # detector's RZ2/USB: 'rz2adc' is the RZ2 analog rig, 'joystick' and
    # 'mouse' are both standard USB HID devices as far as the
    # signal-processing pipeline below is concerned.
    PARAM_SOURCE_MAP = {"rz2adc": "rz2adc", "joystick": "usb", "mouse": "usb"}

    def detect(self, frame, path=None):
        tag = self.run_tag(path) if path is not None else ""
        for key, source in self.overrides.items():
            if key and (key == tag or key in tag):
                return source, "manual override in INPUT_SOURCE_OVERRIDES"
        session_params = globals().get("SESSION_PARAMS") or {}
        raw_source = session_params.get("input_source")
        mapped = self.PARAM_SOURCE_MAP.get(raw_source)
        if mapped is not None:
            fname = Path(session_params["source"]).name
            return mapped, f"orgParams.inputSource='{raw_source}' in '{fname}'"
        if "RZ2Idx" in frame.columns and pd.to_numeric(frame["RZ2Idx"], errors="coerce").notna().any():
            return self.RZ2, "RZ2Idx has indexed rows"
        step = self.median_step_ms(frame)
        if not np.isfinite(step):
            return self.UNKNOWN, "no RZ2Idx values and no usable Time_ms steps"
        source = self.RZ2 if step < self.max_rz2_step_ms else self.USB
        return source, f"no RZ2Idx values; median sample step {step:.2f} ms"


@dataclass
class Trial:
    date: str
    block: int
    trial: int
    attempt: int
    time_ms: np.ndarray
    move_time_ms: np.ndarray
    x: np.ndarray
    y: np.ndarray
    fs_hz: float
    move_window_ms: tuple = None
    hold_start_ms: float = None
    hold_window_ms: tuple = None
    hold_max_speed_cm: float = None
    hold_excursion_px: float = None

    @property
    def n_samples(self):
        return int(self.x.size)

    @property
    def label(self):
        return f"b{self.block:02d}_t{self.trial:02d}_a{self.attempt:02d}"


class TrajectoryDataset:
    GROUP_KEYS = ["Block", "TrialNumInBlock", "Attempt"]

    def __init__(self, csv_path, move_epochs=None, detector=None):
        """
        move_epochs: None = no epoch filtering (use every row in csv_path
        as-is). Otherwise a single TaskEpoch code (e.g. MOVEMENT_EPOCH) or
        an iterable of codes (e.g. MOVE_EPOCHS = (DECISION_EPOCH,
        MOVEMENT_EPOCH, TARGET_HOLD_EPOCH)) -- a row is kept when its Epoch
        matches ANY of them. This mirrors CenterOutTask.m's kinematicsEpochs
        / the ismember(...) filter TrialKinematics.m and
        SaveMovementTrajectory.m now use on the MATLAB side, applied here
        explicitly instead of trusting the CSV to already be restricted the
        way we expect. Note this can only ever KEEP rows that exist in
        csv_path already -- if the session was recorded before
        kinematicsEpochs included a given epoch, that epoch's rows were
        never exported and this filter finds nothing to add for it (see the
        module-level comment above MOVE_EPOCHS).
        """
        self.path = Path(csv_path)
        self.frame = pd.read_csv(self.path)
        if "Epoch" in self.frame.columns and not pd.api.types.is_numeric_dtype(self.frame["Epoch"]):
            mapped = self.frame["Epoch"].map(_epoch_to_code)
            n_bad = int(mapped.isna().sum())
            if n_bad:
                unknown = sorted(set(self.frame.loc[mapped.isna(), "Epoch"].astype(str)))[:6]
                print(f"WARNING: {n_bad} rows have unrecognized Epoch names {unknown}; "
                    f"left unmatched -- add them to _EPOCH_NAME_TO_CODE if needed.")
            self.frame["Epoch"] = mapped
            print(f"Epoch column was text; normalized to numeric codes "
                f"{sorted(set(self.frame['Epoch'].dropna().astype(int)))}.")
        self.input_source, self.source_reason = (detector or InputSourceDetector()).detect(self.frame, self.path)
        self.profile = PIPELINE_PROFILES.get(self.input_source, PIPELINE_PROFILES["unknown"])
        self.n_unindexed_dropped = 0
        print(f"Input source of '{self.path.name}': {self.input_source} ({self.source_reason}).")
        if self.profile.drop_unindexed_rows and "RZ2Idx" in self.frame.columns:
            unindexed = pd.to_numeric(self.frame["RZ2Idx"], errors="coerce").isna()
            self.n_unindexed_dropped = int(unindexed.sum())
            self.frame = self.frame[~unindexed]
            print(f"  Dropped {self.n_unindexed_dropped} rows with NaN RZ2Idx (cached or un-indexed samples).")
        elif self.profile.drop_unindexed_rows:
            print("  No RZ2Idx column (export older than v8.21): cached rows cannot be identified and are kept.")
        if move_epochs is not None:
            if "Epoch" not in self.frame.columns:
                print(f"WARNING: '{self.path.name}' has no Epoch column -- "
                    f"cannot filter to move_epochs={move_epochs}; using every row as-is.")
            else:
                epochs = np.atleast_1d(move_epochs)
                before = len(self.frame)
                self.frame = self.frame[self.frame["Epoch"].isin(epochs)]
                print(f"Epoch filter {tuple(epochs)}: kept {len(self.frame)}/{before} rows of '{self.path.name}'.")

    @staticmethod
    def estimate_sampling_rate(time_ms):
        time_ms = np.asarray(time_ms, dtype=float)
        if time_ms.size < 2:
            return float("nan")
        steps = np.diff(np.sort(time_ms))
        steps = steps[steps > 0]
        if steps.size == 0:
            return float("nan")
        return 1000.0 / float(np.mean(steps))

    @staticmethod
    def _move_time_ms(group):
        if "MoveTime_ms" in group.columns:
            return group["MoveTime_ms"].to_numpy(dtype=float)
        tms = group["Time_ms"].to_numpy(dtype=float)
        return tms - float(tms.min()) if tms.size else tms

    def trials(self):
        for (block, trial, attempt), group in self.frame.groupby(self.GROUP_KEYS, sort=True):
            group = group.sort_values("Time_ms")
            move_window_ms = None
            hold_start_ms = None
            hold_window_ms = None
            hold_max_speed_cm = None
            hold_excursion_px = None
            if "Epoch" in group.columns:
                ep = group["Epoch"].to_numpy()
                tms = group["Time_ms"].to_numpy(dtype=float)
                xs_ = group["X_px"].to_numpy(dtype=float)
                ys_ = group["Y_px"].to_numpy(dtype=float)
                mv = tms[ep == MOVEMENT_EPOCH]
                if mv.size:
                    move_window_ms = (float(mv.min()), float(mv.max()))
                hmask = ep == TARGET_HOLD_EPOCH
                hold = tms[hmask]
                if hold.size:
                    hold_start_ms = float(hold.min())
                if hold.size >= 2:
                    hx, hy = xs_[hmask], ys_[hmask]
                    hold_window_ms = (float(hold.min()), float(hold.max()))
                    hdt = np.diff(hold) / 1000.0
                    good = hdt > 0
                    if good.any():
                        seg = np.hypot(np.diff(hx), np.diff(hy))[good] / hdt[good] * CM_PER_PX
                        hold_max_speed_cm = float(seg.max()) if seg.size else None
                    hold_excursion_px = float(np.max(np.hypot(hx - hx[0], hy - hy[0])))
            yield Trial(
                date=str(group["Date"].iloc[0]),
                block=int(block),
                trial=int(trial),
                attempt=int(attempt),
                time_ms=group["Time_ms"].to_numpy(dtype=float),
                move_time_ms=self._move_time_ms(group),
                x=group["X_px"].to_numpy(dtype=float),
                y=group["Y_px"].to_numpy(dtype=float),
                fs_hz=self.estimate_sampling_rate(group["Time_ms"].to_numpy()),
                move_window_ms=move_window_ms,
                hold_start_ms=hold_start_ms,
                hold_window_ms=hold_window_ms,
                hold_max_speed_cm=hold_max_speed_cm,
                hold_excursion_px=hold_excursion_px,
            )

    def global_limits(self, margin=0.03, max_time_ms=None):
        t_max = 0.0
        p_min, p_max = np.inf, -np.inf
        over_cap = []
        for tr in self.trials():
            tr_t_max = float(tr.move_time_ms.max())
            t_max = max(t_max, tr_t_max)
            if max_time_ms is not None and tr_t_max > max_time_ms:
                over_cap.append((tr.block, tr.trial, tr.attempt, tr_t_max))
            p_min = min(p_min, float(tr.x.min()), float(tr.y.min()))
            p_max = max(p_max, float(tr.x.max()), float(tr.y.max()))
        if max_time_ms is not None:
            t_max = min(t_max, max_time_ms)
        pad = (p_max - p_min) * margin
        return (0.0, t_max), (p_min - pad, p_max + pad), over_cap

    def global_space_limits(self, margin=0.05):
        x_min, x_max = np.inf, -np.inf
        y_min, y_max = np.inf, -np.inf
        for tr in self.trials():
            x_min = min(x_min, float(tr.x.min()))
            x_max = max(x_max, float(tr.x.max()))
            y_min = min(y_min, float(tr.y.min()))
            y_max = max(y_max, float(tr.y.max()))
        px = (x_max - x_min) * margin
        py = (y_max - y_min) * margin
        return (x_min - px, x_max + px), (y_min - py, y_max + py)


def normalize_trial_schema(td):
    """Canonicalize the timing columns of a trial_data_*.csv across engine
    generations, so old and new sessions can be analysed by the same code.

    Four layouts exist in the wild. BOTH header spellings of the total are
    accepted on input, and the older ReactionTime_s header names a DIFFERENT
    interval in two of them -- which is the whole reason this function exists:

    gen A (v2_2 / pre-rename)   DecisionTime_s, ReactionTime_s, TotalTime_s
                                  ReactionTime_s = leave-center -> reach-target
                                  TotalTime_s    = decision + execution
    gen B (interim)             DecisionTime_s, ExecutionTime_s, TotalTime_s
    gen C (v8.19)               DecisionTime_s, ExecutionTime_s, ReactionTime_s
                                  ReactionTime_s = decision + execution (TOTAL)
    gen D (v8.20 onward)        DecisionTime_s, ExecutionTime_s, TotalTime_s
                                  TotalTime_s    = decision + execution (TOTAL)

    gen A is separated from the rest by whether ExecutionTime_s is present,
    which is the one signal that distinguishes them:
    present -> a ReactionTime_s column, if any, is the TOTAL   (gen C)
    absent  -> ReactionTime_s is the EXECUTION time            (gen A)

    Canonical output, always these three whatever came in:
    DecisionTime_s   target-onset  -> leave-center
    ExecutionTime_s  leave-center  -> reach-target
    TotalTime_s      DecisionTime_s + ExecutionTime_s

    ReactionTime_s is dropped after being folded into TotalTime_s: keeping
    both would put two identical columns into the correlation heatmap and
    the PCA, where an exact duplicate is not a second measurement.
    """
    td = td.copy()
    if "ExecutionTime_s" not in td.columns and "ReactionTime_s" in td.columns:
        td["ExecutionTime_s"] = td["ReactionTime_s"]
        td = td.drop(columns=["ReactionTime_s"])
    if "TotalTime_s" not in td.columns:
        if "ReactionTime_s" in td.columns:
            td["TotalTime_s"] = td["ReactionTime_s"]
        elif {"DecisionTime_s", "ExecutionTime_s"} <= set(td.columns):
            td["TotalTime_s"] = td["DecisionTime_s"] + td["ExecutionTime_s"]
    if "ReactionTime_s" in td.columns:
        td = td.drop(columns=["ReactionTime_s"])
    return td


class TrialInfoTable:
    """Per-trial lookups keyed on (Block, TrialNumInBlock, Attempt).

    bar_size_deg / direction_correct / is_correct / error_type always come
    from trial_data_<runTag>.csv.

    move_samples (NumMovementSamples) moved OUT of trial_data_<runTag>.csv
    into its own companion file, trial_kinematics_<runTag>.csv, on the
    MATLAB side (same runTag, same Block+TrialNumInBlock+Attempt key --
    see CenterOutTask.m). To stay usable on BOTH older sessions (column
    still sitting in trial_data) and current ones (column only in
    trial_kinematics), this checks trial_data first and falls back to the
    companion kinematics file, auto-detected by swapping the
    'trial_data_' filename prefix for 'trial_kinematics_' unless
    kinematics_path is given explicitly. Never raises on a missing column
    or missing file -- move_samples just comes back None, and the caller
    (TrialFigure.build) already falls back to the trial's own raw sample
    count in that case.
    """
    KEYS = ["Block", "TrialNumInBlock", "Attempt"]

    def __init__(self, path, kinematics_path=None):
        self.frame = pd.read_csv(path) if path and Path(path).exists() else None
        if self.frame is not None:
            self.frame = normalize_trial_schema(self.frame)

        if kinematics_path is None and path:
            guess = Path(path).with_name(Path(path).name.replace("trial_data_", "trial_kinematics_", 1))
            kinematics_path = guess if guess.exists() else None
        self.kin_frame = pd.read_csv(kinematics_path) if kinematics_path and Path(kinematics_path).exists() else None

    @staticmethod
    def _match(frame, block, trial, attempt):
        row = frame[(frame.Block == block) & (frame.TrialNumInBlock == trial) & (frame.Attempt == attempt)]
        return row.iloc[0] if len(row) == 1 else None

    def lookup(self, block, trial, attempt):
        result = {"bar_size_deg": None, "move_samples": None,
                  "direction_correct": None, "is_correct": None, "error_type": None,
                  "decision_time_s": None, "execution_time_s": None, "total_time_s": None}

        if self.frame is not None:
            row = self._match(self.frame, block, trial, attempt)
            if row is not None:
                result["bar_size_deg"] = float(row["BarSizeVA_deg"])
                result["direction_correct"] = str(row["DirectionCorrect"])
                result["is_correct"] = int(row["IsCorrect"])
                result["error_type"] = int(row["ErrorType"])
                if "DecisionTime_s" in row.index:
                    result["decision_time_s"] = float(row["DecisionTime_s"])
                if "ExecutionTime_s" in row.index:
                    result["execution_time_s"] = float(row["ExecutionTime_s"])
                if "TotalTime_s" in row.index:
                    result["total_time_s"] = float(row["TotalTime_s"])
                if "NumMovementSamples" in row.index:
                    result["move_samples"] = int(row["NumMovementSamples"])

        if result["move_samples"] is None and self.kin_frame is not None:
            krow = self._match(self.kin_frame, block, trial, attempt)
            if krow is not None and "NumMovementSamples" in krow.index:
                result["move_samples"] = int(krow["NumMovementSamples"])

        return result


def estimate_screen_layout(dataset, info_table, target_dist=TARGET_DIST_PX,
                           target_radius=TARGET_RADIUS_PX, center_radius=CENTER_RADIUS_PX,
                           center_override=None):
    if center_override is not None:
        cx, cy = center_override
    else:
        directions = ["Right_0", "Up_90", "Left_180", "Down_270"]
        ends = {d: [] for d in directions}
        for trial in dataset.trials():
            info = info_table.lookup(trial.block, trial.trial, trial.attempt)
            d = info.get("direction_correct")
            if info.get("is_correct") == 1 and d in ends:
                ends[d].append([trial.x[-1], trial.y[-1]])
        means = {}
        for d, pts in ends.items():
            if pts:
                arr = np.asarray(pts)
                means[d] = (float(arr[:, 0].mean()), float(arr[:, 1].mean()))
        cx = cy = None
        if "Right_0" in means and "Left_180" in means:
            cx = (means["Right_0"][0] + means["Left_180"][0]) / 2.0
        if "Up_90" in means and "Down_270" in means:
            cy = (means["Up_90"][1] + means["Down_270"][1]) / 2.0
        if cx is None:
            xs = [p[0] for p in means.values()]
            cx = float(np.mean(xs)) if xs else None
        if cy is None:
            ys = [p[1] for p in means.values()]
            cy = float(np.mean(ys)) if ys else None
        if cx is None or cy is None:
            return None
    offsets = {"Right_0": (target_dist, 0.0), "Up_90": (0.0, -target_dist),
               "Left_180": (-target_dist, 0.0), "Down_270": (0.0, target_dist)}
    targets = {d: (cx + ox, cy + oy) for d, (ox, oy) in offsets.items()}
    return {"center": (cx, cy), "targets": targets,
            "target_radius": target_radius, "center_radius": center_radius}


class OfflineClockFit:
    SEED_RATE_HZ = 24414.0625 / 26.0

    def __init__(self, index, time_ms, source=""):
        idx = np.asarray(index, dtype=float)
        tms = np.asarray(time_ms, dtype=float)
        self.source = source
        self.n_rows = int(idx.size)
        self.n_indexed = int(np.isfinite(idx).sum())
        ok = np.isfinite(idx) & np.isfinite(tms) & (tms > 0)
        order = np.argsort(idx[ok], kind="stable")
        self.index = idx[ok][order]
        self.time_ms = tms[ok][order]
        self.slope_ms = float("nan")
        self.intercept_ms = float("nan")
        if self.index.size >= 3 and np.ptp(self.index) > 0:
            x = self.index - self.index.mean()
            y = self.time_ms - self.time_ms.mean()
            self.slope_ms = float(np.dot(x, y) / np.dot(x, x))
            self.intercept_ms = float(self.time_ms.mean() - self.slope_ms * self.index.mean())

    @classmethod
    def from_csv(cls, path):
        path = Path(path)
        frame = pd.read_csv(path)
        if "RZ2Idx" not in frame.columns or "Time_ms" not in frame.columns:
            return cls([], [], source=path.name)
        return cls(frame["RZ2Idx"].to_numpy(), frame["Time_ms"].to_numpy(), source=path.name)

    @property
    def fitted(self):
        return bool(np.isfinite(self.slope_ms) and self.slope_ms > 0)

    @property
    def rate_hz(self):
        return 1000.0 / self.slope_ms if self.fitted else float("nan")

    def predict_ms(self, index):
        return self.intercept_ms + self.slope_ms * np.asarray(index, dtype=float)

    def residual_ms(self):
        return self.time_ms - self.predict_ms(self.index)

    def summary(self):
        nan = float("nan")
        out = {"rows": self.n_rows, "indexed_rows": self.n_indexed,
               "unindexed_rows": self.n_rows - self.n_indexed,
               "rate_hz": self.rate_hz,
               "rate_dev_ppm": (self.rate_hz / self.SEED_RATE_HZ - 1.0) * 1e6 if self.fitted else nan,
               "resid_rms_ms": nan, "resid_p99_abs_ms": nan, "resid_max_abs_ms": nan}
        if self.fitted:
            r = self.residual_ms()
            out["resid_rms_ms"] = float(np.sqrt(np.mean(r ** 2)))
            out["resid_p99_abs_ms"] = float(np.percentile(np.abs(r), 99))
            out["resid_max_abs_ms"] = float(np.max(np.abs(r)))
        steps = np.diff(self.time_ms)
        back = steps[steps < 0]
        out["backward_steps"] = int(back.size)
        out["worst_backward_ms"] = float(-back.min()) if back.size else 0.0
        return out

    def report(self):
        if not self.fitted:
            return (f"Offline clock fit: no usable RZ2Idx rows in '{self.source}' "
                    f"(USB or mouse input, or an export older than v8.21); "
                    f"Time_ms is the only time base.")
        s = self.summary()
        return (f"Offline clock fit on '{self.source}' (OLS of Time_ms on RZ2Idx, "
                f"{s['indexed_rows']}/{s['rows']} rows indexed):\n"
                f"  rate {s['rate_hz']:.4f} Hz ({s['rate_dev_ppm']:+.0f} ppm vs seed "
                f"{self.SEED_RATE_HZ:.4f} Hz)\n"
                f"  online Time_ms minus offline fit: RMS {s['resid_rms_ms']:.2f} ms, "
                f"p99 |r| {s['resid_p99_abs_ms']:.2f} ms, max |r| {s['resid_max_abs_ms']:.2f} ms\n"
                f"  backward steps of Time_ms in index order: {s['backward_steps']} "
                f"(worst {s['worst_backward_ms']:.2f} ms)")


ACCEL_CMAP = plt.get_cmap("viridis")


class TrialFigure:
    X_COLOR = "#75d054"
    Y_COLOR = "#440154"
    OUTLIER_COLOR = "#E69F00"
    SCREEN_BG = "#0a0a0a"
    PATH_COLOR = "#c8c8c8"
    RAW_PATH_COLOR = "#5a5a5a"
    START_COLOR = "#009E73"
    TARGET_COLOR = "#e8e8e8"
    END_COLOR = "#D55E00"
    START_POINT_COLOR = "#20a386"
    END_POINT_COLOR = "#fde725"
    ACCEL_CMAP = ACCEL_CMAP

    def __init__(self, time_lim=None, pos_lim=None, space_lim=None, tick_step_ms=100,
                 layout=None, accel_vmax=None, hold_ref_ms=None, target_radius_px=TARGET_RADIUS_PX,
                 input_source="unknown"):
        self.time_lim = time_lim
        self.pos_lim = pos_lim
        self.space_lim = space_lim
        self.tick_step_ms = tick_step_ms
        self.layout = layout
        self.hold_ref_ms = hold_ref_ms
        self.accel_vmax = accel_vmax
        self.input_source = input_source
        self.target_radius_px = target_radius_px

    def _apply_axes(self, ax, apply_ylim=True):
        if self.time_lim is not None:
            ax.set_xlim(*self.time_lim)
            ax.xaxis.set_major_locator(MultipleLocator(self.tick_step_ms))
            ax.xaxis.set_minor_locator(MultipleLocator(self.tick_step_ms / 2.0))
            ax.tick_params(axis="x", labelsize=7, rotation=0)
        if apply_ylim and self.pos_lim is not None:
            ax.set_ylim(*self.pos_lim)
        ax.grid(False)
        ax.tick_params(which="both", length=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    @staticmethod
    def _outward_align(pos, anchor):
        dx = pos[0] - anchor[0]
        dy = pos[1] - anchor[1]
        if dx > 12:
            ha = "left"
        elif dx < -12:
            ha = "right"
        else:
            ha = "center"
        if dy > 12:
            va = "top"
        elif dy < -12:
            va = "bottom"
        else:
            va = "center"
        return ha, va

    @staticmethod
    def _best_label_pos(anchor, radii, point_obstacles, circle_obstacles, bounds, n_angles=48, cap=45.0):
        if not isinstance(radii, (list, tuple)):
            radii = [radii]
        ax0, ay0 = anchor
        (xmin, xmax), (ymin, ymax) = bounds
        pts = np.asarray(point_obstacles, dtype=float) if len(point_obstacles) else np.empty((0, 2))
        best = (ax0 + radii[0], ay0)
        best_score = -1e18
        for r in radii:
            for k in range(n_angles):
                ang = 2.0 * np.pi * k / n_angles
                x = ax0 + r * np.cos(ang)
                y = ay0 + r * np.sin(ang)
                if not (xmin + 45 <= x <= xmax - 45 and ymin + 25 <= y <= ymax - 25):
                    continue
                clearance = 1e18
                if pts.shape[0]:
                    clearance = float(np.min(np.hypot(pts[:, 0] - x, pts[:, 1] - y)))
                for ccx, ccy, rr in circle_obstacles:
                    clearance = min(clearance, abs(float(np.hypot(x - ccx, y - ccy)) - rr))
                score = min(clearance, cap) - 0.05 * r
                if score > best_score:
                    best_score = score
                    best = (x, y)
        return best

    def _draw_path(self, ax, cbar_ax, trial, result, info):
        from matplotlib.patches import Circle
        rx, ry = result.raw_xy[:, 0], result.raw_xy[:, 1]
        gx, gy = result.grid_xy[:, 0], result.grid_xy[:, 1]
        ax.set_facecolor(self.SCREEN_BG)
        target_radius = 100.0
        correct_dir = info.get("direction_correct")
        correct_pos = None
        if self.layout is not None:
            cx, cy = self.layout["center"]
            center_radius = self.layout.get("center_radius", 100.0)
            target_radius = self.layout.get("target_radius", 100.0)
            ax.add_patch(Circle((cx, cy), center_radius, fill=False,
                                edgecolor="#666666", lw=1.0, zorder=1))
            for d, (tx, ty) in self.layout["targets"].items():
                is_correct = (d == correct_dir)
                edge = self.START_COLOR if is_correct else "#4a4a4a"
                lw = 2.2 if is_correct else 1.0
                ax.add_patch(Circle((tx, ty), target_radius, fill=False,
                                    edgecolor=edge, lw=lw, zorder=1))
                if is_correct:
                    correct_pos = (tx, ty)
        ax.plot(rx, ry, color=self.RAW_PATH_COLOR, lw=0.8, alpha=0.6, zorder=2)
        accel = result.accel_cm
        n_segments = max(gx.size - 1, 0)
        colorable = n_segments > 0 and accel.size == n_segments
        if colorable:
            points = np.column_stack([gx, gy]).reshape(-1, 1, 2)
            segments = np.concatenate([points[:-1], points[1:]], axis=1)
            shared_scale = (self.accel_vmax is not None and np.isfinite(self.accel_vmax)
                            and self.accel_vmax > 0)
            if shared_scale:
                vmax = self.accel_vmax
            else:
                local_max = float(np.nanmax(accel)) if np.isfinite(accel).any() else 0.0
                vmax = max(local_max, 1e-6)
            norm = Normalize(vmin=0.0, vmax=vmax)
            lc = LineCollection(segments, array=accel, cmap=self.ACCEL_CMAP,
                                norm=norm, linewidths=2.6, zorder=3)
            ax.add_collection(lc)
            cbar = ax.figure.colorbar(lc, cax=cbar_ax, orientation="horizontal")
            label = "Acceleration (cm/s^2)" if shared_scale else "Acceleration (cm/s^2) -- per-trial scale"
            cbar.set_label(label, color="#555555", fontsize=8)
            cbar.ax.tick_params(colors="#888888", labelsize=7, length=0)
            cbar.outline.set_edgecolor("#cccccc")
        else:
            ax.plot(gx, gy, color=self.PATH_COLOR, lw=2.0, zorder=3)
            cbar_ax.axis("off")
        ax.scatter([gx[0]], [gy[0]], s=60, facecolor=self.START_POINT_COLOR,
                edgecolor="white", linewidth=0.8, zorder=4)
        ax.scatter([gx[-1]], [gy[-1]], s=70, facecolor=self.END_POINT_COLOR,
                edgecolor="white", linewidth=0.8, zorder=6)
        if np.isfinite(result.move_offset_ms) and result.grid_time_ms.size:
            _ai = int(np.argmin(np.abs(result.grid_time_ms - result.move_offset_ms)))
            ax.scatter([gx[_ai]], [gy[_ai]], s=55, marker="D", facecolor="#ffffff",
                    edgecolor="black", linewidth=0.8, zorder=7)
        if self.space_lim is not None:
            bounds = self.space_lim
        else:
            bounds = ((float(min(gx.min(), rx.min())), float(max(gx.max(), rx.max()))),
                    (float(min(gy.min(), ry.min())), float(max(gy.max(), ry.max()))))
        traj_pts = np.column_stack([np.concatenate([gx, rx]), np.concatenate([gy, ry])])
        if traj_pts.shape[0] > 60:
            idx = np.linspace(0, traj_pts.shape[0] - 1, 60).astype(int)
            traj_pts = traj_pts[idx]
        target_label_pos = None
        if self.layout is not None:
            all_circles = [(cx, cy, center_radius)]
            all_circles += [(tx, ty, target_radius) for (tx, ty) in self.layout["targets"].values()]
            if correct_pos is None and correct_dir in self.layout["targets"]:
                correct_pos = self.layout["targets"][correct_dir]
            if correct_pos is not None:
                other_circles = [(cx, cy, center_radius)]
                other_circles += [(tx, ty, target_radius)
                                for d, (tx, ty) in self.layout["targets"].items() if d != correct_dir]
                pts = list(traj_pts) + [(gx[0], gy[0])]
                target_label_pos = self._best_label_pos(correct_pos, [target_radius + 30, target_radius + 52],
                                                        pts, other_circles, bounds)
                tha, tva = self._outward_align(target_label_pos, correct_pos)
                ax.text(target_label_pos[0], target_label_pos[1], "target",
                        color=self.START_COLOR, fontsize=8, ha=tha, va=tva, zorder=5)
            start_pts = list(traj_pts)
            if target_label_pos is not None:
                start_pts.append(target_label_pos)
            start_label_pos = self._best_label_pos((gx[0], gy[0]), [58, 82, 108, 136], start_pts, all_circles, bounds)
            sha, sva = self._outward_align(start_label_pos, (gx[0], gy[0]))
        else:
            start_label_pos = (gx[0], gy[0] - 40)
            sha, sva = "center", "center"
        ax.text(start_label_pos[0], start_label_pos[1], "start", color="#dddddd",
                fontsize=8, ha=sha, va=sva, zorder=5)
        if self.space_lim is not None:
            ax.set_xlim(*self.space_lim[0])
            ax.set_ylim(*self.space_lim[1])
        ax.invert_yaxis()
        ax.set_aspect("equal", adjustable="box")
        ax.set_title("Screen path: center to target", fontsize=11, loc="left")
        ax.set_xlabel("X (px)")
        ax.set_ylabel("Y (px)")
        ax.grid(False)
        ax.tick_params(colors="#888888", labelsize=7, length=0)
        outcome, ocolor = self._outcome(info)
        ax.text(0.03, 0.95, outcome, transform=ax.transAxes, color=ocolor,
                fontsize=11, fontweight="bold", va="top", ha="left")

    @staticmethod
    def _fmt(value, unit):
        return "n/a" if value is None or not np.isfinite(value) else f"{value:.1f} {unit}"

    @staticmethod
    def _outcome(info):
        if info.get("is_correct") == 1:
            return "Correct", "#009E73"
        if info.get("error_type") == 1:
            return "Early exit", "#CC79A7"
        if info.get("error_type") == 2:
            return "Wrong target", "#ffffff"
        if info.get("error_type") == 3:
            return "Hold-break", "#E69F00"
        return "n/a", "#aaaaaa"

    def build(self, trial, result, info):
        t_raw = result.raw_time_ms
        rx, ry = result.raw_xy[:, 0], result.raw_xy[:, 1]
        t_grid = result.grid_time_ms
        fig = plt.figure(figsize=(18.0, 9.5))
        outer = fig.add_gridspec(1, 2, width_ratios=[1.35, 1.0], wspace=0.16)
        left = outer[0, 0].subgridspec(3, 1, hspace=0.38)
        top = fig.add_subplot(left[0])
        bottom = fig.add_subplot(left[1], sharex=top)
        vel_ax = fig.add_subplot(left[2], sharex=top)
        right = outer[0, 1].subgridspec(3, 1, height_ratios=[5.0, 0.4, 2.8], hspace=0.5)
        path = fig.add_subplot(right[0])
        path.set_anchor("N")
        cbar_ax = fig.add_subplot(right[1])
        stats_ax = fig.add_subplot(right[2])
        stats_ax.axis("off")
        if np.isfinite(result.move_takeoff_ms):
            for _i, _ax in enumerate((top, bottom, vel_ax)):
                _ax.axvline(result.move_takeoff_ms, color="#D55E00", lw=1.4, ls="--", alpha=0.9, zorder=1,
                            label=f"takeoff {MOVE_SPEED_FRAC:.0%} of peak" if _i == 0 else None)
        for _t_ms, _lab in ((result.move_onset_ms, "leave-center"),
                            (result.move_offset_ms, "hold onset")):
            if np.isfinite(_t_ms):
                for _i, _ax in enumerate((top, bottom, vel_ax)):
                    _ax.axvline(_t_ms, color="#009E73", lw=1.4, ls="--",
                                alpha=0.9, zorder=1,
                                label=_lab if _i == 0 else None)
        if trial.hold_window_ms is not None:
            _t0 = float(np.min(trial.time_ms))
            _h_lo = trial.hold_window_ms[0] - _t0
            _h_hi = trial.hold_window_ms[1] - _t0
            for _i, _ax in enumerate((top, bottom, vel_ax)):
                _ax.axvspan(_h_lo, _h_hi, color="#009E73", alpha=0.08, zorder=0)
                _ax.axvline(_h_hi, color="#009E73", lw=1.0, ls=":", alpha=0.7, zorder=1,
                            label="hold end" if _i == 0 else None)
        if result.hampel_threshold.any():
            top.fill_between(t_raw,
                            result.hampel_median[:, 0] - result.hampel_threshold[:, 0],
                            result.hampel_median[:, 0] + result.hampel_threshold[:, 0],
                            color=self.X_COLOR, alpha=0.15, label="X MAD band")
            top.fill_between(t_raw,
                            result.hampel_median[:, 1] - result.hampel_threshold[:, 1],
                            result.hampel_median[:, 1] + result.hampel_threshold[:, 1],
                            color=self.Y_COLOR, alpha=0.15, label="Y MAD band")
        top.plot(t_raw, rx, color=self.X_COLOR, lw=1.2, label="X raw")
        top.plot(t_raw, ry, color=self.Y_COLOR, lw=1.2, label="Y raw")
        mx, my = result.outlier_mask[:, 0], result.outlier_mask[:, 1]
        if mx.any():
            top.scatter(t_raw[mx], rx[mx], s=32, color=self.OUTLIER_COLOR, zorder=5,
                        label="Hampel outliers")
        if my.any():
            top.scatter(t_raw[my], ry[my], s=32, color=self.OUTLIER_COLOR, zorder=5)
        top.set_ylabel("Position (px)")
        top.set_title("Raw noisy signal with Hampel MAD band", loc="left")
        top.legend(loc="upper right", fontsize=8, ncol=1, frameon=False)
        self._apply_axes(top)
        bottom.plot(t_raw, rx, color=self.X_COLOR, lw=0.8, alpha=0.25)
        bottom.plot(t_raw, ry, color=self.Y_COLOR, lw=0.8, alpha=0.25)
        bottom.plot(t_grid, result.grid_xy[:, 0], color=self.X_COLOR, lw=1.8, label="X filtered")
        bottom.plot(t_grid, result.grid_xy[:, 1], color=self.Y_COLOR, lw=1.8, label="Y filtered")
        stage_txt = (f"Hampel + Kalman + pchip grid + zero-phase Butterworth "
                     f"(design {CUTOFF_HZ:g} Hz, effective -3 dB {EFFECTIVE_CUTOFF_HZ:.1f} Hz)") if result.butter_applied \
            else "Hampel + Kalman + pchip grid (Butterworth skipped)"
        bottom.set_ylabel("Position (px)")
        bottom.set_title(stage_txt, loc="left")
        bottom.legend(loc="upper right", fontsize=8, frameon=False)
        self._apply_axes(bottom)
        has_vel = result.vel_cm.size > 0
        if has_vel:
            vel_ax.plot(result.vel_time_ms, result.vel_cm, color=self.PATH_COLOR, lw=1.8,
                        label="Speed (filtered)")
            if np.isfinite(result.peak_vel_cm):
                vel_ax.scatter([result.peak_vel_time_ms], [result.peak_vel_cm],
                               s=40, facecolor=self.END_COLOR, edgecolor="white",
                               linewidth=0.8, zorder=5, label="Peak speed")
            if np.isfinite(result.mean_vel_cm):
                vel_ax.axhline(result.mean_vel_cm, color=self.PATH_COLOR, lw=1.0,
                               ls="--", alpha=0.6, label="Mean speed")
            vel_ax.legend(loc="upper right", fontsize=8, frameon=False)
        else:
            vel_ax.text(0.5, 0.5, "not enough samples to differentiate",
                        transform=vel_ax.transAxes, ha="center", va="center",
                        fontsize=9, color="#888888")
        vel_ax.set_xlabel("Time since epoch-window onset (ms)")
        vel_ax.set_ylabel("Speed (cm/s)")
        vel_ax.set_title("Speed profile: |d(x,y)/dt| of the filtered trace", loc="left")
        self._apply_axes(vel_ax, apply_ylim=False)
        vel_ax.set_ylim(bottom=0)
        plt.setp(top.get_xticklabels(), visible=False)
        plt.setp(bottom.get_xticklabels(), visible=False)
        samples = info.get("move_samples")
        samples_txt = str(samples) if samples is not None else str(trial.n_samples)
        def _ts(v):
            return "n/a" if v is None or not np.isfinite(v) else f"{v:.3f} s"
        def _fms(v):
            return "n/a" if v is None or not np.isfinite(v) else f"{v:.0f} ms"
        def _fpx(v):
            return "n/a" if v is None or not np.isfinite(v) else f"{v:.0f} px"
        hold_dur_ms = (trial.hold_window_ms[1] - trial.hold_window_ms[0]) if trial.hold_window_ms else None
        hold_unstable = (trial.hold_excursion_px is not None and np.isfinite(trial.hold_excursion_px)
                        and trial.hold_excursion_px > self.target_radius_px)
        hold_incomplete = (hold_dur_ms is not None and np.isfinite(hold_dur_ms)
                           and self.hold_ref_ms and hold_dur_ms < 0.8 * self.hold_ref_ms)
        takeoff_target_s = ((result.window_end_ms - result.window_start_ms) / 1000.0
                            if np.isfinite(result.window_start_ms) and np.isfinite(result.window_end_ms)
                            else None)
        box = (f"Move samples: {samples_txt}\n"
            f"Peak vel (takeoff-target): {self._fmt(result.peak_vel_cm, 'cm/s')}\n"
            f"Mean vel (takeoff-target): {self._fmt(result.mean_vel_cm, 'cm/s')}\n"
            f"Median vel (takeoff-target): {self._fmt(result.median_vel_cm, 'cm/s')}\n"
            f"Peak acc (takeoff-target, p{ACCEL_PEAK_PERCENTILE:g}, Sav-Gol): {self._fmt(result.peak_accel_cm, 'cm/s^2')}\n"
            f"Mean acc (takeoff-target, Sav-Gol): {self._fmt(result.mean_accel_cm, 'cm/s^2')}\n"
            f"Median acc (takeoff-target, Sav-Gol): {self._fmt(result.median_accel_cm, 'cm/s^2')}\n"
            f"\n"
            f"Decision time: {_ts(info.get('decision_time_s'))}\n"
            f"Execution time: {_ts(info.get('execution_time_s'))}\n"
            f"Total time: {_ts(info.get('total_time_s'))}\n"
            f"Takeoff-to-target time: {_ts(takeoff_target_s)}\n"
            f"\n"
            f"Hold dur: {_fms(hold_dur_ms)}{'  [incomplete]' if hold_incomplete else ''}\n"
            f"Hold max spd: {self._fmt(trial.hold_max_speed_cm, 'cm/s')}\n"
            f"Hold excursion: {_fpx(trial.hold_excursion_px)}{'  [unstable]' if hold_unstable else ''}")
        self._draw_path(path, cbar_ax, trial, result, info)
        stats_ax.text(0.0, 1.0, box, transform=stats_ax.transAxes, fontsize=10,
                    va="top", ha="left")
        fs_txt = "n/a" if not np.isfinite(trial.fs_hz) else f"{trial.fs_hz:.1f} Hz"
        bar = info.get("bar_size_deg")
        bar_txt = f"{bar:.2f} deg" if bar is not None else "n/a"
        outcome, _ = self._outcome(info)
        qc = ("  [under-resolved]" if result.under_resolved else "") \
            + ("  [hold unstable]" if hold_unstable else "") \
            + ("  [hold incomplete]" if hold_incomplete else "") \
            + (f"  [multi-peak x{result.n_speed_peaks}]" if result.n_speed_peaks > 1 else "")
        fig.suptitle(
            f"Joystick trajectory   |   Date {trial.date}   |   Block {trial.block}   |   "
            f"Trial {trial.trial}   |   Attempt {trial.attempt}   |   "
            f"Bar {bar_txt}   |   {outcome}   |   {self.input_source}, fs {fs_txt}{qc}",
            fontsize=12, y=0.99,
        )
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        return fig


def run(traj_path=None, trial_data_path=None, output_dir=OUTPUT_DIR,
        grid_dt_s=GRID_DT_S, cm_per_px=CM_PER_PX,
        show_inline=SHOW_INLINE, fix_time_axis=FIX_TIME_AXIS,
        fix_pos_axis=FIX_POS_AXIS, tick_step_ms=TIME_TICK_STEP_MS, margin=AXIS_MARGIN,
        move_epochs=MOVE_EPOCHS, kinematics_path=None,
        fix_accel_scale=FIX_ACCEL_SCALE, accel_scale_percentile=ACCEL_SCALE_PERCENTILE):
    """
    traj_path         : trajectory_movement_<runTag>.csv (has an Epoch column;
                         see MOVE_EPOCHS above for which codes are kept).
    trial_data_path    : trial_data_<runTag>.csv.
    kinematics_path    : trial_kinematics_<runTag>.csv, optional -- only
                         needed to recover NumMovementSamples on sessions
                         recorded after the CSV split; auto-detected next to
                         trial_data_path if not given (see TrialInfoTable).
    move_epochs        : epoch code(s) to keep from traj_path; defaults to
                         MOVE_EPOCHS = (DECISION_EPOCH, MOVEMENT_EPOCH,
                         TARGET_HOLD_EPOCH). Pass MOVEMENT_EPOCH alone, or
                         (DECISION_EPOCH, MOVEMENT_EPOCH), to reproduce an
                         earlier window, or None to disable filtering
                         entirely. Only ever narrows what's already IN
                         traj_path -- see the MOVE_EPOCHS comment above about
                         epochs missing from older exports.
    fix_accel_scale    : share one acceleration->color scale across every
                         figure (see FIX_ACCEL_SCALE above); False scales each
                         figure to its own peak acceleration instead.
    accel_scale_percentile : percentile of the pooled acceleration
                         distribution used as the shared colorbar ceiling when
                         fix_accel_scale is True (see ACCEL_SCALE_PERCENTILE).
    """
    traj_path = traj_path or TRAJ_PATH
    trial_data_path = trial_data_path or TRIAL_DATA_PATH
    _invalid = globals().get("invalid_reason")
    _reason = (_invalid(Path(str(trial_data_path)).stem.replace("trial_data_", "", 1))
               if callable(_invalid) else None)
    if _reason:
        print(f"WARNING: this session is listed in INVALID_SESSIONS: {_reason}")
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    dataset = TrajectoryDataset(traj_path, move_epochs=move_epochs)
    info_table = TrialInfoTable(trial_data_path, kinematics_path=kinematics_path)
    full_guess = Path(traj_path).with_name(Path(traj_path).name.replace("trajectory_movement_", "trajectory_", 1))
    if dataset.input_source == InputSourceDetector.RZ2:
        clock = OfflineClockFit.from_csv(full_guess if full_guess.is_file() else traj_path)
        print(clock.report())
    else:
        print(f"Offline clock fit skipped: input source is {dataset.input_source}, "
              f"so Time_ms is the only time base.")
    time_lim, pos_lim, over_cap_trials = dataset.global_limits(margin=margin, max_time_ms=MAX_TRAJ_TIME_MS)
    center_override = (SCREEN_WIDTH_PX / 2.0, SCREEN_HEIGHT_PX / 2.0) if USE_SCREEN_CENTER else None
    geometry = ScreenGeometry.resolve(trial_data_path, globals().get("GEOMETRY_SPEC_PATH"))
    print(geometry.describe())
    layout = estimate_screen_layout(dataset, info_table, target_dist=geometry.target_dist_px,
                                    target_radius=geometry.target_radius_px,
                                    center_radius=geometry.center_radius_px,
                                    center_override=center_override)
    if layout is not None:
        cx, cy = layout["center"]
        if SCREEN_VIEW_FULL:
            space_lim = ((0.0, float(SCREEN_WIDTH_PX)), (0.0, float(SCREEN_HEIGHT_PX)))
        elif SCREEN_HALF_EXTENT_PX is not None:
            reach = float(SCREEN_HALF_EXTENT_PX)
            space_lim = ((cx - reach, cx + reach), (cy - reach, cy + reach))
        else:
            reach = 0.0
            for trial in dataset.trials():
                reach = max(reach, float(np.max(np.abs(trial.x - cx))),
                            float(np.max(np.abs(trial.y - cy))))
            reach *= 1.06
            space_lim = ((cx - reach, cx + reach), (cy - reach, cy + reach))
    else:
        space_lim = dataset.global_space_limits()
    processor = TrajectoryProcessor.for_profile(dataset.profile, grid_dt_s=grid_dt_s,
                                                cutoff_hz=CUTOFF_HZ, cm_per_px=cm_per_px)
    processed = []
    n_dup_rows = 0
    n_dup_trials = 0
    all_accels = []
    hold_durs = []
    multi_peak_trials = []
    for trial in dataset.trials():
        result = processor.process(trial.time_ms, trial.x, trial.y,
                                   trial.move_window_ms, trial.hold_start_ms)
        info = info_table.lookup(trial.block, trial.trial, trial.attempt)
        processed.append((trial, result, info))
        n_dup_rows += result.n_duplicate_ts
        n_dup_trials += int(result.n_duplicate_ts > 0)
        if result.n_speed_peaks > 1:
            multi_peak_trials.append((trial.block, trial.trial, trial.attempt, result.n_speed_peaks))
        if result.accel_cm.size:
            all_accels.append(result.accel_cm)
        if trial.hold_window_ms is not None:
            hold_durs.append(trial.hold_window_ms[1] - trial.hold_window_ms[0])
    hold_ref_ms = float(np.median(hold_durs)) if hold_durs else None
    print(f"Duplicate timestamps dropped before filtering: {n_dup_rows} rows in "
          f"{n_dup_trials}/{len(processed)} trials (first occurrence of each time kept).")
    accel_vmax = None
    if fix_accel_scale and all_accels:
        pooled = np.concatenate(all_accels)
        pooled = pooled[np.isfinite(pooled)]
        if pooled.size:
            accel_vmax = float(np.percentile(pooled, accel_scale_percentile))
            print(f"Acceleration color scale (path panel): 0-{accel_vmax:.1f} cm/s^2 "
                  f"({accel_scale_percentile:g}th percentile across {len(all_accels)} trials).")
    drawer = TrialFigure(
        time_lim=time_lim if fix_time_axis else None,
        pos_lim=pos_lim if fix_pos_axis else None,
        space_lim=space_lim,
        tick_step_ms=tick_step_ms,
        layout=layout,
        accel_vmax=accel_vmax,
        input_source=dataset.input_source,
        hold_ref_ms=hold_ref_ms,
        target_radius_px=geometry.target_radius_px,
    )
    saved = []
    for trial, result, info in processed:
        fig = drawer.build(trial, result, info)
        path = out / f"{trial.date}_{trial.label}.png"
        fig.savefig(path, dpi=FIGURE_DPI)
        saved.append(path)
        if show_inline:
            plt.show()
        plt.close(fig)
    zip_path = "trial_figures.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in saved:
            zf.write(p, p.name)
    print(f"Generated {len(saved)} figures in '{output_dir}/' and packed '{zip_path}'.")
    if MAX_TRAJ_TIME_MS is not None:
        if over_cap_trials:
            print(f"{len(over_cap_trials)} trial(s) exceeded the {MAX_TRAJ_TIME_MS / 1000:.0f}s time-axis "
                  f"cap (plotted but clipped at {MAX_TRAJ_TIME_MS / 1000:.0f}s):")
            for _block, _trial, _attempt, _t_max in over_cap_trials:
                print(f"  - Block {_block}, TrialNumInBlock {_trial}, Attempt {_attempt}: {_t_max / 1000:.2f}s")
        else:
            print(f"No trials exceeded the {MAX_TRAJ_TIME_MS / 1000:.0f}s time-axis cap.")
    if multi_peak_trials:
        print(f"{len(multi_peak_trials)} trial(s) have more than one local speed peak "
              f"(candidate corrections / changes of mind, TAKEOFF_PEAK_CHOICE={TAKEOFF_PEAK_CHOICE!r}):")
        for _block, _trial, _attempt, _n_peaks in multi_peak_trials:
            print(f"  - Block {_block}, TrialNumInBlock {_trial}, Attempt {_attempt}: {_n_peaks} peaks")
    else:
        print("No trials with more than one local speed peak.")
    return zip_path

In [ ]:
ZIP_PATH = run()
try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception:
    pass

In [ ]:
"""Trajectory EDA. Category-agnostic: the per-trial figures are driven entirely
by the CSV columns (DirectionCorrect / IsCorrect / ErrorType / BarSizeVA_deg)
and make no assumption about how many categories the session used, so the same
script handles a 2-category (Short/Long), 3-category (Short/Mid/Long), or any
other configuration. The screen-path panel draws all four cardinal target
positions as fixed reference circles regardless of how many were actually
shown.

## 1. Signal Processing and Data Quality
- Analyze the efficacy of the filtering techniques applied to the continuous
  joystick coordinates.
- Compare the raw signal (TakeRZ2JoystickSamples.m) against the outputs of the
  signal cleaning modules.
- Evaluate specific noise attenuation and smoothing methods, such as low-pass
  filtering (ButterworthLowpass.m), trajectory tracking/smoothing
  (KalmanTrajectorySmoother.m), and spatial anomaly detection
  (HampelOutlierDetector.m).
"""


import glob
import json
import zipfile
from dataclasses import dataclass, field
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.signal import lfilter, savgol_filter, find_peaks
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import MultipleLocator
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize

GRID_DT_S = 0.008
CUTOFF_HZ = 6.0
ACCEL_PEAK_PERCENTILE = 99.0
SAVGOL_WINDOW = 11
SAVGOL_POLYORDER = 3
# Movement-onset ("takeoff") threshold. Adjustable from the config cell in
# section 0.2 -- this is only the fallback default for running this cell
# standalone. move_takeoff_ms (used everywhere downstream, including the
# window search below) is retrospective: it is found by first locating the
# peak speed in the search window, then walking BACKWARD from that peak to
# the first sample still above TAKEOFF_SPEED_FRAC_CHOICE's fraction of it
# (see _takeoff_index below). Only one threshold is ever active: whichever
# of "5%" / "10%" is chosen is both the reference line drawn on the
# trajectory figures and the one used for the window -- the two are no
# longer shown or computed side by side.
TAKEOFF_SPEED_FRAC_OPTIONS = {"5%": 0.05, "10%": 0.10}
TAKEOFF_SPEED_FRAC_CHOICE = globals().get("TAKEOFF_SPEED_FRAC_CHOICE", "5%")
MOVE_SPEED_FRAC = TAKEOFF_SPEED_FRAC_OPTIONS[TAKEOFF_SPEED_FRAC_CHOICE]
# Which local speed peak anchors the takeoff walk-back, when a trial has more
# than one (e.g. a correction / change of mind -- see 1.9): "largest" (default,
# original behaviour) anchors to the tallest peak in the window, which can sit
# on a LATER bump; "first" anchors to the earliest peak that clears
# MULTI_PEAK_PROMINENCE_FRAC of the window's own max speed, so takeoff lands on
# the first real movement instead. Single-peak trials are unaffected either way.
TAKEOFF_PEAK_CHOICE = globals().get("TAKEOFF_PEAK_CHOICE", "largest")  # "largest" or "first"
MULTI_PEAK_PROMINENCE_FRAC = globals().get("MULTI_PEAK_PROMINENCE_FRAC", 0.15)
PIXEL_PITCH_MM = 0.3108
SHOW_INLINE = True
OUTPUT_DIR = "figures"
FIX_TIME_AXIS = True
FIX_POS_AXIS = True
TIME_TICK_STEP_MS = 300
MAX_TRAJ_TIME_MS = 4000.0  # cap the shared trajectory time axis at 4s so a few long outlier trials don't stretch it
AXIS_MARGIN = 0.03
SCREEN_HALF_EXTENT_PX = 750

FIX_ACCEL_SCALE = True
ACCEL_SCALE_PERCENTILE = 99.0

DECISION_EPOCH = 6
MOVEMENT_EPOCH = 7
TARGET_HOLD_EPOCH = 8
MOVE_EPOCHS = (DECISION_EPOCH, MOVEMENT_EPOCH)

_EPOCH_NAME_TO_CODE = {
    "REACTION": DECISION_EPOCH, "DECISION": DECISION_EPOCH,
    "DECISION_TIME": DECISION_EPOCH, "DECISIONTIME": DECISION_EPOCH,
    "MOVEMENT": MOVEMENT_EPOCH, "MOVE": MOVEMENT_EPOCH, "EXECUTION": MOVEMENT_EPOCH,
    "TARGET_HOLD": TARGET_HOLD_EPOCH, "TARGETHOLD": TARGET_HOLD_EPOCH, "HOLD": TARGET_HOLD_EPOCH,
}


def _epoch_to_code(v):
    """Numeric TaskEpoch code from a numeric OR string Epoch cell (NaN if unknown)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    try:
        return int(v)
    except (ValueError, TypeError):
        return _EPOCH_NAME_TO_CODE.get(str(v).strip().upper(), np.nan)

CENTER_TO_TARGET_DIST_PX = 320
TARGET_DIST_SCALE = 1.27
TARGET_DIAMETER_PX = 180
CENTER_DIAMETER_PX = 200
TARGET_DIST_PX = CENTER_TO_TARGET_DIST_PX * TARGET_DIST_SCALE
TARGET_RADIUS_PX = TARGET_DIAMETER_PX / 2.0
CENTER_RADIUS_PX = CENTER_DIAMETER_PX / 2.0
SCREEN_WIDTH_PX = 1920
SCREEN_HEIGHT_PX = 1080
USE_SCREEN_CENTER = True
SCREEN_VIEW_FULL = True

CM_PER_PX = PIXEL_PITCH_MM / 10.0


def effective_cutoff_hz(cutoff_hz=CUTOFF_HZ, fs_hz=1.0 / GRID_DT_S, order=2, n_passes=2):
    ratio = (2.0 ** (1.0 / n_passes) - 1.0) ** (1.0 / (2.0 * order))
    return float(fs_hz / np.pi * np.arctan(ratio * np.tan(np.pi * cutoff_hz / fs_hz)))


EFFECTIVE_CUTOFF_HZ = effective_cutoff_hz()

GEOMETRY_SPEC_GLOBS = ["geometry_spec*.json", "geometry_spec*.csv"]
GEOMETRY_KEYS = {
    "CENTER_TO_TARGET_DIST_PX": "center_to_target_dist_px",
    "TARGET_DIST_SCALE": "target_dist_scale",
    "TARGET_DIAMETER_PX": "target_diameter_px",
    "CENTER_DIAMETER_PX": "center_diameter_px",
}


@dataclass
class ScreenGeometry:
    center_to_target_dist_px: float = CENTER_TO_TARGET_DIST_PX
    target_dist_scale: float = TARGET_DIST_SCALE
    target_diameter_px: float = TARGET_DIAMETER_PX
    center_diameter_px: float = CENTER_DIAMETER_PX
    source: str = "defaults (module constants)"

    @property
    def target_dist_px(self):
        return self.center_to_target_dist_px * self.target_dist_scale

    @property
    def target_radius_px(self):
        return self.target_diameter_px / 2.0

    @property
    def center_radius_px(self):
        return self.center_diameter_px / 2.0

    @staticmethod
    def _read_pairs(path):
        path = Path(path)
        if path.suffix.lower() == ".json":
            with open(path) as fh:
                data = json.load(fh)
            return {str(k).strip().upper(): v for k, v in data.items()}
        raw = pd.read_csv(path, header=None, dtype=str)
        first_col = raw.iloc[:, 0].astype(str).str.strip().str.upper()
        if raw.shape[1] >= 2 and first_col.isin(list(GEOMETRY_KEYS)).any():
            return dict(zip(first_col, raw.iloc[:, 1]))
        if len(raw) < 2:
            return {}
        header = raw.iloc[0].astype(str).str.strip().str.upper()
        return dict(zip(header, raw.iloc[1]))

    @classmethod
    def from_file(cls, path):
        geometry = cls()
        try:
            pairs = cls._read_pairs(path)
        except Exception as exc:
            print(f"WARNING: could not read geometry spec '{path}' ({exc}); using the defaults.")
            return geometry
        missing = []
        for key, attr in GEOMETRY_KEYS.items():
            value = pd.to_numeric(pd.Series([pairs.get(key)]), errors="coerce").iloc[0]
            if pd.notna(value) and np.isfinite(value):
                setattr(geometry, attr, float(value))
            else:
                missing.append(key)
        geometry.source = f"'{Path(path).name}'"
        if missing:
            geometry.source += f" (defaults kept for {missing})"
        return geometry

    @classmethod
    def from_session_params(cls, session_params):
        """Geometry read straight from that session's params_sess*.mat
        (orgParams.centerRad / targetRad / centerToTargetDist), the real
        source of truth for the screen the task actually rendered that day --
        see load_session_params in section 0.1. geometry_spec*.json/csv
        (from_file, above) is kept only as a manual-override path for
        sessions that predate params_sess exports."""
        geometry = cls()
        keys = {
            "center_to_target_dist_px": "center_to_target_dist_px",
            "target_diameter_px": "target_diameter_px",
            "center_diameter_px": "center_diameter_px",
        }
        missing = []
        for src_key, attr in keys.items():
            value = session_params.get(src_key)
            if value is not None and np.isfinite(value):
                setattr(geometry, attr, float(value))
            else:
                missing.append(src_key)
        geometry.source = f"orgParams in '{Path(session_params['source']).name}'"
        if missing:
            geometry.source += f" (defaults kept for {missing})"
        return geometry

    @classmethod
    def resolve(cls, trial_data_path=None, explicit_path=None, patterns=GEOMETRY_SPEC_GLOBS,
               session_params=None):
        session_params = globals().get("SESSION_PARAMS") if session_params is None else session_params
        if session_params and session_params.get("center_to_target_dist_px") is not None:
            return cls.from_session_params(session_params)
        candidates = []
        if trial_data_path:
            folder = Path(trial_data_path).parent
            for pattern in patterns:
                candidates.extend(sorted(folder.glob(pattern)))
        if explicit_path and Path(explicit_path).suffix.lower() != ".mat":
            candidates.append(Path(explicit_path))
        for cand in candidates:
            if Path(cand).is_file():
                return cls.from_file(cand)
        return cls()

    def describe(self):
        return (f"Screen geometry from {self.source}: centre diameter {self.center_diameter_px:g} px, "
                f"target diameter {self.target_diameter_px:g} px, centre-to-target "
                f"{self.center_to_target_dist_px:g} px x {self.target_dist_scale:g} = "
                f"{self.target_dist_px:g} px.")

FONT_CANDIDATES = ["fonts/harding.ttf", "fonts/Harding.ttf"]
FONT_GLOBS = ["/content/*/fonts/harding.ttf", "/content/*/fonts/Harding.ttf"]


def _resolve_font():
    for cand in FONT_CANDIDATES:
        if Path(cand).exists():
            return cand
    for pattern in FONT_GLOBS:
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits[0]
    return None


def _register_font(path):
    try:
        if path and Path(path).exists():
            fm.fontManager.addfont(path)
            plt.rcParams["font.family"] = fm.FontProperties(fname=path).get_name()
            return True
    except Exception:
        pass
    return False


plt.rcParams["axes.unicode_minus"] = False
FONT_PATH = _resolve_font()
if _register_font(FONT_PATH):
    print(f"Using font: {FONT_PATH}")
else:
    print("Harding font not found (looked in fonts/harding.ttf and /content/*/fonts/); using default font.")


FIGURE_DPI = 300
BASE_FONT_SIZE = 10

# Figure ink, storytelling-with-data style: everything that is context is grey,
# colour is spent only on the mark the reader should look at. Category colours
# stay the rig's (ColorCategoryMap.m), because a figure and the screen must agree.
INK = "#333333"          # text, emphasised lines
GRAY = "#8c8c8c"         # context marks, error bars, reference lines
GRAY_LIGHT = "#d9d9d9"   # de-emphasised fills, bands, chance lines
ACCENT = "#31688e"       # the one series the panel is about
ACCENT_2 = "#d44842"     # a second, warm accent (incorrect, a contrasting series)


def apply_figure_style(dpi=FIGURE_DPI, base=BASE_FONT_SIZE):
    """One font, one export resolution and one ink for every figure.

    rcParams are global to the kernel, so calling this once here -- before
    anything is drawn -- is what makes the whole notebook consistent instead
    of each cell setting its own sizes. savefig.dpi is set as well as the
    explicit dpi= arguments, so even a figure saved without one comes out at
    the same resolution. font.family is left alone: it was just set from
    harding.ttf above, and overriding it here would undo that.

    The chrome follows the storytelling-with-data rules: no top/right spines,
    grey hairline axes and ticks, no grid, frameless legends, left-aligned
    titles, so the data ink is the darkest thing on the page.
    """
    plt.rcParams.update({
        "figure.dpi": 110,
        "savefig.dpi": dpi,
        "savefig.bbox": "tight",
        "font.size": base,
        "axes.titlesize": base + 1,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "figure.titlesize": base + 2,
        "axes.unicode_minus": False,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": GRAY,
        "axes.linewidth": 0.8,
        "axes.labelcolor": INK,
        "axes.titlecolor": INK,
        "axes.titlelocation": "left",
        "axes.grid": False,
        "xtick.color": GRAY,
        "ytick.color": GRAY,
        "xtick.labelcolor": INK,
        "ytick.labelcolor": INK,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "text.color": INK,
        "legend.frameon": False,
        "lines.linewidth": 1.6,
        "lines.markersize": 5,
        "errorbar.capsize": 0,
    })


apply_figure_style()


class HampelScreen:
    MAD_SCALE = 1.4826

    def __init__(self, half_window=3, n_sigma=3.0):
        self.half_window = max(1, int(round(half_window)))
        self.n_sigma = float(n_sigma)

    def detect(self, x):
        x = np.atleast_2d(np.asarray(x, dtype=float))
        if x.shape[0] == 1 and x.shape[1] > 1:
            x = x.T
        n, d = x.shape
        cleaned = x.copy()
        median_env = x.copy()
        threshold_env = np.zeros_like(x)
        mask = np.zeros_like(x, dtype=bool)
        if n < 3:
            return cleaned, mask, median_env, threshold_env
        h = self.half_window
        for col in range(d):
            source = x[:, col]
            for i in range(n):
                lo = max(0, i - h)
                hi = min(n, i + h + 1)
                window = source[lo:hi]
                med = np.median(window)
                mad = self.MAD_SCALE * np.median(np.abs(window - med))
                thr = self.n_sigma * mad
                median_env[i, col] = med
                threshold_env[i, col] = thr
                if mad > 0 and abs(source[i] - med) > thr:
                    cleaned[i, col] = med
                    mask[i, col] = True
        return cleaned, mask, median_env, threshold_env


class ButterworthLowpass:
    N_FACT = 6

    def __init__(self, cutoff_hz=20.0):
        self.cutoff_hz = float(cutoff_hz)

    def _design(self, fs):
        k = np.tan(np.pi * self.cutoff_hz / fs)
        norm = 1.0 / (1.0 + np.sqrt(2.0) * k + k ** 2)
        b = np.array([k ** 2, 2.0 * k ** 2, k ** 2]) * norm
        a = np.array([1.0, 2.0 * (k ** 2 - 1.0) * norm, (1.0 - np.sqrt(2.0) * k + k ** 2) * norm])
        return b, a

    @staticmethod
    def _steady_state(b, a):
        companion = np.array([[-a[1], 1.0], [-a[2], 0.0]])
        rhs = np.array([b[1] - a[1] * b[0], b[2] - a[2] * b[0]])
        return np.linalg.solve(np.eye(2) - companion, rhs)

    def __call__(self, x, fs):
        x = np.atleast_2d(np.asarray(x, dtype=float))
        if x.shape[0] == 1 and x.shape[1] > 1:
            x = x.T
        n, d = x.shape
        y = x.copy()
        if n == 0 or fs <= 0 or self.cutoff_hz <= 0 or self.cutoff_hz >= fs / 2.0:
            return y, False
        if n <= self.N_FACT:
            return y, False
        b, a = self._design(fs)
        zi = self._steady_state(b, a)
        nf = self.N_FACT
        for col in range(d):
            series = x[:, col]
            pre = 2.0 * series[0] - series[nf:0:-1]
            post = 2.0 * series[-1] - series[-2:-nf - 2:-1]
            ext = np.concatenate([pre, series, post])
            ext, _ = lfilter(b, a, ext, zi=zi * ext[0])
            ext = ext[::-1]
            ext, _ = lfilter(b, a, ext, zi=zi * ext[0])
            ext = ext[::-1]
            y[:, col] = ext[nf:-nf]
        return y, True


class KalmanTrajectorySmoother:
    def __init__(self, meas_sigma_px=2.0, jerk_sigma_px_s3=1e5, gate_sigma=np.inf):
        self.meas_sigma_px = float(meas_sigma_px)
        self.jerk_sigma_px_s3 = float(jerk_sigma_px_s3)
        self.gate_sigma = float(gate_sigma)

    def __call__(self, t, xy):
        t = np.asarray(t, dtype=float).ravel()
        xy = np.atleast_2d(np.asarray(xy, dtype=float))
        if xy.shape[0] == 1 and xy.shape[1] > 1:
            xy = xy.T
        n = t.size
        smoothed = xy.copy()
        n_gated = 0
        if n < 3 or xy.shape[0] != n:
            return smoothed, n_gated
        r = self.meas_sigma_px ** 2
        q = self.jerk_sigma_px_s3 ** 2
        dt_all = np.diff(t)
        acc_std0 = self.jerk_sigma_px_s3 * 20.0 * np.median(dt_all)
        gate2 = self.gate_sigma ** 2
        for col in range(xy.shape[1]):
            z = xy[:, col]
            xf = np.zeros((3, n))
            pf = np.zeros((3, 3, n))
            xp = np.zeros((3, n))
            pp = np.zeros((3, 3, n))
            fk = np.zeros((3, 3, n))
            dt0 = dt_all[0]
            xp[:, 0] = [z[0], (z[1] - z[0]) / dt0, 0.0]
            pp[:, :, 0] = np.diag([r, 2.0 * r / dt0 ** 2, acc_std0 ** 2])
            fk[:, :, 0] = np.eye(3)
            for k in range(n):
                if k > 0:
                    dt = dt_all[k - 1]
                    f = np.array([[1.0, dt, dt ** 2 / 2.0],
                                  [0.0, 1.0, dt],
                                  [0.0, 0.0, 1.0]])
                    qm = q * np.array([[dt ** 5 / 20.0, dt ** 4 / 8.0, dt ** 3 / 6.0],
                                       [dt ** 4 / 8.0, dt ** 3 / 3.0, dt ** 2 / 2.0],
                                       [dt ** 3 / 6.0, dt ** 2 / 2.0, dt]])
                    fk[:, :, k] = f
                    xp[:, k] = f @ xf[:, k - 1]
                    pp[:, :, k] = f @ pf[:, :, k - 1] @ f.T + qm
                innov = z[k] - xp[0, k]
                s = pp[0, 0, k] + r
                if innov ** 2 <= gate2 * s:
                    gain = pp[:, 0, k] / s
                    xf[:, k] = xp[:, k] + gain * innov
                    pf[:, :, k] = pp[:, :, k] - np.outer(gain, pp[0, :, k])
                    pf[:, :, k] = (pf[:, :, k] + pf[:, :, k].T) / 2.0
                else:
                    xf[:, k] = xp[:, k]
                    pf[:, :, k] = pp[:, :, k]
                    n_gated += 1
            xs = xf.copy()
            for k in range(n - 2, -1, -1):
                f = fk[:, :, k + 1]
                c = pf[:, :, k] @ f.T @ np.linalg.inv(pp[:, :, k + 1])
                xs[:, k] = xf[:, k] + c @ (xs[:, k + 1] - xp[:, k + 1])
            smoothed[:, col] = xs[0, :]
        return smoothed, n_gated


@dataclass
class KinematicSummary:
    peak_vel_cm: float = float("nan")
    mean_vel_cm: float = float("nan")
    median_vel_cm: float = float("nan")
    peak_accel_cm: float = float("nan")
    mean_accel_cm: float = float("nan")
    median_accel_cm: float = float("nan")
    vel_time_ms: np.ndarray = field(default_factory=lambda: np.array([]))
    vel_cm: np.ndarray = field(default_factory=lambda: np.array([]))
    accel_cm: np.ndarray = field(default_factory=lambda: np.array([]))
    move_takeoff_ms: float = float("nan")
    n_speed_peaks: int = 0
    peak_vel_time_ms: float = float("nan")
    window_start_ms: float = float("nan")
    window_end_ms: float = float("nan")


@dataclass
class TrialResult:
    raw_time_ms: np.ndarray
    raw_xy: np.ndarray
    grid_time_ms: np.ndarray
    grid_xy: np.ndarray
    outlier_mask: np.ndarray
    hampel_median: np.ndarray
    hampel_threshold: np.ndarray
    n_replaced: int
    butter_applied: bool
    under_resolved: bool
    peak_vel_cm: float
    mean_vel_cm: float
    median_vel_cm: float
    peak_accel_cm: float
    mean_accel_cm: float
    median_accel_cm: float
    vel_time_ms: np.ndarray
    vel_cm: np.ndarray
    accel_cm: np.ndarray
    move_onset_ms: float = float("nan")
    move_offset_ms: float = float("nan")
    move_takeoff_ms: float = float("nan")
    n_duplicate_ts: int = 0
    n_speed_peaks: int = 0
    peak_vel_time_ms: float = float("nan")
    window_start_ms: float = float("nan")
    window_end_ms: float = float("nan")


class TrajectoryProcessor:
    def __init__(self, grid_dt_s=0.008, cutoff_hz=CUTOFF_HZ, cm_per_px=CM_PER_PX,
                min_move_samples=5, min_move_dur_s=None,
                hampel_half_window=3, hampel_n_sigma=3.0,
                meas_sigma_px=2.0, jerk_sigma_px_s3=1e5, gate_sigma=np.inf):
        self.grid_dt_s = grid_dt_s
        self.cutoff_hz = cutoff_hz
        self.cm_per_px = cm_per_px
        self.min_move_samples = min_move_samples
        self.min_move_dur_s = min_move_dur_s
        self.hampel = HampelScreen(hampel_half_window, hampel_n_sigma)
        self.butter = ButterworthLowpass(cutoff_hz)
        self.kalman = KalmanTrajectorySmoother(meas_sigma_px, jerk_sigma_px_s3, gate_sigma)

    @classmethod
    def for_profile(cls, profile, **kwargs):
        return cls(hampel_half_window=profile.hampel_half_window,
                   hampel_n_sigma=profile.hampel_n_sigma,
                   meas_sigma_px=profile.meas_sigma_px,
                   jerk_sigma_px_s3=profile.jerk_sigma_px_s3,
                   gate_sigma=profile.gate_sigma, **kwargs)

    @staticmethod
    def _savgol_accel(vel_cm, dt_s, window, poly):
        n = int(vel_cm.size)
        if n == 0:
            return np.array([])
        if n == 1:
            return np.array([0.0])
        if n < 5:
            return np.abs(np.gradient(vel_cm, dt_s))
        w = min(int(window), n if n % 2 == 1 else n - 1)
        if w % 2 == 0:
            w -= 1
        w = max(w, 5)
        p = min(int(poly), w - 1)
        return np.abs(savgol_filter(vel_cm, w, p, deriv=1, delta=dt_s))

    def _build_grid(self, t):
        total = t[-1] - t[0]
        if total < self.grid_dt_s:
            return np.array([t[0], t[-1]])
        grid = np.arange(t[0], t[-1] + 1e-12, self.grid_dt_s)
        if t[-1] - grid[-1] > self.grid_dt_s / 4.0:
            grid = np.append(grid, t[-1])
        else:
            grid[-1] = t[-1]
        return grid

    @staticmethod
    def _takeoff_index(vel_cm, pk_idx, frac):
        """Walk backward from an ALREADY-KNOWN peak-speed sample (pk_idx) to
        the first earlier sample still at or above frac * peak.

        This is a retrospective (offline) definition of movement onset: pk_idx
        is the index of the largest speed in the search window, found by
        _kinematics before this is called, so takeoff is only ever set after
        the peak is known. When a trial has more than one local speed peak
        (e.g. a correction / change of mind), this returns the takeoff for
        whichever peak is largest overall, not necessarily the first peak in
        time -- see section 1.9 for trials where that distinction matters.
        """
        thr = frac * float(vel_cm[pk_idx])
        lo_idx = pk_idx
        while lo_idx > 0 and vel_cm[lo_idx - 1] >= thr:
            lo_idx -= 1
        return lo_idx

    def _kinematics(self, grid_time_ms, grid_xy, under_resolved, window_end_ms=None):
        if under_resolved:
            return KinematicSummary()
        t = grid_time_ms / 1000.0
        gdt = np.diff(t)
        if gdt.size < 1 or np.any(gdt <= 0):
            return KinematicSummary()
        vel = np.hypot(np.diff(grid_xy[:, 0]), np.diff(grid_xy[:, 1])) / gdt
        tv_ms = (t[:-1] + gdt / 2.0) * 1000.0
        vel_cm = vel * self.cm_per_px
        accel_cm = self._savgol_accel(vel_cm, float(np.median(gdt)), SAVGOL_WINDOW, SAVGOL_POLYORDER)
        summary = KinematicSummary(vel_time_ms=tv_ms, vel_cm=vel_cm, accel_cm=accel_cm)
        if window_end_ms is not None and np.isfinite(window_end_ms):
            search = tv_ms <= window_end_ms
        else:
            search = np.ones(tv_ms.shape, dtype=bool)
        if not search.any():
            search = np.ones(tv_ms.shape, dtype=bool)
        search_idx = np.where(search)[0]
        seg = vel_cm[search_idx]
        local_peaks = (find_peaks(seg, prominence=MULTI_PEAK_PROMINENCE_FRAC * float(seg.max()))[0]
                      if seg.size >= 3 and seg.max() > 0 else np.array([], dtype=int))
        summary.n_speed_peaks = max(int(local_peaks.size), 1)
        if TAKEOFF_PEAK_CHOICE == "first" and local_peaks.size:
            pk_idx = int(search_idx[local_peaks[0]])
        else:
            pk_idx = int(search_idx[np.argmax(seg)])
        if float(vel[pk_idx]) < 1e-6:
            return summary
        lo_idx = self._takeoff_index(vel_cm, pk_idx, MOVE_SPEED_FRAC)
        summary.move_takeoff_ms = float(tv_ms[lo_idx])
        mask = search & (np.arange(tv_ms.size) >= lo_idx)
        win_time = tv_ms[mask]
        win_vel = vel_cm[mask]
        summary.window_start_ms = float(win_time[0])
        summary.window_end_ms = float(win_time[-1])
        k = int(np.argmax(win_vel))
        summary.peak_vel_cm = float(win_vel[k])
        summary.peak_vel_time_ms = float(win_time[k])
        summary.mean_vel_cm = float(np.mean(win_vel))
        summary.median_vel_cm = float(np.median(win_vel))
        win_acc = accel_cm[mask] if accel_cm.size == vel_cm.size else np.array([])
        win_acc = win_acc[np.isfinite(win_acc)]
        if win_acc.size:
            summary.peak_accel_cm = float(np.percentile(win_acc, ACCEL_PEAK_PERCENTILE))
            summary.mean_accel_cm = float(np.mean(win_acc))
            summary.median_accel_cm = float(np.median(win_acc))
        return summary

    def process(self, time_ms, x, y, move_window_ms=None, hold_start_ms=None):
        t = np.asarray(time_ms, dtype=float) / 1000.0
        order = np.argsort(t, kind="stable")
        t = t[order]
        xy = np.column_stack([np.asarray(x, dtype=float)[order],
                            np.asarray(y, dtype=float)[order]])
        n_raw = t.size
        t_unique, keep = np.unique(t, return_index=True)
        n_duplicate_ts = int(n_raw - t_unique.size)
        t = t_unique
        xy = xy[keep]
        n = t.size
        under_resolved = n < self.min_move_samples
        if self.min_move_dur_s is not None and n >= 2:
            under_resolved = under_resolved or (t[-1] - t[0]) < self.min_move_dur_s
        screened, outlier_mask, median_env, threshold_env = self.hampel.detect(xy)
        n_replaced = int(outlier_mask.sum())
        stage1, _ = self.kalman(t, screened)
        if n < 2:
            grid = t.copy()
            grid_xy = stage1.copy()
            applied = False
        else:
            grid = self._build_grid(t)
            grid_xy = np.column_stack([
                PchipInterpolator(t, stage1[:, 0])(grid),
                PchipInterpolator(t, stage1[:, 1])(grid),
            ])
            grid_fs = 1.0 / self.grid_dt_s
            grid_xy, applied = self.butter(grid_xy, grid_fs)
        grid_time_ms = (grid - t[0]) * 1000.0
        t0_ms = t[0] * 1000.0
        move_window_ms0 = None
        move_onset_ms = float("nan")
        if move_window_ms is not None:
            move_window_ms0 = (move_window_ms[0] - t0_ms, move_window_ms[1] - t0_ms)
            move_onset_ms = move_window_ms0[0]
        move_offset_ms = (hold_start_ms - t0_ms) if hold_start_ms is not None else float("nan")
        window_end_ms = move_window_ms0[1] if move_window_ms0 is not None else None
        kin = self._kinematics(grid_time_ms, grid_xy, under_resolved, window_end_ms)
        return TrialResult(
            raw_time_ms=(t - t[0]) * 1000.0,
            raw_xy=xy,
            grid_time_ms=grid_time_ms,
            grid_xy=grid_xy,
            outlier_mask=outlier_mask,
            hampel_median=median_env,
            hampel_threshold=threshold_env,
            n_replaced=n_replaced,
            butter_applied=applied,
            under_resolved=under_resolved,
            peak_vel_cm=kin.peak_vel_cm,
            mean_vel_cm=kin.mean_vel_cm,
            median_vel_cm=kin.median_vel_cm,
            peak_accel_cm=kin.peak_accel_cm,
            mean_accel_cm=kin.mean_accel_cm,
            median_accel_cm=kin.median_accel_cm,
            vel_time_ms=kin.vel_time_ms,
            vel_cm=kin.vel_cm,
            accel_cm=kin.accel_cm,
            move_onset_ms=move_onset_ms,
            move_offset_ms=move_offset_ms,
            move_takeoff_ms=kin.move_takeoff_ms,
            n_duplicate_ts=n_duplicate_ts,
            n_speed_peaks=kin.n_speed_peaks,
            peak_vel_time_ms=kin.peak_vel_time_ms,
            window_start_ms=kin.window_start_ms,
            window_end_ms=kin.window_end_ms,
        )


INPUT_SOURCE_OVERRIDES = {}
RZ2_MAX_MEDIAN_STEP_MS = 3.0


@dataclass
class PipelineProfile:
    hampel_half_window: int = 3
    hampel_n_sigma: float = 3.0
    meas_sigma_px: float = 2.0
    jerk_sigma_px_s3: float = 1e5
    gate_sigma: float = np.inf
    drop_unindexed_rows: bool = False


PIPELINE_PROFILES = {
    "rz2adc": PipelineProfile(drop_unindexed_rows=True),
    "usb": PipelineProfile(),
    "unknown": PipelineProfile(),
}


class InputSourceDetector:
    RZ2 = "rz2adc"
    USB = "usb"
    UNKNOWN = "unknown"
    GROUP_KEYS = ["Block", "TrialNumInBlock", "Attempt"]
    TAG_PREFIXES = ("trajectory_movement_", "trajectory_", "trial_data_", "trial_kinematics_")

    def __init__(self, overrides=None, max_rz2_step_ms=None):
        self.overrides = INPUT_SOURCE_OVERRIDES if overrides is None else overrides
        self.max_rz2_step_ms = RZ2_MAX_MEDIAN_STEP_MS if max_rz2_step_ms is None else max_rz2_step_ms

    @classmethod
    def run_tag(cls, path):
        name = Path(str(path)).stem
        for prefix in cls.TAG_PREFIXES:
            if name.startswith(prefix):
                return name[len(prefix):]
        return name

    @classmethod
    def median_step_ms(cls, frame):
        if "Time_ms" not in frame.columns:
            return float("nan")
        keys = [k for k in cls.GROUP_KEYS if k in frame.columns]
        d = frame.assign(_t=pd.to_numeric(frame["Time_ms"], errors="coerce")).dropna(subset=["_t"])
        if keys:
            d = d.sort_values(keys + ["_t"], kind="stable")
            steps = d.groupby(keys)["_t"].diff()
        else:
            steps = d["_t"].sort_values(kind="stable").diff()
        steps = steps[steps > 0]
        return float(steps.median()) if len(steps) else float("nan")

    # orgParams.inputSource values (ConfigOrgParams.m) mapped onto this
    # detector's RZ2/USB: 'rz2adc' is the RZ2 analog rig, 'joystick' and
    # 'mouse' are both standard USB HID devices as far as the
    # signal-processing pipeline below is concerned.
    PARAM_SOURCE_MAP = {"rz2adc": "rz2adc", "joystick": "usb", "mouse": "usb"}

    def detect(self, frame, path=None):
        tag = self.run_tag(path) if path is not None else ""
        for key, source in self.overrides.items():
            if key and (key == tag or key in tag):
                return source, "manual override in INPUT_SOURCE_OVERRIDES"
        session_params = globals().get("SESSION_PARAMS") or {}
        raw_source = session_params.get("input_source")
        mapped = self.PARAM_SOURCE_MAP.get(raw_source)
        if mapped is not None:
            fname = Path(session_params["source"]).name
            return mapped, f"orgParams.inputSource='{raw_source}' in '{fname}'"
        if "RZ2Idx" in frame.columns and pd.to_numeric(frame["RZ2Idx"], errors="coerce").notna().any():
            return self.RZ2, "RZ2Idx has indexed rows"
        step = self.median_step_ms(frame)
        if not np.isfinite(step):
            return self.UNKNOWN, "no RZ2Idx values and no usable Time_ms steps"
        source = self.RZ2 if step < self.max_rz2_step_ms else self.USB
        return source, f"no RZ2Idx values; median sample step {step:.2f} ms"


@dataclass
class Trial:
    date: str
    block: int
    trial: int
    attempt: int
    time_ms: np.ndarray
    move_time_ms: np.ndarray
    x: np.ndarray
    y: np.ndarray
    fs_hz: float
    move_window_ms: tuple = None
    hold_start_ms: float = None
    hold_window_ms: tuple = None
    hold_max_speed_cm: float = None
    hold_excursion_px: float = None

    @property
    def n_samples(self):
        return int(self.x.size)

    @property
    def label(self):
        return f"b{self.block:02d}_t{self.trial:02d}_a{self.attempt:02d}"


class TrajectoryDataset:
    GROUP_KEYS = ["Block", "TrialNumInBlock", "Attempt"]

    def __init__(self, csv_path, move_epochs=None, detector=None):
        """
        move_epochs: None = no epoch filtering (use every row in csv_path
        as-is). Otherwise a single TaskEpoch code (e.g. MOVEMENT_EPOCH) or
        an iterable of codes (e.g. MOVE_EPOCHS = (DECISION_EPOCH,
        MOVEMENT_EPOCH, TARGET_HOLD_EPOCH)) -- a row is kept when its Epoch
        matches ANY of them. This mirrors CenterOutTask.m's kinematicsEpochs
        / the ismember(...) filter TrialKinematics.m and
        SaveMovementTrajectory.m now use on the MATLAB side, applied here
        explicitly instead of trusting the CSV to already be restricted the
        way we expect. Note this can only ever KEEP rows that exist in
        csv_path already -- if the session was recorded before
        kinematicsEpochs included a given epoch, that epoch's rows were
        never exported and this filter finds nothing to add for it (see the
        module-level comment above MOVE_EPOCHS).
        """
        self.path = Path(csv_path)
        self.frame = pd.read_csv(self.path)
        if "Epoch" in self.frame.columns and not pd.api.types.is_numeric_dtype(self.frame["Epoch"]):
            mapped = self.frame["Epoch"].map(_epoch_to_code)
            n_bad = int(mapped.isna().sum())
            if n_bad:
                unknown = sorted(set(self.frame.loc[mapped.isna(), "Epoch"].astype(str)))[:6]
                print(f"WARNING: {n_bad} rows have unrecognized Epoch names {unknown}; "
                    f"left unmatched -- add them to _EPOCH_NAME_TO_CODE if needed.")
            self.frame["Epoch"] = mapped
            print(f"Epoch column was text; normalized to numeric codes "
                f"{sorted(set(self.frame['Epoch'].dropna().astype(int)))}.")
        self.input_source, self.source_reason = (detector or InputSourceDetector()).detect(self.frame, self.path)
        self.profile = PIPELINE_PROFILES.get(self.input_source, PIPELINE_PROFILES["unknown"])
        self.n_unindexed_dropped = 0
        print(f"Input source of '{self.path.name}': {self.input_source} ({self.source_reason}).")
        if self.profile.drop_unindexed_rows and "RZ2Idx" in self.frame.columns:
            unindexed = pd.to_numeric(self.frame["RZ2Idx"], errors="coerce").isna()
            self.n_unindexed_dropped = int(unindexed.sum())
            self.frame = self.frame[~unindexed]
            print(f"  Dropped {self.n_unindexed_dropped} rows with NaN RZ2Idx (cached or un-indexed samples).")
        elif self.profile.drop_unindexed_rows:
            print("  No RZ2Idx column (export older than v8.21): cached rows cannot be identified and are kept.")
        if move_epochs is not None:
            if "Epoch" not in self.frame.columns:
                print(f"WARNING: '{self.path.name}' has no Epoch column -- "
                    f"cannot filter to move_epochs={move_epochs}; using every row as-is.")
            else:
                epochs = np.atleast_1d(move_epochs)
                before = len(self.frame)
                self.frame = self.frame[self.frame["Epoch"].isin(epochs)]
                print(f"Epoch filter {tuple(epochs)}: kept {len(self.frame)}/{before} rows of '{self.path.name}'.")

    @staticmethod
    def estimate_sampling_rate(time_ms):
        time_ms = np.asarray(time_ms, dtype=float)
        if time_ms.size < 2:
            return float("nan")
        steps = np.diff(np.sort(time_ms))
        steps = steps[steps > 0]
        if steps.size == 0:
            return float("nan")
        return 1000.0 / float(np.mean(steps))

    @staticmethod
    def _move_time_ms(group):
        if "MoveTime_ms" in group.columns:
            return group["MoveTime_ms"].to_numpy(dtype=float)
        tms = group["Time_ms"].to_numpy(dtype=float)
        return tms - float(tms.min()) if tms.size else tms

    def trials(self):
        for (block, trial, attempt), group in self.frame.groupby(self.GROUP_KEYS, sort=True):
            group = group.sort_values("Time_ms")
            move_window_ms = None
            hold_start_ms = None
            hold_window_ms = None
            hold_max_speed_cm = None
            hold_excursion_px = None
            if "Epoch" in group.columns:
                ep = group["Epoch"].to_numpy()
                tms = group["Time_ms"].to_numpy(dtype=float)
                xs_ = group["X_px"].to_numpy(dtype=float)
                ys_ = group["Y_px"].to_numpy(dtype=float)
                mv = tms[ep == MOVEMENT_EPOCH]
                if mv.size:
                    move_window_ms = (float(mv.min()), float(mv.max()))
                hmask = ep == TARGET_HOLD_EPOCH
                hold = tms[hmask]
                if hold.size:
                    hold_start_ms = float(hold.min())
                if hold.size >= 2:
                    hx, hy = xs_[hmask], ys_[hmask]
                    hold_window_ms = (float(hold.min()), float(hold.max()))
                    hdt = np.diff(hold) / 1000.0
                    good = hdt > 0
                    if good.any():
                        seg = np.hypot(np.diff(hx), np.diff(hy))[good] / hdt[good] * CM_PER_PX
                        hold_max_speed_cm = float(seg.max()) if seg.size else None
                    hold_excursion_px = float(np.max(np.hypot(hx - hx[0], hy - hy[0])))
            yield Trial(
                date=str(group["Date"].iloc[0]),
                block=int(block),
                trial=int(trial),
                attempt=int(attempt),
                time_ms=group["Time_ms"].to_numpy(dtype=float),
                move_time_ms=self._move_time_ms(group),
                x=group["X_px"].to_numpy(dtype=float),
                y=group["Y_px"].to_numpy(dtype=float),
                fs_hz=self.estimate_sampling_rate(group["Time_ms"].to_numpy()),
                move_window_ms=move_window_ms,
                hold_start_ms=hold_start_ms,
                hold_window_ms=hold_window_ms,
                hold_max_speed_cm=hold_max_speed_cm,
                hold_excursion_px=hold_excursion_px,
            )

    def global_limits(self, margin=0.03, max_time_ms=None):
        t_max = 0.0
        p_min, p_max = np.inf, -np.inf
        over_cap = []
        for tr in self.trials():
            tr_t_max = float(tr.move_time_ms.max())
            t_max = max(t_max, tr_t_max)
            if max_time_ms is not None and tr_t_max > max_time_ms:
                over_cap.append((tr.block, tr.trial, tr.attempt, tr_t_max))
            p_min = min(p_min, float(tr.x.min()), float(tr.y.min()))
            p_max = max(p_max, float(tr.x.max()), float(tr.y.max()))
        if max_time_ms is not None:
            t_max = min(t_max, max_time_ms)
        pad = (p_max - p_min) * margin
        return (0.0, t_max), (p_min - pad, p_max + pad), over_cap

    def global_space_limits(self, margin=0.05):
        x_min, x_max = np.inf, -np.inf
        y_min, y_max = np.inf, -np.inf
        for tr in self.trials():
            x_min = min(x_min, float(tr.x.min()))
            x_max = max(x_max, float(tr.x.max()))
            y_min = min(y_min, float(tr.y.min()))
            y_max = max(y_max, float(tr.y.max()))
        px = (x_max - x_min) * margin
        py = (y_max - y_min) * margin
        return (x_min - px, x_max + px), (y_min - py, y_max + py)


def normalize_trial_schema(td):
    """Canonicalize the timing columns of a trial_data_*.csv across engine
    generations, so old and new sessions can be analysed by the same code.

    Four layouts exist in the wild. BOTH header spellings of the total are
    accepted on input, and the older ReactionTime_s header names a DIFFERENT
    interval in two of them -- which is the whole reason this function exists:

      gen A (v2_2 / pre-rename)   DecisionTime_s, ReactionTime_s, TotalTime_s
                                  ReactionTime_s = leave-center -> reach-target
                                  TotalTime_s    = decision + execution
      gen B (interim)             DecisionTime_s, ExecutionTime_s, TotalTime_s
      gen C (v8.19)               DecisionTime_s, ExecutionTime_s, ReactionTime_s
                                  ReactionTime_s = decision + execution (TOTAL)
      gen D (v8.20 onward)        DecisionTime_s, ExecutionTime_s, TotalTime_s
                                  TotalTime_s    = decision + execution (TOTAL)

    gen A is separated from the rest by whether ExecutionTime_s is present,
    which is the one signal that distinguishes them:
      present -> a ReactionTime_s column, if any, is the TOTAL   (gen C)
      absent  -> ReactionTime_s is the EXECUTION time            (gen A)

    Canonical output, always these three whatever came in:
      DecisionTime_s   target-onset  -> leave-center
      ExecutionTime_s  leave-center  -> reach-target
      TotalTime_s      DecisionTime_s + ExecutionTime_s

    ReactionTime_s is dropped after being folded into TotalTime_s: keeping
    both would put two identical columns into the correlation heatmap and
    the PCA, where an exact duplicate is not a second measurement.
    """
    td = td.copy()
    if "ExecutionTime_s" not in td.columns and "ReactionTime_s" in td.columns:
        td["ExecutionTime_s"] = td["ReactionTime_s"]
        td = td.drop(columns=["ReactionTime_s"])
    if "TotalTime_s" not in td.columns:
        if "ReactionTime_s" in td.columns:
            td["TotalTime_s"] = td["ReactionTime_s"]
        elif {"DecisionTime_s", "ExecutionTime_s"} <= set(td.columns):
            td["TotalTime_s"] = td["DecisionTime_s"] + td["ExecutionTime_s"]
    if "ReactionTime_s" in td.columns:
        td = td.drop(columns=["ReactionTime_s"])
    return td


class TrialInfoTable:
    """Per-trial lookups keyed on (Block, TrialNumInBlock, Attempt).

    bar_size_deg / direction_correct / is_correct / error_type always come
    from trial_data_<runTag>.csv.

    move_samples (NumMovementSamples) moved OUT of trial_data_<runTag>.csv
    into its own companion file, trial_kinematics_<runTag>.csv, on the
    MATLAB side (same runTag, same Block+TrialNumInBlock+Attempt key --
    see CenterOutTask.m). To stay usable on BOTH older sessions (column
    still sitting in trial_data) and current ones (column only in
    trial_kinematics), this checks trial_data first and falls back to the
    companion kinematics file, auto-detected by swapping the
    'trial_data_' filename prefix for 'trial_kinematics_' unless
    kinematics_path is given explicitly. Never raises on a missing column
    or missing file -- move_samples just comes back None, and the caller
    (TrialFigure.build) already falls back to the trial's own raw sample
    count in that case.
    """
    KEYS = ["Block", "TrialNumInBlock", "Attempt"]

    def __init__(self, path, kinematics_path=None):
        self.frame = pd.read_csv(path) if path and Path(path).exists() else None
        if self.frame is not None:
            self.frame = normalize_trial_schema(self.frame)

        if kinematics_path is None and path:
            guess = Path(path).with_name(Path(path).name.replace("trial_data_", "trial_kinematics_", 1))
            kinematics_path = guess if guess.exists() else None
        self.kin_frame = pd.read_csv(kinematics_path) if kinematics_path and Path(kinematics_path).exists() else None

    @staticmethod
    def _match(frame, block, trial, attempt):
        row = frame[(frame.Block == block) & (frame.TrialNumInBlock == trial) & (frame.Attempt == attempt)]
        return row.iloc[0] if len(row) == 1 else None

    def lookup(self, block, trial, attempt):
        result = {"bar_size_deg": None, "move_samples": None,
                  "direction_correct": None, "is_correct": None, "error_type": None,
                  "decision_time_s": None, "execution_time_s": None, "total_time_s": None}

        if self.frame is not None:
            row = self._match(self.frame, block, trial, attempt)
            if row is not None:
                result["bar_size_deg"] = float(row["BarSizeVA_deg"])
                result["direction_correct"] = str(row["DirectionCorrect"])
                result["is_correct"] = int(row["IsCorrect"])
                result["error_type"] = int(row["ErrorType"])
                if "DecisionTime_s" in row.index:
                    result["decision_time_s"] = float(row["DecisionTime_s"])
                if "ExecutionTime_s" in row.index:
                    result["execution_time_s"] = float(row["ExecutionTime_s"])
                if "TotalTime_s" in row.index:
                    result["total_time_s"] = float(row["TotalTime_s"])
                if "NumMovementSamples" in row.index:
                    result["move_samples"] = int(row["NumMovementSamples"])

        if result["move_samples"] is None and self.kin_frame is not None:
            krow = self._match(self.kin_frame, block, trial, attempt)
            if krow is not None and "NumMovementSamples" in krow.index:
                result["move_samples"] = int(krow["NumMovementSamples"])

        return result


def estimate_screen_layout(dataset, info_table, target_dist=TARGET_DIST_PX,
                           target_radius=TARGET_RADIUS_PX, center_radius=CENTER_RADIUS_PX,
                           center_override=None):
    if center_override is not None:
        cx, cy = center_override
    else:
        directions = ["Right_0", "Up_90", "Left_180", "Down_270"]
        ends = {d: [] for d in directions}
        for trial in dataset.trials():
            info = info_table.lookup(trial.block, trial.trial, trial.attempt)
            d = info.get("direction_correct")
            if info.get("is_correct") == 1 and d in ends:
                ends[d].append([trial.x[-1], trial.y[-1]])
        means = {}
        for d, pts in ends.items():
            if pts:
                arr = np.asarray(pts)
                means[d] = (float(arr[:, 0].mean()), float(arr[:, 1].mean()))
        cx = cy = None
        if "Right_0" in means and "Left_180" in means:
            cx = (means["Right_0"][0] + means["Left_180"][0]) / 2.0
        if "Up_90" in means and "Down_270" in means:
            cy = (means["Up_90"][1] + means["Down_270"][1]) / 2.0
        if cx is None:
            xs = [p[0] for p in means.values()]
            cx = float(np.mean(xs)) if xs else None
        if cy is None:
            ys = [p[1] for p in means.values()]
            cy = float(np.mean(ys)) if ys else None
        if cx is None or cy is None:
            return None
    offsets = {"Right_0": (target_dist, 0.0), "Up_90": (0.0, -target_dist),
               "Left_180": (-target_dist, 0.0), "Down_270": (0.0, target_dist)}
    targets = {d: (cx + ox, cy + oy) for d, (ox, oy) in offsets.items()}
    return {"center": (cx, cy), "targets": targets,
            "target_radius": target_radius, "center_radius": center_radius}


ACCEL_CMAP = plt.get_cmap("viridis")


class TrialFigure:
    X_COLOR = "#75d054"
    Y_COLOR = "#440154"
    OUTLIER_COLOR = "#E69F00"
    SCREEN_BG = "#0a0a0a"
    PATH_COLOR = "#c8c8c8"
    RAW_PATH_COLOR = "#5a5a5a"
    START_COLOR = "#009E73"
    TARGET_COLOR = "#e8e8e8"
    END_COLOR = "#D55E00"
    START_POINT_COLOR = "#20a386"
    END_POINT_COLOR = "#fde725"
    ACCEL_CMAP = ACCEL_CMAP

    def __init__(self, time_lim=None, pos_lim=None, space_lim=None, tick_step_ms=100,
                 layout=None, accel_vmax=None, hold_ref_ms=None, input_source="unknown"):
        self.time_lim = time_lim
        self.pos_lim = pos_lim
        self.space_lim = space_lim
        self.tick_step_ms = tick_step_ms
        self.layout = layout
        self.hold_ref_ms = hold_ref_ms
        self.accel_vmax = accel_vmax
        self.input_source = input_source

    def _apply_axes(self, ax, apply_ylim=True):
        if self.time_lim is not None:
            ax.set_xlim(*self.time_lim)
            ax.xaxis.set_major_locator(MultipleLocator(self.tick_step_ms))
            ax.xaxis.set_minor_locator(MultipleLocator(self.tick_step_ms / 2.0))
            ax.tick_params(axis="x", labelsize=7, rotation=0)
        if apply_ylim and self.pos_lim is not None:
            ax.set_ylim(*self.pos_lim)
        ax.grid(False)
        ax.tick_params(which="both", length=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    @staticmethod
    def _outward_align(pos, anchor):
        dx = pos[0] - anchor[0]
        dy = pos[1] - anchor[1]
        if dx > 12:
            ha = "left"
        elif dx < -12:
            ha = "right"
        else:
            ha = "center"
        if dy > 12:
            va = "top"
        elif dy < -12:
            va = "bottom"
        else:
            va = "center"
        return ha, va

    @staticmethod
    def _best_label_pos(anchor, radii, point_obstacles, circle_obstacles, bounds, n_angles=48, cap=45.0):
        if not isinstance(radii, (list, tuple)):
            radii = [radii]
        ax0, ay0 = anchor
        (xmin, xmax), (ymin, ymax) = bounds
        pts = np.asarray(point_obstacles, dtype=float) if len(point_obstacles) else np.empty((0, 2))
        best = (ax0 + radii[0], ay0)
        best_score = -1e18
        for r in radii:
            for k in range(n_angles):
                ang = 2.0 * np.pi * k / n_angles
                x = ax0 + r * np.cos(ang)
                y = ay0 + r * np.sin(ang)
                if not (xmin + 45 <= x <= xmax - 45 and ymin + 25 <= y <= ymax - 25):
                    continue
                clearance = 1e18
                if pts.shape[0]:
                    clearance = float(np.min(np.hypot(pts[:, 0] - x, pts[:, 1] - y)))
                for ccx, ccy, rr in circle_obstacles:
                    clearance = min(clearance, abs(float(np.hypot(x - ccx, y - ccy)) - rr))
                score = min(clearance, cap) - 0.05 * r
                if score > best_score:
                    best_score = score
                    best = (x, y)
        return best

    def _draw_path(self, ax, cbar_ax, trial, result, info):
        from matplotlib.patches import Circle
        rx, ry = result.raw_xy[:, 0], result.raw_xy[:, 1]
        gx, gy = result.grid_xy[:, 0], result.grid_xy[:, 1]
        ax.set_facecolor(self.SCREEN_BG)
        target_radius = 100.0
        correct_dir = info.get("direction_correct")
        correct_pos = None
        if self.layout is not None:
            cx, cy = self.layout["center"]
            center_radius = self.layout.get("center_radius", 100.0)
            target_radius = self.layout.get("target_radius", 100.0)
            ax.add_patch(Circle((cx, cy), center_radius, fill=False,
                                edgecolor="#666666", lw=1.0, zorder=1))
            for d, (tx, ty) in self.layout["targets"].items():
                is_correct = (d == correct_dir)
                edge = self.START_COLOR if is_correct else "#4a4a4a"
                lw = 2.2 if is_correct else 1.0
                ax.add_patch(Circle((tx, ty), target_radius, fill=False,
                                    edgecolor=edge, lw=lw, zorder=1))
                if is_correct:
                    correct_pos = (tx, ty)
        ax.plot(rx, ry, color=self.RAW_PATH_COLOR, lw=0.8, alpha=0.6, zorder=2)
        accel = result.accel_cm
        n_segments = max(gx.size - 1, 0)
        colorable = n_segments > 0 and accel.size == n_segments
        if colorable:
            points = np.column_stack([gx, gy]).reshape(-1, 1, 2)
            segments = np.concatenate([points[:-1], points[1:]], axis=1)
            shared_scale = (self.accel_vmax is not None and np.isfinite(self.accel_vmax)
                             and self.accel_vmax > 0)
            if shared_scale:
                vmax = self.accel_vmax
            else:
                local_max = float(np.nanmax(accel)) if np.isfinite(accel).any() else 0.0
                vmax = max(local_max, 1e-6)
            norm = Normalize(vmin=0.0, vmax=vmax)
            lc = LineCollection(segments, array=accel, cmap=self.ACCEL_CMAP,
                                norm=norm, linewidths=2.6, zorder=3)
            ax.add_collection(lc)
            cbar = ax.figure.colorbar(lc, cax=cbar_ax, orientation="horizontal")
            label = "Acceleration (cm/s^2)" if shared_scale else "Acceleration (cm/s^2) -- per-trial scale"
            cbar.set_label(label, color="#555555", fontsize=8)
            cbar.ax.tick_params(colors="#888888", labelsize=7, length=0)
            cbar.outline.set_edgecolor("#cccccc")
        else:
            ax.plot(gx, gy, color=self.PATH_COLOR, lw=2.0, zorder=3)
            cbar_ax.axis("off")
        ax.scatter([gx[0]], [gy[0]], s=60, facecolor=self.START_POINT_COLOR,
                   edgecolor="white", linewidth=0.8, zorder=4)
        ax.scatter([gx[-1]], [gy[-1]], s=85, marker="D", facecolor=self.END_POINT_COLOR,
                   edgecolor="white", linewidth=0.8, zorder=6)
        if np.isfinite(result.move_offset_ms) and result.grid_time_ms.size:
            _ai = int(np.argmin(np.abs(result.grid_time_ms - result.move_offset_ms)))
            ax.scatter([gx[_ai]], [gy[_ai]], s=55, marker="D", facecolor="#ffffff",
                       edgecolor="black", linewidth=0.8, zorder=7)
        if self.space_lim is not None:
            bounds = self.space_lim
        else:
            bounds = ((float(min(gx.min(), rx.min())), float(max(gx.max(), rx.max()))),
                      (float(min(gy.min(), ry.min())), float(max(gy.max(), ry.max()))))
        traj_pts = np.column_stack([np.concatenate([gx, rx]), np.concatenate([gy, ry])])
        if traj_pts.shape[0] > 60:
            idx = np.linspace(0, traj_pts.shape[0] - 1, 60).astype(int)
            traj_pts = traj_pts[idx]
        target_label_pos = None
        if self.layout is not None:
            all_circles = [(cx, cy, center_radius)]
            all_circles += [(tx, ty, target_radius) for (tx, ty) in self.layout["targets"].values()]
            if correct_pos is None and correct_dir in self.layout["targets"]:
                correct_pos = self.layout["targets"][correct_dir]
            if correct_pos is not None:
                other_circles = [(cx, cy, center_radius)]
                other_circles += [(tx, ty, target_radius)
                                  for d, (tx, ty) in self.layout["targets"].items() if d != correct_dir]
                pts = list(traj_pts) + [(gx[0], gy[0])]
                target_label_pos = self._best_label_pos(correct_pos, [target_radius + 30, target_radius + 52],
                                                        pts, other_circles, bounds)
                tha, tva = self._outward_align(target_label_pos, correct_pos)
                ax.text(target_label_pos[0], target_label_pos[1], "target",
                        color=self.START_COLOR, fontsize=8, ha=tha, va=tva, zorder=5)
            start_pts = list(traj_pts)
            if target_label_pos is not None:
                start_pts.append(target_label_pos)
            start_label_pos = self._best_label_pos((gx[0], gy[0]), [58, 82, 108, 136], start_pts, all_circles, bounds)
            sha, sva = self._outward_align(start_label_pos, (gx[0], gy[0]))
        else:
            start_label_pos = (gx[0], gy[0] - 40)
            sha, sva = "center", "center"
        ax.text(start_label_pos[0], start_label_pos[1], "start", color="#dddddd",
                fontsize=8, ha=sha, va=sva, zorder=5)
        if self.space_lim is not None:
            ax.set_xlim(*self.space_lim[0])
            ax.set_ylim(*self.space_lim[1])
        ax.invert_yaxis()
        ax.set_aspect("equal", adjustable="box")
        ax.set_title("Screen path: center to target", fontsize=11, loc="left")
        ax.set_xlabel("X (px)")
        ax.set_ylabel("Y (px)")
        ax.grid(False)
        ax.tick_params(colors="#888888", labelsize=7, length=0)
        outcome, ocolor = self._outcome(info)
        ax.text(0.03, 0.95, outcome, transform=ax.transAxes, color=ocolor,
                fontsize=11, fontweight="bold", va="top", ha="left")

    @staticmethod
    def _fmt(value, unit):
        return "n/a" if value is None or not np.isfinite(value) else f"{value:.1f} {unit}"

    @staticmethod
    def _outcome(info):
        if info.get("is_correct") == 1:
            return "Correct", "#009E73"
        if info.get("error_type") == 1:
            return "Early exit", "#CC79A7"
        if info.get("error_type") == 2:
            return "Wrong target", "#ffffff"
        if info.get("error_type") == 3:
            return "Hold-break", "#E69F00"
        return "n/a", "#aaaaaa"

    def build(self, trial, result, info):
        t_raw = result.raw_time_ms
        rx, ry = result.raw_xy[:, 0], result.raw_xy[:, 1]
        t_grid = result.grid_time_ms
        fig = plt.figure(figsize=(18.0, 9.5))
        outer = fig.add_gridspec(1, 2, width_ratios=[1.35, 1.0], wspace=0.16)
        left = outer[0, 0].subgridspec(3, 1, hspace=0.38)
        top = fig.add_subplot(left[0])
        bottom = fig.add_subplot(left[1], sharex=top)
        vel_ax = fig.add_subplot(left[2], sharex=top)
        right = outer[0, 1].subgridspec(3, 1, height_ratios=[5.0, 0.4, 2.8], hspace=0.5)
        path = fig.add_subplot(right[0])
        path.set_anchor("N")
        cbar_ax = fig.add_subplot(right[1])
        stats_ax = fig.add_subplot(right[2])
        stats_ax.axis("off")
        if np.isfinite(result.move_takeoff_ms):
            for _i, _ax in enumerate((top, bottom, vel_ax)):
                _ax.axvline(result.move_takeoff_ms, color="#D55E00", lw=1.4, ls="--", alpha=0.9, zorder=1,
                            label=f"takeoff {MOVE_SPEED_FRAC:.0%} of peak" if _i == 0 else None)
        # This variant keeps only the DECISION and MOVEMENT epochs (see
        # MOVE_EPOCHS), so the target hold is never in the data: the trace
        # ends at the diamond, where the target was reached, and no hold
        # window, hold onset or hold metrics are drawn or reported.
        if np.isfinite(result.move_onset_ms):
            for _i, _ax in enumerate((top, bottom, vel_ax)):
                _ax.axvline(result.move_onset_ms, color="#009E73", lw=1.4, ls="--",
                            alpha=0.9, zorder=1,
                            label="leave-center" if _i == 0 else None)
        if result.hampel_threshold.any():
            top.fill_between(t_raw,
                             result.hampel_median[:, 0] - result.hampel_threshold[:, 0],
                             result.hampel_median[:, 0] + result.hampel_threshold[:, 0],
                             color=self.X_COLOR, alpha=0.15, label="X MAD band")
            top.fill_between(t_raw,
                             result.hampel_median[:, 1] - result.hampel_threshold[:, 1],
                             result.hampel_median[:, 1] + result.hampel_threshold[:, 1],
                             color=self.Y_COLOR, alpha=0.15, label="Y MAD band")
        top.plot(t_raw, rx, color=self.X_COLOR, lw=1.2, label="X raw")
        top.plot(t_raw, ry, color=self.Y_COLOR, lw=1.2, label="Y raw")
        mx, my = result.outlier_mask[:, 0], result.outlier_mask[:, 1]
        if mx.any():
            top.scatter(t_raw[mx], rx[mx], s=32, color=self.OUTLIER_COLOR, zorder=5,
                        label="Hampel outliers")
        if my.any():
            top.scatter(t_raw[my], ry[my], s=32, color=self.OUTLIER_COLOR, zorder=5)
        top.set_ylabel("Position (px)")
        top.set_title("Raw noisy signal with Hampel MAD band", loc="left")
        top.legend(loc="upper right", fontsize=8, ncol=1, frameon=False)
        self._apply_axes(top)
        bottom.plot(t_raw, rx, color=self.X_COLOR, lw=0.8, alpha=0.25)
        bottom.plot(t_raw, ry, color=self.Y_COLOR, lw=0.8, alpha=0.25)
        bottom.plot(t_grid, result.grid_xy[:, 0], color=self.X_COLOR, lw=1.8, label="X filtered")
        bottom.plot(t_grid, result.grid_xy[:, 1], color=self.Y_COLOR, lw=1.8, label="Y filtered")
        stage_txt = (f"Hampel + Kalman + pchip grid + zero-phase Butterworth "
                     f"(design {CUTOFF_HZ:g} Hz, effective -3 dB {EFFECTIVE_CUTOFF_HZ:.1f} Hz)") if result.butter_applied \
            else "Hampel + Kalman + pchip grid (Butterworth skipped)"
        bottom.set_ylabel("Position (px)")
        bottom.set_title(stage_txt, loc="left")
        bottom.legend(loc="upper right", fontsize=8, frameon=False)
        self._apply_axes(bottom)
        has_vel = result.vel_cm.size > 0
        if has_vel:
            vel_ax.plot(result.vel_time_ms, result.vel_cm, color=self.PATH_COLOR, lw=1.8,
                        label="Speed (filtered)")
            if np.isfinite(result.peak_vel_cm):
                vel_ax.scatter([result.peak_vel_time_ms], [result.peak_vel_cm],
                               s=40, facecolor=self.END_COLOR, edgecolor="white",
                               linewidth=0.8, zorder=5, label="Peak speed")
            if np.isfinite(result.mean_vel_cm):
                vel_ax.axhline(result.mean_vel_cm, color=self.PATH_COLOR, lw=1.0,
                               ls="--", alpha=0.6, label="Mean speed")
            vel_ax.legend(loc="upper right", fontsize=8, frameon=False)
        else:
            vel_ax.text(0.5, 0.5, "not enough samples to differentiate",
                        transform=vel_ax.transAxes, ha="center", va="center",
                        fontsize=9, color="#888888")
        vel_ax.set_xlabel("Time since epoch-window onset (ms)")
        vel_ax.set_ylabel("Speed (cm/s)")
        vel_ax.set_title("Speed profile: |d(x,y)/dt| of the filtered trace", loc="left")
        self._apply_axes(vel_ax, apply_ylim=False)
        vel_ax.set_ylim(bottom=0)
        plt.setp(top.get_xticklabels(), visible=False)
        plt.setp(bottom.get_xticklabels(), visible=False)
        samples = info.get("move_samples")
        samples_txt = str(samples) if samples is not None else str(trial.n_samples)
        def _ts(v):
            return "n/a" if v is None or not np.isfinite(v) else f"{v:.3f} s"
        takeoff_target_s = ((result.window_end_ms - result.window_start_ms) / 1000.0
                            if np.isfinite(result.window_start_ms) and np.isfinite(result.window_end_ms)
                            else None)
        box = (f"Move samples: {samples_txt}\n"
               f"Peak vel (takeoff-target): {self._fmt(result.peak_vel_cm, 'cm/s')}\n"
               f"Mean vel (takeoff-target): {self._fmt(result.mean_vel_cm, 'cm/s')}\n"
               f"Median vel (takeoff-target): {self._fmt(result.median_vel_cm, 'cm/s')}\n"
               f"Peak acc (takeoff-target, p{ACCEL_PEAK_PERCENTILE:g}, Sav-Gol): {self._fmt(result.peak_accel_cm, 'cm/s^2')}\n"
               f"Mean acc (takeoff-target, Sav-Gol): {self._fmt(result.mean_accel_cm, 'cm/s^2')}\n"
               f"Median acc (takeoff-target, Sav-Gol): {self._fmt(result.median_accel_cm, 'cm/s^2')}\n"
               f"\n"
               f"Decision time: {_ts(info.get('decision_time_s'))}\n"
               f"Execution time: {_ts(info.get('execution_time_s'))}\n"
               f"Total time: {_ts(info.get('total_time_s'))}\n"
               f"Takeoff-to-target time: {_ts(takeoff_target_s)}")
        self._draw_path(path, cbar_ax, trial, result, info)
        stats_ax.text(0.0, 1.0, box, transform=stats_ax.transAxes, fontsize=10,
                      va="top", ha="left")
        fs_txt = "n/a" if not np.isfinite(trial.fs_hz) else f"{trial.fs_hz:.1f} Hz"
        bar = info.get("bar_size_deg")
        bar_txt = f"{bar:.2f} deg" if bar is not None else "n/a"
        outcome, _ = self._outcome(info)
        qc = ("  [under-resolved]" if result.under_resolved else "") \
            + (f"  [multi-peak x{result.n_speed_peaks}]" if result.n_speed_peaks > 1 else "")
        fig.suptitle(
            f"Joystick trajectory   |   Date {trial.date}   |   Block {trial.block}   |   "
            f"Trial {trial.trial}   |   Attempt {trial.attempt}   |   "
            f"Bar {bar_txt}   |   {outcome}   |   {self.input_source}, fs {fs_txt}{qc}",
            fontsize=12, y=0.99,
        )
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        return fig


def run(traj_path=None, trial_data_path=None, output_dir=OUTPUT_DIR,
        grid_dt_s=GRID_DT_S, cm_per_px=CM_PER_PX,
        show_inline=SHOW_INLINE, fix_time_axis=FIX_TIME_AXIS,
        fix_pos_axis=FIX_POS_AXIS, tick_step_ms=TIME_TICK_STEP_MS, margin=AXIS_MARGIN,
        move_epochs=MOVE_EPOCHS, kinematics_path=None,
        fix_accel_scale=FIX_ACCEL_SCALE, accel_scale_percentile=ACCEL_SCALE_PERCENTILE):
    """
    traj_path         : trajectory_movement_<runTag>.csv (has an Epoch column;
                         see MOVE_EPOCHS above for which codes are kept).
    trial_data_path    : trial_data_<runTag>.csv.
    kinematics_path    : trial_kinematics_<runTag>.csv, optional -- only
                         needed to recover NumMovementSamples on sessions
                         recorded after the CSV split; auto-detected next to
                         trial_data_path if not given (see TrialInfoTable).
    move_epochs        : epoch code(s) to keep from traj_path; defaults to
                         MOVE_EPOCHS = (DECISION_EPOCH, MOVEMENT_EPOCH,
                         TARGET_HOLD_EPOCH). Pass MOVEMENT_EPOCH alone, or
                         (DECISION_EPOCH, MOVEMENT_EPOCH), to reproduce an
                         earlier window, or None to disable filtering
                         entirely. Only ever narrows what's already IN
                         traj_path -- see the MOVE_EPOCHS comment above about
                         epochs missing from older exports.
    fix_accel_scale    : share one acceleration->color scale across every
                         figure (see FIX_ACCEL_SCALE above); False scales each
                         figure to its own peak acceleration instead.
    accel_scale_percentile : percentile of the pooled acceleration
                         distribution used as the shared colorbar ceiling when
                         fix_accel_scale is True (see ACCEL_SCALE_PERCENTILE).
    """
    traj_path = traj_path or TRAJ_PATH
    trial_data_path = trial_data_path or TRIAL_DATA_PATH
    _invalid = globals().get("invalid_reason")
    _reason = (_invalid(Path(str(trial_data_path)).stem.replace("trial_data_", "", 1))
               if callable(_invalid) else None)
    if _reason:
        print(f"WARNING: this session is listed in INVALID_SESSIONS: {_reason}")
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    dataset = TrajectoryDataset(traj_path, move_epochs=move_epochs)
    info_table = TrialInfoTable(trial_data_path, kinematics_path=kinematics_path)
    time_lim, pos_lim, over_cap_trials = dataset.global_limits(margin=margin, max_time_ms=MAX_TRAJ_TIME_MS)
    center_override = (SCREEN_WIDTH_PX / 2.0, SCREEN_HEIGHT_PX / 2.0) if USE_SCREEN_CENTER else None
    geometry = ScreenGeometry.resolve(trial_data_path, globals().get("GEOMETRY_SPEC_PATH"))
    print(geometry.describe())
    layout = estimate_screen_layout(dataset, info_table, target_dist=geometry.target_dist_px,
                                    target_radius=geometry.target_radius_px,
                                    center_radius=geometry.center_radius_px,
                                    center_override=center_override)
    if layout is not None:
        cx, cy = layout["center"]
        if SCREEN_VIEW_FULL:
            space_lim = ((0.0, float(SCREEN_WIDTH_PX)), (0.0, float(SCREEN_HEIGHT_PX)))
        elif SCREEN_HALF_EXTENT_PX is not None:
            reach = float(SCREEN_HALF_EXTENT_PX)
            space_lim = ((cx - reach, cx + reach), (cy - reach, cy + reach))
        else:
            reach = 0.0
            for trial in dataset.trials():
                reach = max(reach, float(np.max(np.abs(trial.x - cx))),
                            float(np.max(np.abs(trial.y - cy))))
            reach *= 1.06
            space_lim = ((cx - reach, cx + reach), (cy - reach, cy + reach))
    else:
        space_lim = dataset.global_space_limits()
    processor = TrajectoryProcessor.for_profile(dataset.profile, grid_dt_s=grid_dt_s,
                                                cutoff_hz=CUTOFF_HZ, cm_per_px=cm_per_px)
    processed = []
    n_dup_rows = 0
    n_dup_trials = 0
    all_accels = []
    hold_durs = []
    multi_peak_trials = []
    for trial in dataset.trials():
        result = processor.process(trial.time_ms, trial.x, trial.y,
                                   trial.move_window_ms, trial.hold_start_ms)
        info = info_table.lookup(trial.block, trial.trial, trial.attempt)
        processed.append((trial, result, info))
        n_dup_rows += result.n_duplicate_ts
        n_dup_trials += int(result.n_duplicate_ts > 0)
        if result.n_speed_peaks > 1:
            multi_peak_trials.append((trial.block, trial.trial, trial.attempt, result.n_speed_peaks))
        if result.accel_cm.size:
            all_accels.append(result.accel_cm)
        if trial.hold_window_ms is not None:
            hold_durs.append(trial.hold_window_ms[1] - trial.hold_window_ms[0])
    hold_ref_ms = float(np.median(hold_durs)) if hold_durs else None
    print(f"Duplicate timestamps dropped before filtering: {n_dup_rows} rows in "
          f"{n_dup_trials}/{len(processed)} trials (first occurrence of each time kept).")
    accel_vmax = None
    if fix_accel_scale and all_accels:
        pooled = np.concatenate(all_accels)
        pooled = pooled[np.isfinite(pooled)]
        if pooled.size:
            accel_vmax = float(np.percentile(pooled, accel_scale_percentile))
            print(f"Acceleration color scale (path panel): 0-{accel_vmax:.1f} cm/s^2 "
                  f"({accel_scale_percentile:g}th percentile across {len(all_accels)} trials).")
    drawer = TrialFigure(
        time_lim=time_lim if fix_time_axis else None,
        pos_lim=pos_lim if fix_pos_axis else None,
        space_lim=space_lim,
        tick_step_ms=tick_step_ms,
        layout=layout,
        accel_vmax=accel_vmax,
        input_source=dataset.input_source,
        hold_ref_ms=hold_ref_ms,
    )
    saved = []
    for trial, result, info in processed:
        fig = drawer.build(trial, result, info)
        path = out / f"{trial.date}_{trial.label}.png"
        fig.savefig(path, dpi=FIGURE_DPI)
        saved.append(path)
        if show_inline:
            plt.show()
        plt.close(fig)
    zip_path = "trial_figures.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in saved:
            zf.write(p, p.name)
    print(f"Generated {len(saved)} figures in '{output_dir}/' and packed '{zip_path}'.")
    if MAX_TRAJ_TIME_MS is not None:
        if over_cap_trials:
            print(f"{len(over_cap_trials)} trial(s) exceeded the {MAX_TRAJ_TIME_MS / 1000:.0f}s time-axis "
                  f"cap (plotted but clipped at {MAX_TRAJ_TIME_MS / 1000:.0f}s):")
            for _block, _trial, _attempt, _t_max in over_cap_trials:
                print(f"  - Block {_block}, TrialNumInBlock {_trial}, Attempt {_attempt}: {_t_max / 1000:.2f}s")
        else:
            print(f"No trials exceeded the {MAX_TRAJ_TIME_MS / 1000:.0f}s time-axis cap.")
    if multi_peak_trials:
        print(f"{len(multi_peak_trials)} trial(s) have more than one local speed peak "
              f"(candidate corrections / changes of mind, TAKEOFF_PEAK_CHOICE={TAKEOFF_PEAK_CHOICE!r}):")
        for _block, _trial, _attempt, _n_peaks in multi_peak_trials:
            print(f"  - Block {_block}, TrialNumInBlock {_trial}, Attempt {_attempt}: {_n_peaks} peaks")
    else:
        print("No trials with more than one local speed peak.")
    return zip_path

In [ ]:
ZIP_PATH = run()
try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception:
    pass

### 1.2. Feature engineering

In [ ]:
try:
    import seaborn as sns
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn"], check=True)
    import seaborn as sns

import colorsys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def normalize_trial_schema(td):
    """Canonicalize the timing columns of a trial_data_*.csv across engine
    generations, so old and new sessions can be analysed by the same code.

    Four layouts exist in the wild. BOTH header spellings of the total are
    accepted on input, and the older ReactionTime_s header names a DIFFERENT
    interval in two of them -- which is the whole reason this function exists:

      gen A (v2_2 / pre-rename)   DecisionTime_s, ReactionTime_s, TotalTime_s
                                  ReactionTime_s = leave-center -> reach-target
                                  TotalTime_s    = decision + execution
      gen B (interim)             DecisionTime_s, ExecutionTime_s, TotalTime_s
      gen C (v8.19)               DecisionTime_s, ExecutionTime_s, ReactionTime_s
                                  ReactionTime_s = decision + execution (TOTAL)
      gen D (v8.20 onward)        DecisionTime_s, ExecutionTime_s, TotalTime_s
                                  TotalTime_s    = decision + execution (TOTAL)

    gen A is separated from the rest by whether ExecutionTime_s is present,
    which is the one signal that distinguishes them:
      present -> a ReactionTime_s column, if any, is the TOTAL   (gen C)
      absent  -> ReactionTime_s is the EXECUTION time            (gen A)

    Canonical output, always these three whatever came in:
      DecisionTime_s   target-onset  -> leave-center
      ExecutionTime_s  leave-center  -> reach-target
      TotalTime_s      DecisionTime_s + ExecutionTime_s

    ReactionTime_s is dropped after being folded into TotalTime_s: keeping
    both would put two identical columns into the correlation heatmap and
    the PCA, where an exact duplicate is not a second measurement.
    """
    td = td.copy()
    if "ExecutionTime_s" not in td.columns and "ReactionTime_s" in td.columns:
        td["ExecutionTime_s"] = td["ReactionTime_s"]
        td = td.drop(columns=["ReactionTime_s"])
    if "TotalTime_s" not in td.columns:
        if "ReactionTime_s" in td.columns:
            td["TotalTime_s"] = td["ReactionTime_s"]
        elif {"DecisionTime_s", "ExecutionTime_s"} <= set(td.columns):
            td["TotalTime_s"] = td["DecisionTime_s"] + td["ExecutionTime_s"]
    if "ReactionTime_s" in td.columns:
        td = td.drop(columns=["ReactionTime_s"])
    return td


EARLY_EXIT_COL = "ErrorType"
EARLY_EXIT_CODE = 1
EARLY_EXIT_CHECK_COL = "ChosenTarget"


def drop_early_exits(df, col=EARLY_EXIT_COL, code=EARLY_EXIT_CODE,
                     check_col=EARLY_EXIT_CHECK_COL, verbose=True):
    if col not in df.columns:
        print(f"WARNING: '{col}' not in the table; keeping all {len(df)} trials.")
        return df
    error_type = pd.to_numeric(df[col], errors="coerce")
    early = error_type == code
    out = df[~early].copy()
    if verbose:
        n_drop = int(early.sum())
        pct = 100.0 * n_drop / len(df) if len(df) else 0.0
        print(f"Early exits excluded: dropped {n_drop}/{len(df)} trials "
              f"({pct:.1f}%) with {col} == {code}; {len(out)} trials go into the figures.")
        n_unknown = int(error_type.isna().sum())
        if n_unknown:
            print(f"  WARNING: {n_unknown} trials have no numeric {col} and were kept.")
        if check_col in df.columns:
            no_choice = ~(pd.to_numeric(df[check_col], errors="coerce") >= 1)
            early_with_choice = int((early & ~no_choice).sum())
            kept_without_choice = int((~early & no_choice).sum())
            if early_with_choice or kept_without_choice:
                print(f"  WARNING: {col} and {check_col} disagree on "
                      f"{early_with_choice + kept_without_choice} trials: "
                      f"{early_with_choice} dropped trials have {check_col} >= 1 and "
                      f"{kept_without_choice} kept trials have no {check_col} >= 1.")
            else:
                print(f"  Check: {col} == {code} coincides with {check_col} < 1 on every trial.")
    return out


PAIRPLOT_FEATURES = ["BarSizeVA_deg", "DecisionTime_s", "ExecutionTime_s",
                     "n_samples", "fs_hz", "move_takeoff_ms",
                     "peak_vel_cm", "mean_vel_cm", "peak_accel_cm",
                     "path_length_cm", "straightness"]
VIOLIN_GROUP = "ChosenTarget"
CORRECTNESS_GROUP = "IsCorrect"
VIOLIN_FEATURES = ["DecisionTime_s", "ExecutionTime_s", "n_samples", "move_takeoff_ms",
                   "peak_vel_cm", "mean_vel_cm", "peak_accel_cm",
                   "path_length_cm", "straightness"]
HUE = "StimulusGroup"
CORRELATION_METHOD = "spearman"

def _pastelize(hexcolor, sat=0.60, light=0.72):
    """Same hue as the rig's real category colour, softened to a pastel tone.

    HSL saturation/lightness are pinned to a fixed pastel target while the
    hue is kept from the source colour. Used instead of the raw,
    near-fluorescent rig colours (FFA500/00FF00/0000FF) because those are
    legible as a physical target on a dark booth screen but read as harsh,
    over-saturated ink on a printed or on-screen figure.
    """
    h = str(hexcolor).lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) / 255.0 for i in (0, 2, 4))
    hue, _l, _s = colorsys.rgb_to_hls(r, g, b)
    r2, g2, b2 = colorsys.hls_to_rgb(hue, light, sat)
    return "#{:02X}{:02X}{:02X}".format(round(r2 * 255), round(g2 * 255), round(b2 * 255))


# Real on-screen rig colours: ColorCategoryMap.m paints category 1 ORANGE, 2
# GREEN, 3 BLUE, and ConfigOrgParams.m exposes the same three as
# color3CatShort/Mid/Long. Read from THIS session's params_sess*.mat when
# available (SESSION_PARAMS, section 0.1) so a figure always matches what
# that session's rig actually painted; otherwise fall back to the
# ConfigOrgParams.m defaults below. Either way the hue is kept and only
# pastelized for the figure -- see _pastelize.
_RIG_CATEGORY_HEX = {"ShortGroup": "FFA500", "MidGroup": "00FF00", "LongGroup": "0000FF"}
_session_category_hex = (globals().get("SESSION_PARAMS") or {}).get("color3cat_hex") or {}
_CATEGORY_HEX = {k: (_session_category_hex.get(k) or v) for k, v in _RIG_CATEGORY_HEX.items()}

CATEGORY_COLORS = {k: _pastelize(v) for k, v in _CATEGORY_HEX.items()}
CATEGORY_COLORS_BY_ID = {1: CATEGORY_COLORS["ShortGroup"],
                         2: CATEGORY_COLORS["MidGroup"],
                         3: CATEGORY_COLORS["LongGroup"]}
CORRECTNESS_COLORS = {1: "#8c8c8c", 0: "#d44842"}   # correct = grey context, incorrect = the accent
# Trial dots on the violin panels. Mid grey, not near-white: the violins
# are drawn hollow, so the dots now sit against the white page instead of
# against a filled body, and #DDDDDD disappeared there.
POINT_COLOR = "#B3B3B3"
POINT_EDGE = "#555555"
CORRECTNESS_LABELS = {0: "incorrect", 1: "correct"}


def category_palette(levels):
    """Pastel colours that match the HUE of what the subject actually saw on
    screen (see CATEGORY_COLORS / _pastelize above). Levels that are not one
    of the three fall back to grey rather than silently borrowing a
    category's colour.
    """
    out = {}
    for lv in levels:
        key = lv
        if isinstance(lv, (int, float)) and not isinstance(lv, bool):
            key = int(lv)
        out[lv] = CATEGORY_COLORS.get(key, CATEGORY_COLORS_BY_ID.get(key, "#999999"))
    return out


def correctness_palette(levels):
    """Grey for correct, one warm accent for incorrect.

    Correct trials are the majority and the context; the incorrect ones are
    what the reader is looking for, so they get the only colour. Deliberately
    outside the category palette, so a correct/incorrect split can never be
    mistaken for a category split."""
    return {lv: CORRECTNESS_COLORS.get(int(lv), "#999999") for lv in levels}
ID_COLS = ["Block", "TrialNumInBlock", "Attempt"]


def _path_metrics(result, cm_per_px):
    gt = result.grid_time_ms
    gxy = result.grid_xy
    if gt.size < 2:
        return np.nan, np.nan
    lo = result.move_onset_ms if np.isfinite(result.move_onset_ms) else gt[0]
    hi = result.move_offset_ms if np.isfinite(result.move_offset_ms) else gt[-1]
    mask = (gt >= lo) & (gt <= hi)
    if mask.sum() < 2:
        mask = np.ones(gt.shape, dtype=bool)
    seg = gxy[mask]
    steps = np.hypot(np.diff(seg[:, 0]), np.diff(seg[:, 1]))
    path_len = float(steps.sum()) * cm_per_px
    net = float(np.hypot(seg[-1, 0] - seg[0, 0], seg[-1, 1] - seg[0, 1])) * cm_per_px
    straight = net / path_len if path_len > 1e-9 else np.nan
    return path_len, straight


def build_trial_features(traj_path, trial_data_path, move_epochs=None):
    dataset = TrajectoryDataset(traj_path, move_epochs=move_epochs)
    processor = TrajectoryProcessor.for_profile(dataset.profile, grid_dt_s=GRID_DT_S,
                                                cutoff_hz=CUTOFF_HZ, cm_per_px=CM_PER_PX)
    recs = []
    n_dup_rows = 0
    n_dup_trials = 0
    for tr in dataset.trials():
        r = processor.process(tr.time_ms, tr.x, tr.y, tr.move_window_ms, tr.hold_start_ms)
        n_dup_rows += r.n_duplicate_ts
        n_dup_trials += int(r.n_duplicate_ts > 0)
        path_len, straight = _path_metrics(r, CM_PER_PX)
        hold_dur_ms = (tr.hold_window_ms[1] - tr.hold_window_ms[0]) if tr.hold_window_ms else np.nan
        recs.append({
            "Block": tr.block, "TrialNumInBlock": tr.trial, "Attempt": tr.attempt,
            "fs_hz": tr.fs_hz, "n_samples": tr.n_samples,
            "peak_vel_cm": r.peak_vel_cm, "mean_vel_cm": r.mean_vel_cm, "median_vel_cm": r.median_vel_cm,
            "peak_accel_cm": r.peak_accel_cm, "mean_accel_cm": r.mean_accel_cm, "median_accel_cm": r.median_accel_cm,
            "path_length_cm": path_len, "straightness": straight,
            "move_takeoff_ms": r.move_takeoff_ms,
            "takeoff_lead_ms": r.move_onset_ms - r.move_takeoff_ms,
            "hold_dur_ms": hold_dur_ms, "hold_max_speed_cm": tr.hold_max_speed_cm,
            "hold_excursion_px": tr.hold_excursion_px,
            "under_resolved": int(r.under_resolved),
        })
    kin = pd.DataFrame.from_records(recs)
    td = normalize_trial_schema(pd.read_csv(trial_data_path))
    df = td.merge(kin, on=ID_COLS, how="left")
    if "DecisionTime_s" in df.columns:
        df["TakeoffTime_s"] = (pd.to_numeric(df["DecisionTime_s"], errors="coerce")
                               - pd.to_numeric(df["takeoff_lead_ms"], errors="coerce") / 1000.0)
    df["InputSource"] = dataset.input_source
    df.attrs["input_source"] = dataset.input_source
    df.attrs["source_reason"] = dataset.source_reason
    df.attrs["n_unindexed_dropped"] = dataset.n_unindexed_dropped
    df.attrs["n_duplicate_ts"] = n_dup_rows
    df.attrs["n_trials_with_duplicates"] = n_dup_trials
    return df


def pairplot_trials(df, features=PAIRPLOT_FEATURES, hue=HUE):
    cols = [c for c in features if c in df.columns]
    missing = [c for c in features if c not in df.columns]
    if missing:
        print(f"WARNING: requested features not in the table, skipped: {missing}. "
              f"Available columns: {list(df.columns)}")
    use_hue = hue if hue in df.columns else None
    sub = df.dropna(subset=cols).copy()
    print(f"Pairplot on {len(sub)}/{len(df)} trials (rows with any NaN in {cols} dropped).")
    pal = None
    if use_hue == "StimulusGroup":
        pal = category_palette(sorted(pd.unique(sub[use_hue])))
    elif use_hue == CORRECTNESS_GROUP:
        pal = correctness_palette(sorted(pd.unique(sub[use_hue])))
    g = sns.pairplot(sub, vars=cols, hue=use_hue, palette=pal, corner=True,
                     diag_kind="kde", plot_kws=dict(s=20, alpha=0.6, edgecolor="none"))
    g.figure.suptitle("Per-trial feature pairplot", y=1.02)
    g.figure.savefig("trial_pairplot.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return g


def _bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    ok = np.isfinite(p)
    out = np.full(p.shape, np.nan)
    pv = p[ok]
    n = pv.size
    if n:
        order = np.argsort(pv)
        ranked = pv[order] * n / (np.arange(n) + 1)
        q_sorted = np.minimum.accumulate(ranked[::-1])[::-1]
        q = np.empty(n)
        q[order] = np.clip(q_sorted, 0, 1)
        out[ok] = q
    return out


def violin_by_group(df, features=VIOLIN_FEATURES, group_col=VIOLIN_GROUP, alpha=0.05,
                    palette=None, savename=None):
    from scipy import stats
    if group_col not in df.columns:
        print(f"'{group_col}' not in the table; skipping violin plots.")
        return None
    feats = [f for f in features if f in df.columns]
    d = df[[group_col] + feats].copy()
    d = d[d[group_col].notna()]
    levels = sorted(pd.unique(d[group_col]))
    if len(levels) < 2:
        print(f"'{group_col}' has < 2 levels ({levels}); nothing to compare.")
        return None
    ncol = 3
    nrow = int(np.ceil(len(feats) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 3.4 * nrow))
    axes = np.atleast_1d(axes).ravel()
    rows = []
    for i, f in enumerate(feats):
        ax = axes[i]
        sub = d[[group_col, f]].dropna()
        arrays = [sub.loc[sub[group_col] == lv, f].to_numpy() for lv in levels]
        arrays = [a for a in arrays if a.size > 0]
        H = p = np.nan
        if len(arrays) >= 2 and all(a.size >= 1 for a in arrays) and sub[f].nunique() > 1:
            H, p = stats.kruskal(*arrays)
        pal = palette
        if pal is None:
            pal = (category_palette(levels) if group_col in ("StimulusGroup", "ChosenTarget")
                   else correctness_palette(levels) if group_col == CORRECTNESS_GROUP else None)
        # Hollow violins with the boxplot inside. The outline carries the
        # category colour and the box (IQR bar, whisker line, white median
        # dot) sits in the middle, so the SHAPE of the distribution and its
        # median/IQR are both readable without one hiding the other -- which
        # the filled violin with quartile lines could not do. Points go down
        # FIRST so they stay under the outline and the box, instead of
        # peppering over the summary that is the point of the panel.
        sns.stripplot(x=group_col, y=f, data=sub, order=levels, ax=ax,
                      color=POINT_COLOR, size=2.4, jitter=0.28, alpha=0.55,
                      edgecolor=POINT_EDGE, linewidth=0.2, zorder=0.5)
        sns.violinplot(x=group_col, y=f, data=sub, order=levels, ax=ax, hue=group_col,
                       palette=pal, legend=False, inner="box", cut=0,
                       density_norm="width", fill=False, linewidth=1.6,
                       inner_kws=dict(box_width=5.5, whis_width=1.4))
        ax.set_title(f"{f}  (KW p={p:.3f})" if np.isfinite(p) else f"{f}  (KW n/a)",
                     fontsize=9)
        ax.set_xlabel(group_col, fontsize=8)
        ax.set_ylabel(f, fontsize=8)
        ax.tick_params(labelsize=7)
        rows.append({"feature": f, "kruskal_H": H, "kruskal_p": p})
    for j in range(len(feats), len(axes)):
        axes[j].axis("off")
    fig.suptitle(f"Per-trial features by {group_col} (Kruskal-Wallis per feature)", y=1.0)
    fig.tight_layout()
    fig.savefig(savename or f"violin_by_{group_col.lower()}.png", dpi=FIGURE_DPI, bbox_inches="tight")
    res = pd.DataFrame(rows)
    res["q_fdr"] = _bh_fdr(res["kruskal_p"].to_numpy())
    res[f"sig(FDR<{alpha})"] = np.where(res["q_fdr"] < alpha, "yes", "no")
    res = res.reindex(res["kruskal_p"].fillna(1).sort_values().index).reset_index(drop=True)
    print(f"\nDoes each feature differ across {group_col} levels {levels}? "
          f"(Kruskal-Wallis, FDR-corrected):\n")
    print(res.round(4).to_string(index=False))
    n_sig = int((res["q_fdr"] < alpha).sum())
    print(f"\n{n_sig}/{len(res)} features differ significantly across {group_col} at FDR < {alpha}.")
    return res


def correlation_heatmap(df, method=CORRELATION_METHOD):
    num = df.select_dtypes("number").drop(columns=[c for c in ID_COLS if c in df.columns],
                                           errors="ignore")
    num = num.loc[:, num.nunique(dropna=True) > 1]
    corr = num.corr(method=method)
    fig, ax = plt.subplots(figsize=(0.6 * len(corr) + 3, 0.6 * len(corr) + 2))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0,
                square=True, cbar_kws={"shrink": 0.8}, ax=ax,
                annot_kws={"size": 7})
    ax.set_title(f"{method.capitalize()} correlation of per-trial features")
    fig.tight_layout()
    fig.savefig("trial_corr_heatmap.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return corr


def correct_vs_incorrect(df, features=VIOLIN_FEATURES, alpha=0.05,
                         group_col=CORRECTNESS_GROUP):
    """Per-feature comparison of correct against incorrect trials.

    The violins above split by ChosenTarget, which asks whether the movement
    differs by WHICH category the subject picked. This asks the other
    question: whether it differs by whether the pick was RIGHT. The two come
    apart, because a subject can reach the wrong target with a perfectly
    ordinary movement, and the interesting case is when they cannot.

    Mann-Whitney per feature with a rank-biserial effect size, FDR-corrected
    across features. Restricted to trials that reached a target, so an early
    exit (no choice at all) is not scored as an incorrect categorisation.
    """
    from scipy import stats
    if group_col not in df.columns:
        print(f"'{group_col}' not in the table; correct/incorrect comparison skipped.")
        return None
    d = df.copy()
    if "ChosenTarget" in d.columns:
        d = d[pd.to_numeric(d["ChosenTarget"], errors="coerce") >= 1]
    d[group_col] = pd.to_numeric(d[group_col], errors="coerce")
    d = d[d[group_col].isin([0, 1])]
    n_ok, n_bad = int((d[group_col] == 1).sum()), int((d[group_col] == 0).sum())
    print(f"Correct vs incorrect on {len(d)} trials that reached a target "
          f"({n_ok} correct, {n_bad} incorrect).")
    if n_ok < 5 or n_bad < 5:
        print("  Too few trials on one side for a meaningful comparison.")
        return None
    rows = []
    for f in [c for c in features if c in d.columns]:
        s = d[[group_col, f]].apply(pd.to_numeric, errors="coerce").dropna()
        a = s.loc[s[group_col] == 1, f].to_numpy()
        b = s.loc[s[group_col] == 0, f].to_numpy()
        if a.size < 5 or b.size < 5 or s[f].nunique() < 2:
            rows.append({"feature": f, "median_correct": np.nan, "median_incorrect": np.nan,
                         "U": np.nan, "p": np.nan, "rank_biserial": np.nan})
            continue
        U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        rows.append({"feature": f,
                     "median_correct": float(np.median(a)),
                     "median_incorrect": float(np.median(b)),
                     "delta": float(np.median(a) - np.median(b)),
                     "n_correct": a.size, "n_incorrect": b.size,
                     "U": float(U), "p": float(p),
                     "rank_biserial": float(2.0 * U / (a.size * b.size) - 1.0)})
    res = pd.DataFrame(rows)
    res["q_fdr"] = _bh_fdr(res["p"].to_numpy())
    res[f"sig(FDR<{alpha})"] = np.where(res["q_fdr"] < alpha, "yes", "no")
    res = res.reindex(res["p"].fillna(1).sort_values().index).reset_index(drop=True)
    print(f"\nDoes each feature differ between correct and incorrect trials? "
          f"(Mann-Whitney, FDR-corrected):\n")
    print(res.round(4).to_string(index=False))
    print(f"\n{int((res['q_fdr'] < alpha).sum())}/{len(res)} features differ at FDR < {alpha}. "
          f"rank_biserial > 0 means the feature is LARGER on correct trials.")
    return res


def accuracy_breakdown(df):
    """Accuracy by true category and by target position, in one table.

    Category accuracy is what the psychometric section models; position
    accuracy is the motor control, and it should be FLAT because the correct
    position is randomised independently of bar length. A position that
    stands out is a rig or a handedness effect, not a categorisation result.
    """
    out = {}
    if "StimulusGroup" in df.columns:
        g = df.groupby("StimulusGroup")["IsCorrect"].agg(["mean", "count"])
        g.columns = ["accuracy", "n"]
        out["by_category"] = g
        print("\nAccuracy by true category:")
        print(g.round(4).to_string())
    if "DirectionCorrect" in df.columns:
        g = df.groupby("DirectionCorrect")["IsCorrect"].agg(["mean", "count"])
        g.columns = ["accuracy", "n"]
        out["by_direction"] = g
        print("\nAccuracy by correct target position:")
        print(g.round(4).to_string())
    fig, axes = plt.subplots(1, len(out), figsize=(6.0 * len(out), 4.0), squeeze=False)
    for ax, (name, g) in zip(axes.ravel(), out.items()):
        cols = ([CATEGORY_COLORS.get(i, "#999999") for i in g.index]
                if name == "by_category" else "#31688e")
        se = np.sqrt(g["accuracy"] * (1 - g["accuracy"]) / g["n"])
        ax.bar(g.index.astype(str), g["accuracy"], yerr=1.96 * se, color=cols,
               edgecolor="none", alpha=0.9, width=0.6, ecolor=GRAY, capsize=0)
        ax.axhline(1.0 / 3.0, color=GRAY_LIGHT, lw=1)
        ax.text(len(g) - 0.5, 1.0 / 3.0 + 0.015, "chance", ha="right", va="bottom",
                fontsize=8, color=GRAY)
        ax.set_ylim(0, 1.02)
        ax.set_ylabel("P(correct)")
        ax.set_title(name.replace("_", " "))
        ax.tick_params(axis="x", rotation=20, labelsize=8)
    fig.suptitle("Accuracy by true category and by target position", y=1.02)
    fig.tight_layout()
    fig.savefig("accuracy_breakdown.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return out


EDA_EPOCHS = (DECISION_EPOCH, MOVEMENT_EPOCH)
TRAJ = globals().get("TRAJ_FULL_PATH") or globals().get("TRAJ_PATH")
FEATURES_DF = build_trial_features(TRAJ, TRIAL_DATA_PATH, move_epochs=EDA_EPOCHS)
FEATURES_DF.to_csv("trial_features.csv", index=False)
print(f"Built feature table: {FEATURES_DF.shape[0]} trials x {FEATURES_DF.shape[1]} columns.")
print("Columns:", list(FEATURES_DF.columns))
print(f"Duplicate timestamps dropped before filtering: {FEATURES_DF.attrs.get('n_duplicate_ts', 0)} rows in "
      f"{FEATURES_DF.attrs.get('n_trials_with_duplicates', 0)} trials (first occurrence of each time kept).")

ANALYSIS_DF = drop_early_exits(FEATURES_DF)
ANALYSIS_DF.to_csv("trial_features_no_early_exits.csv", index=False)

pairplot_trials(ANALYSIS_DF)
plt.show()
violin_by_group(ANALYSIS_DF, group_col=VIOLIN_GROUP)
plt.show()
violin_by_group(ANALYSIS_DF, group_col=CORRECTNESS_GROUP)
plt.show()
CORRECTNESS_STATS = correct_vs_incorrect(ANALYSIS_DF)
plt.show()
ACCURACY_BREAKDOWN = accuracy_breakdown(ANALYSIS_DF)
plt.show()
correlation_heatmap(ANALYSIS_DF)
plt.show()

### 1.3. Dimensionality Reduction (PCA)

In [ ]:
try:
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"], check=True)
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PCA_FEATURES = ["BarSizeVA_deg", "DecisionTime_s", "ExecutionTime_s",
                "move_takeoff_ms", "n_samples",
                "peak_vel_cm", "mean_vel_cm", "peak_accel_cm", "mean_accel_cm",
                "straightness", "hold_dur_ms", "hold_max_speed_cm",
                "hold_excursion_px"]
VARIANCE_TARGET = 0.90
N_LOADING_PCS = 5
SCATTER_LABELS = ["StimulusGroup", "IsCorrect"]

if "ANALYSIS_DF" in globals():
    SOURCE_DF = ANALYSIS_DF
elif "FEATURES_DF" in globals():
    print("ANALYSIS_DF not found; filtering FEATURES_DF here instead.")
    SOURCE_DF = drop_early_exits(FEATURES_DF)
else:
    raise NameError("Run the feature-engineering cell above first: it defines "
                    "FEATURES_DF and ANALYSIS_DF.")

DF = SOURCE_DF.copy()

missing = [c for c in PCA_FEATURES if c not in DF.columns]
if missing:
    print(f"WARNING: not in the table, skipped: {missing}")
constant = [c for c in PCA_FEATURES
            if c in DF.columns and DF[c].nunique(dropna=True) <= 1]
if constant:
    print(f"Dropped constant columns (zero variance): {constant}")
COLS = [c for c in PCA_FEATURES if c in DF.columns and c not in constant]
if len(COLS) < 2:
    raise ValueError(f"Need at least 2 usable features for PCA; got {COLS}.")

FIT_DF = DF.dropna(subset=COLS)
n_dropped = len(DF) - len(FIT_DF)
if n_dropped:
    print(f"Dropped {n_dropped}/{len(DF)} trials with a NaN in {COLS}.")
if len(FIT_DF) < 3:
    raise ValueError(f"Only {len(FIT_DF)} complete trials; not enough for PCA.")

SCALER = StandardScaler()
X = SCALER.fit_transform(FIT_DF[COLS].to_numpy(dtype=float))
PCA_MODEL = PCA().fit(X)
SCORES = PCA_MODEL.transform(X)

EVR = PCA_MODEL.explained_variance_ratio_
CUM = np.cumsum(EVR)
K90 = int(np.searchsorted(CUM, VARIANCE_TARGET) + 1)

print(f"\nPCA on {SCORES.shape[0]} trials x {len(COLS)} features "
      f"-> {SCORES.shape[1]} components.")
print(f"K90 = {K90} components reach {VARIANCE_TARGET:.0%} of the variance "
      f"(PC1 alone: {EVR[0]:.1%}).")
print("\nExplained variance per component:")
print(pd.DataFrame({"PC": [f"PC{i+1}" for i in range(len(EVR))],
                    "variance": EVR.round(4),
                    "cumulative": CUM.round(4)}).to_string(index=False))

pcs = np.arange(1, len(EVR) + 1)
fig, ax1 = plt.subplots(figsize=(7.5, 4.5))
ax1.bar(pcs, EVR, color="#4477aa", alpha=0.85, label="Variance per PC")
ax1.set_xlabel("Principal component")
ax1.set_ylabel("Explained variance", color="#4477aa")
ax1.tick_params(axis="y", labelcolor="#4477aa")
ax1.set_xticks(pcs)
ax2 = ax1.twinx()
ax2.plot(pcs, CUM, "-o", color="#ee6677", label="Cumulative")
ax2.axhline(VARIANCE_TARGET, ls="--", lw=1, color="#888888")
ax2.axvline(K90, ls=":", lw=1.2, color="#333333")
ax2.annotate(f"K90 = {K90}", xy=(K90, CUM[K90 - 1]),
             xytext=(6, -14), textcoords="offset points", fontsize=9)
ax2.set_ylabel("Cumulative variance", color="#ee6677")
ax2.tick_params(axis="y", labelcolor="#ee6677")
ax2.set_ylim(0, 1.02)
ax1.set_title(f"PCA scree ({SCORES.shape[0]} trials, {len(COLS)} features)")
fig.tight_layout()
fig.savefig("pca_scree.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()

n_show = min(N_LOADING_PCS, PCA_MODEL.components_.shape[0])
LOADINGS = pd.DataFrame(PCA_MODEL.components_[:n_show].T,
                        index=COLS, columns=[f"PC{i+1}" for i in range(n_show)])
fig, ax = plt.subplots(figsize=(1.15 * n_show + 3.5, 0.42 * len(COLS) + 2))
im = ax.imshow(LOADINGS.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(n_show))
ax.set_xticklabels([f"PC{i+1}\n{EVR[i]:.0%}" for i in range(n_show)], fontsize=9)
ax.set_yticks(range(len(COLS)))
ax.set_yticklabels(COLS, fontsize=9)
for i in range(len(COLS)):
    for j in range(n_show):
        v = LOADINGS.iat[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5,
                color="white" if abs(v) > 0.55 else "black")
ax.set_title("PCA loadings (feature weight per component)")
fig.colorbar(im, ax=ax, shrink=0.75, label="loading")
fig.tight_layout()
fig.savefig("pca_loadings.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()

print("\nTop |loading| per component:")
for j in range(n_show):
    col = LOADINGS.iloc[:, j].abs().sort_values(ascending=False)
    top = ", ".join(f"{k} ({LOADINGS.iloc[:, j][k]:+.2f})" for k in col.index[:3])
    print(f"  PC{j+1} ({EVR[j]:.1%}): {top}")

labels = [c for c in SCATTER_LABELS if c in FIT_DF.columns]
if labels:
    fig, axes = plt.subplots(1, len(labels), figsize=(6.2 * len(labels), 5.2),
                             squeeze=False)
    for ax, lab in zip(axes[0], labels):
        vals = FIT_DF[lab].to_numpy()
        for lv in pd.unique(pd.Series(vals).dropna()):
            m = vals == lv
            ax.scatter(SCORES[m, 0], SCORES[m, 1], s=26, alpha=0.75,
                       edgecolor="none", label=f"{lab}={lv}",
                   color=(category_palette([lv]).get(lv) if lab == "StimulusGroup"
                          else correctness_palette([lv]).get(lv) if lab == "IsCorrect" else None))
        ax.axhline(0, lw=0.6, color="#bbbbbb")
        ax.axvline(0, lw=0.6, color="#bbbbbb")
        ax.set_xlabel(f"PC1 ({EVR[0]:.1%})")
        ax.set_ylabel(f"PC2 ({EVR[1]:.1%})")
        ax.set_title(f"Trials in PC space, coloured by {lab}")
        ax.legend(fontsize=8, frameon=False)
    fig.tight_layout()
    fig.savefig("pca_scores.png", dpi=FIGURE_DPI, bbox_inches="tight")
    plt.show()
else:
    print(f"None of {SCATTER_LABELS} is in the table; skipping the score scatter.")

ID_KEEP = [c for c in ["Block", "TrialNumInBlock", "Attempt", "StimulusGroup",
                       "BarSizeVA_deg", "IsCorrect", "ErrorType", "ChosenTarget"]
           if c in FIT_DF.columns]
SCORES_DF = pd.concat(
    [FIT_DF[ID_KEEP].reset_index(drop=True),
     pd.DataFrame(SCORES, columns=[f"PC{i+1}" for i in range(SCORES.shape[1])])],
    axis=1)
SCORES_DF.to_csv("pca_scores.csv", index=False)
LOADINGS.to_csv("pca_loadings.csv")
print(f"\nWrote pca_scores.csv ({SCORES_DF.shape[0]} rows) and pca_loadings.csv.")
print("Defined for the clustering cell: DF, COLS, SCORES, PCA_MODEL, K90.")


### 1.4. Clustering (K-means)

In [ ]:
try:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import StandardScaler
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"], check=True)
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CLUSTER_ON = "pca"
N_DIMS = None
K_RANGE = range(2, 9)
K_OVERRIDE = None
RANDOM_STATE = 0
LABEL_COLS = ["StimulusGroup", "IsCorrect"]
LABEL_PALETTES = {"StimulusGroup": category_palette, "IsCorrect": correctness_palette}
OVERLAY_LABEL = "IsCorrect"
KINEMATIC_FEATURES = ["peak_vel_cm", "mean_vel_cm", "peak_accel_cm", "straightness"]
COLOR_BY = "BarSizeVA_deg"


def cluster_matrix():
    if CLUSTER_ON == "pca":
        n = N_DIMS or min(int(globals().get("K90", 3)), SCORES.shape[1])
        Z = SCORES[:, :n]
        axis_names = [f"PC{i+1}" for i in range(n)]
    else:
        Z = StandardScaler().fit_transform(DF.dropna(subset=COLS)[COLS].to_numpy(float))
        axis_names = list(COLS)
    labels_df = DF.dropna(subset=COLS).reset_index(drop=True)
    print(f"Clustering on {Z.shape[0]} trials x {Z.shape[1]} dims ({CLUSTER_ON}).")
    return Z, axis_names, labels_df


def select_k(Z):
    ks = list(K_RANGE)
    inertia, sil = [], []
    for k in ks:
        km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Z)
        inertia.append(km.inertia_)
        sil.append(silhouette_score(Z, km.labels_))
    best_k = K_OVERRIDE or ks[int(np.argmax(sil))]
    fig, ax1 = plt.subplots(figsize=(7, 4.5))
    ax1.plot(ks, inertia, "-o", color="#4477aa", label="Inertia (elbow)")
    ax1.set_xlabel("k (clusters)"); ax1.set_ylabel("Inertia", color="#4477aa")
    ax1.tick_params(axis="y", labelcolor="#4477aa")
    ax2 = ax1.twinx()
    ax2.plot(ks, sil, "-s", color="#ee6677", label="Silhouette")
    ax2.set_ylabel("Silhouette", color="#ee6677")
    ax2.tick_params(axis="y", labelcolor="#ee6677")
    ax2.axvline(best_k, ls="--", color="#888888", lw=1)
    ax1.set_title(f"k selection (best silhouette at k={best_k})")
    fig.tight_layout()
    fig.savefig("cluster_selection.png", dpi=FIGURE_DPI, bbox_inches="tight")
    print("k:", ks)
    print("silhouette:", np.round(sil, 3))
    return best_k


def fit_and_plot(Z, axis_names, k):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Z)
    lab = km.labels_
    cmap = plt.get_cmap("tab10", k)
    fig, ax = plt.subplots(figsize=(8, 7))
    for c in range(k):
        m = lab == c
        ax.scatter(Z[m, 0], Z[m, 1], s=30, alpha=0.8, color=cmap(c),
                   edgecolor="none", label=f"cluster {c} (n={int(m.sum())})")
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               s=200, marker="X", color="black", edgecolor="white", linewidth=1.2,
               zorder=5, label="centroids")
    ax.set_xlabel(axis_names[0]); ax.set_ylabel(axis_names[1] if len(axis_names) > 1 else "")
    ax.set_title(f"K-means clusters (k={k}) on {axis_names[0]}/{axis_names[1] if len(axis_names)>1 else ''}")
    ax.legend(frameon=False, loc="best")
    fig.tight_layout()
    fig.savefig("cluster_scatter.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return lab


def plot_clusters_vs_label(Z, axis_names, lab, labels_df, label_col=OVERLAY_LABEL):
    if label_col not in labels_df.columns:
        print(f"'{label_col}' not in the table; skipping the overlay view.")
        return
    y = labels_df[label_col].to_numpy()
    cats = pd.unique(y)
    markers = ["o", "X", "s", "^", "D"]
    k = int(lab.max()) + 1
    ccmap = plt.get_cmap("tab10", k)
    lpal = LABEL_PALETTES.get(label_col)
    lcolors = ([lpal(list(cats)).get(cat) for cat in cats] if lpal is not None
               else [plt.get_cmap("viridis", max(len(cats), 1))(i) for i in range(len(cats))])
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 6.5), sharex=True, sharey=True)
    for c in range(k):
        for mi, cat in enumerate(cats):
            m = (lab == c) & (y == cat)
            if m.any():
                axL.scatter(Z[m, 0], Z[m, 1], s=34, alpha=0.85, color=ccmap(c),
                            marker=markers[mi % len(markers)], edgecolor="none")
    clu_handles = [plt.Line2D([], [], marker="o", ls="", color=ccmap(c), label=f"cluster {c}")
                   for c in range(k)]
    lab_handles = [plt.Line2D([], [], marker=markers[mi % len(markers)], ls="", color="#555555",
                              label=f"{label_col}={cat}") for mi, cat in enumerate(cats)]
    axL.legend(handles=clu_handles + lab_handles, frameon=False, fontsize=8, loc="best")
    axL.set_title("Colored by cluster (marker = " + label_col + ")")
    axL.set_xlabel(axis_names[0]); axL.set_ylabel(axis_names[1] if len(axis_names) > 1 else "")
    for mi, cat in enumerate(cats):
        m = y == cat
        axR.scatter(Z[m, 0], Z[m, 1], s=34, alpha=0.85, color=lcolors[mi],
                    edgecolor="#333333", linewidth=0.3, label=f"{label_col}={cat}")
    axR.legend(frameon=False, fontsize=8, loc="best")
    axR.set_title(f"Colored by {label_col}")
    axR.set_xlabel(axis_names[0])
    fig.suptitle(f"Clusters vs {label_col} on {axis_names[0]}/{axis_names[1] if len(axis_names)>1 else ''}")
    fig.tight_layout()
    fig.savefig("cluster_vs_label.png", dpi=FIGURE_DPI, bbox_inches="tight")


def plot_clusters_kinematic(labels_df, lab, features=KINEMATIC_FEATURES, color_by=COLOR_BY):
    from matplotlib.cm import ScalarMappable
    from matplotlib.colors import Normalize
    feats = [f for f in features if f in labels_df.columns]
    if color_by not in labels_df.columns or len(feats) < 2:
        print(f"Kinematic view skipped (need >=2 of {features} and '{color_by}').")
        return
    d = labels_df.copy()
    d["_cluster"] = lab
    d = d.dropna(subset=feats + [color_by])
    k = int(d["_cluster"].max()) + 1
    markers = ["o", "X", "s", "^", "D", "P"]
    cmap = plt.get_cmap("viridis")
    norm = Normalize(vmin=d[color_by].min(), vmax=d[color_by].max())
    n = len(feats)
    fig, axes = plt.subplots(n, n, figsize=(3.0 * n, 3.0 * n))
    for i in range(n):
        for j in range(n):
            ax = axes[i, j]
            if j > i:
                ax.axis("off")
                continue
            if i == j:
                for c in range(k):
                    vals = d.loc[d["_cluster"] == c, feats[i]]
                    ax.hist(vals, bins=10, histtype="step", lw=1.4, color=plt.get_cmap("tab10")(c))
                ax.set_ylabel("count" if j == 0 else "")
            else:
                for c in range(k):
                    m = d["_cluster"] == c
                    ax.scatter(d.loc[m, feats[j]], d.loc[m, feats[i]],
                               c=d.loc[m, color_by], cmap=cmap, norm=norm,
                               marker=markers[c % len(markers)], s=34, alpha=0.9,
                               edgecolor="#333333", linewidth=0.3)
            if i == n - 1:
                ax.set_xlabel(feats[j])
            if j == 0 and i != 0:
                ax.set_ylabel(feats[i])
    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, shrink=0.6, pad=0.02)
    cbar.set_label(color_by)
    handles = [plt.Line2D([], [], marker=markers[c % len(markers)], ls="", color="#444444",
                          label=f"cluster {c}") for c in range(k)]
    fig.legend(handles=handles, loc="upper right", frameon=False, title="cluster")
    fig.suptitle(f"Clusters in kinematic space (color = {color_by})", y=0.98)
    fig.savefig("cluster_kinematic.png", dpi=FIGURE_DPI, bbox_inches="tight")


def cluster_profile(labels_df, lab, features):
    feats = [f for f in features if f in labels_df.columns]
    d = labels_df[feats].copy()
    d["_cluster"] = lab
    means = d.groupby("_cluster")[feats].mean()
    overall_std = d[feats].std(ddof=0).replace(0, np.nan)
    if means.shape[0] == 2:
        sep = (means.iloc[1] - means.iloc[0]).abs() / overall_std
    else:
        sep = means.std(ddof=0) / overall_std
    tab = means.T
    tab.columns = [f"cluster{c}_mean" for c in means.index]
    tab["separation_sd"] = sep.reindex(tab.index)
    tab = tab.sort_values("separation_sd", ascending=False)
    print("\nWhat separates the clusters (features ranked by standardized "
          "cluster-mean difference; higher = drives the split more):")
    print(tab.round(3).to_string())
    return tab


def cross_tabs(labels_df, lab):
    out = labels_df.copy()
    out["cluster"] = lab
    for col in LABEL_COLS:
        if col in out.columns:
            ct = pd.crosstab(out["cluster"], out[col])
            ctn = pd.crosstab(out["cluster"], out[col], normalize="index").round(2)
            print(f"\nCluster vs {col} (counts):\n{ct}")
            print(f"Cluster vs {col} (row-normalized):\n{ctn}")
    return out


Z, AXIS_NAMES, LABELS_DF = cluster_matrix()
BEST_K = select_k(Z)
plt.show()
CLUSTER_LABELS = fit_and_plot(Z, AXIS_NAMES, BEST_K)
plt.show()
plot_clusters_vs_label(Z, AXIS_NAMES, CLUSTER_LABELS, LABELS_DF)
plt.show()
plot_clusters_kinematic(LABELS_DF, CLUSTER_LABELS)
plt.show()
cluster_profile(LABELS_DF, CLUSTER_LABELS, COLS)
CLUSTERED_DF = cross_tabs(LABELS_DF, CLUSTER_LABELS)
CLUSTERED_DF.to_csv("trial_features_clustered.csv", index=False)
print(f"\nSaved clustered table: trial_features_clustered.csv ({len(CLUSTERED_DF)} trials, k={BEST_K}).")

NameError: name 'SCORES' is not defined

### 1.5. Psychometric functions

The core measurement of a categorization task, and the one the notebook was
missing: **where** the subject puts each category boundary, and **how sharply**
it separates the categories either side of it.

A `K`-category task has `K-1` boundaries, so each one gets its own curve from
the same trials, via the ordinal splits `P(chosen >= 2)` and `P(chosen >= 3)`.
Each curve is a cumulative Gaussian fitted by binomial maximum likelihood:

- **mu** — the boundary in deg VA (the lapse-corrected 50% point).
- **sigma** — the width of the transition. Smaller = finer discrimination.
- **lapse** — stimulus-independent errors.

The lapse parameter is not decorative. Early exits and hold failures
(`ErrorType` 1 and 3) put a floor and a ceiling on the curve; fitting without a
lapse term absorbs them into the slope and biases both mu and sigma. 95%
intervals are percentile bootstraps over the per-level binomial counts.

Sessions running a reduced stimulus set (`prototypes3`, 3 lengths) do not have
enough distinct levels to identify three free parameters per boundary; the cell
reports the raw proportions and says so rather than returning a fit that the
data cannot support.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm

CATEGORY_ORDER = ["ShortGroup", "MidGroup", "LongGroup"]
CATEGORY_ID = {name: i + 1 for i, name in enumerate(CATEGORY_ORDER)}
STIMULUS_COL = "BarSizeVA_deg"
CHOICE_COL = "ChosenTarget"
TRUTH_COL = "StimulusGroup"
MIN_LEVELS_FOR_FIT = 4
N_BOOTSTRAP = 400
LAPSE_MAX = 0.35
PSYCHO_SEED = 0


def ordinal_choice_table(df, stimulus_col=STIMULUS_COL, choice_col=CHOICE_COL):
    """One row per (stimulus level, ordinal split) with binomial counts.

    A K-category ordinal task has K-1 boundaries. Split j (j = 2..K) asks
    'did the subject choose category j or higher?', so each boundary gets its
    own two-alternative psychometric curve fitted on the same trials. Only
    trials with a real choice (ChosenTarget >= 1) contribute: an early exit
    is a missing response, not a response at the bottom category.
    """
    d = df[df[choice_col].notna()].copy()
    d = d[pd.to_numeric(d[choice_col], errors="coerce") >= 1]
    if d.empty:
        return pd.DataFrame(columns=["split", "x", "n", "k", "p"])
    d["_choice"] = pd.to_numeric(d[choice_col], errors="coerce").astype(int)
    kmax = int(d["_choice"].max())
    rows = []
    for j in range(2, kmax + 1):
        for x, g in d.groupby(stimulus_col):
            n = len(g)
            k = int((g["_choice"] >= j).sum())
            rows.append({"split": j, "x": float(x), "n": n, "k": k, "p": k / n if n else np.nan})
    return pd.DataFrame(rows).sort_values(["split", "x"]).reset_index(drop=True)


def _neg_log_lik(theta, x, n, k):
    mu, log_sigma, logit_lapse = theta
    sigma = np.exp(np.clip(log_sigma, -20, 20))
    lapse = LAPSE_MAX / (1.0 + np.exp(-np.clip(logit_lapse, -30, 30)))
    p = lapse / 2.0 + (1.0 - lapse) * norm.cdf((x - mu) / sigma)
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -np.sum(k * np.log(p) + (n - k) * np.log(1.0 - p))


def fit_psychometric(x, n, k):
    """Cumulative-Gaussian fit with a lapse rate, by binomial maximum likelihood.

    p(x) = lapse/2 + (1 - lapse) * Phi((x - mu) / sigma)

    mu is the boundary (the 50% point once lapses are accounted for) and sigma
    is inversely proportional to sensitivity. The lapse term matters here
    because motor errors (ErrorType 1 and 3) put a floor and a ceiling on the
    curve; without it those trials flatten the slope and bias mu.
    """
    x, n, k = np.asarray(x, float), np.asarray(n, float), np.asarray(k, float)
    if x.size < 3 or np.unique(x).size < 3:
        return None
    p_obs = k / np.maximum(n, 1)
    mu0 = float(np.interp(0.5, p_obs, x)) if np.all(np.diff(p_obs) >= 0) else float(np.mean(x))
    span = float(x.max() - x.min()) or 1.0
    best = None
    for s0 in (span / 8.0, span / 4.0, span / 2.0):
        theta0 = np.array([mu0, np.log(s0), -2.0])
        try:
            r = minimize(_neg_log_lik, theta0, args=(x, n, k), method="Nelder-Mead",
                         options={"maxiter": 4000, "xatol": 1e-6, "fatol": 1e-6})
        except Exception:
            continue
        if r.success or r.fun < np.inf:
            if best is None or r.fun < best.fun:
                best = r
    if best is None:
        return None
    mu, log_sigma, logit_lapse = best.x
    return {"mu": float(mu), "sigma": float(np.exp(np.clip(log_sigma, -20, 20))),
            "lapse": float(LAPSE_MAX / (1.0 + np.exp(-np.clip(logit_lapse, -30, 30)))),
            "neg_log_lik": float(best.fun), "n_trials": int(n.sum())}


def bootstrap_psychometric(x, n, k, n_boot=N_BOOTSTRAP, seed=PSYCHO_SEED):
    """Percentile CIs by resampling each stimulus level's binomial counts."""
    rng = np.random.default_rng(seed)
    x, n, k = np.asarray(x, float), np.asarray(n, int), np.asarray(k, int)
    mus, sigmas = [], []
    p_hat = k / np.maximum(n, 1)
    for _ in range(n_boot):
        kb = rng.binomial(n, np.clip(p_hat, 0, 1))
        f = fit_psychometric(x, n, kb)
        if f is not None and np.isfinite(f["mu"]):
            mus.append(f["mu"]); sigmas.append(f["sigma"])
    if not mus:
        return {}
    return {"mu_lo": float(np.percentile(mus, 2.5)), "mu_hi": float(np.percentile(mus, 97.5)),
            "sigma_lo": float(np.percentile(sigmas, 2.5)), "sigma_hi": float(np.percentile(sigmas, 97.5)),
            "n_boot_ok": len(mus)}


def fit_all_boundaries(df, n_boot=N_BOOTSTRAP, verbose=True):
    """Fit one curve per ordinal split; returns (fits DataFrame, counts table)."""
    tab = ordinal_choice_table(df)
    if tab.empty:
        print("No trials with a real choice; psychometric fit skipped.")
        return pd.DataFrame(), tab
    out = []
    for j, g in tab.groupby("split"):
        levels = g["x"].nunique()
        rec = {"split": j, "boundary": f"{CATEGORY_ORDER[j - 2]}|{CATEGORY_ORDER[j - 1]}",
               "n_levels": levels, "n_trials": int(g["n"].sum())}
        if levels < MIN_LEVELS_FOR_FIT:
            rec.update({"mu": np.nan, "sigma": np.nan, "lapse": np.nan,
                        "note": f"only {levels} stimulus levels; need >= {MIN_LEVELS_FOR_FIT}"})
            out.append(rec)
            continue
        f = fit_psychometric(g["x"], g["n"], g["k"])
        if f is None:
            rec.update({"mu": np.nan, "sigma": np.nan, "lapse": np.nan, "note": "fit failed"})
        else:
            rec.update(f)
            rec.update(bootstrap_psychometric(g["x"], g["n"], g["k"], n_boot=n_boot))
            rec["note"] = ""
        out.append(rec)
    fits = pd.DataFrame(out)
    if verbose:
        print("Psychometric fits (cumulative Gaussian with lapse):\n")
        cols = [c for c in ["boundary", "mu", "mu_lo", "mu_hi", "sigma", "sigma_lo",
                            "sigma_hi", "lapse", "n_levels", "n_trials", "note"]
                if c in fits.columns]
        print(fits[cols].round(4).to_string(index=False))
        print("\n  mu    = category boundary in deg VA (the 50% point, lapse-corrected)")
        print("  sigma = width of the transition; SMALLER = sharper discrimination")
        print("  lapse = stimulus-independent error rate (motor lapses, inattention)")
    return fits, tab


def plot_psychometric(fits, tab, stimulus_col=STIMULUS_COL):
    if tab.empty:
        return None
    splits = sorted(tab["split"].unique())
    fig, axes = plt.subplots(1, len(splits), figsize=(5.2 * len(splits), 4.2), squeeze=False)
    axes = axes.ravel()
    for ax, j in zip(axes, splits):
        g = tab[tab["split"] == j]
        row = fits[fits["split"] == j]
        se = np.sqrt(np.clip(g["p"] * (1 - g["p"]), 0, None) / np.maximum(g["n"], 1))
        ax.errorbar(g["x"], g["p"], yerr=1.96 * se, fmt="o", ms=6, capsize=3,
                    color="#31688e", label="observed")
        for _i, (_, r) in enumerate(g.sort_values("x").iterrows()):
            ax.annotate(f"{int(r['n'])}", (r["x"], r["p"]), textcoords="offset points",
                        xytext=(0, 8 if _i % 2 == 0 else 17), ha="center",
                        fontsize=6, color="grey")
        if not row.empty and np.isfinite(row["mu"].iloc[0]):
            mu, sig, lap = row["mu"].iloc[0], row["sigma"].iloc[0], row["lapse"].iloc[0]
            xx = np.linspace(g["x"].min() - 0.2, g["x"].max() + 0.2, 300)
            yy = lap / 2.0 + (1 - lap) * norm.cdf((xx - mu) / sig)
            ax.plot(xx, yy, "-", lw=2, color="#440154", label="fit")
            ax.axvline(mu, ls="--", lw=1.2, color="#fde725")
            lo, hi = row.get("mu_lo"), row.get("mu_hi")
            if lo is not None and np.isfinite(lo.iloc[0]):
                ax.axvspan(lo.iloc[0], hi.iloc[0], color="#fde725", alpha=0.25)
            ax.set_title(f"{row['boundary'].iloc[0]}\nmu={mu:.3f}  sigma={sig:.3f}  lapse={lap:.3f}",
                         fontsize=10)
            lo_cat, hi_cat = CATEGORY_ORDER[j - 2], CATEGORY_ORDER[j - 1]
            ax.axhspan(-0.03, 0.5, color=CATEGORY_COLORS.get(lo_cat, "#999999"), alpha=0.10)
            ax.axhspan(0.5, 1.03, color=CATEGORY_COLORS.get(hi_cat, "#999999"), alpha=0.10)
            ax.text(0.02, 0.06, lo_cat, transform=ax.transAxes, fontsize=8, color="#333333")
            ax.text(0.02, 0.92, hi_cat, transform=ax.transAxes, fontsize=8, color="#333333")
        else:
            ax.set_title(f"split {j}: not enough stimulus levels to fit", fontsize=10)
        ax.axhline(0.5, ls=":", lw=0.8, color="grey")
        ax.set_xlabel(f"{stimulus_col} (deg VA)")
        ax.set_ylabel(f"P(choose category >= {j})")
        ax.set_ylim(-0.03, 1.03)
        ax.legend(fontsize=8, loc="lower right")
    fig.suptitle("Psychometric functions: one ordinal split per category boundary", y=1.02)
    fig.tight_layout()
    fig.savefig("psychometric_functions.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


PSYCHO_SOURCE = ANALYSIS_DF if "ANALYSIS_DF" in globals() else FEATURES_DF
_modal_nc = None
if "NumCategories" in PSYCHO_SOURCE.columns and PSYCHO_SOURCE["NumCategories"].notna().any():
    _modal_nc = int(PSYCHO_SOURCE["NumCategories"].mode().iloc[0])
    _mixed = PSYCHO_SOURCE["NumCategories"].nunique() > 1
    if _mixed:
        print(f"Session mixes 2- and 3-category trials; fitting the modal regime "
              f"(NumCategories == {_modal_nc}) only, because the two regimes use "
              f"different length->category splits.")
    PSYCHO_SOURCE = PSYCHO_SOURCE[PSYCHO_SOURCE["NumCategories"] == _modal_nc]

PSYCHO_FITS, PSYCHO_TABLE = fit_all_boundaries(PSYCHO_SOURCE)
BOUNDARIES = [m for m in PSYCHO_FITS.get("mu", pd.Series(dtype=float)).tolist() if np.isfinite(m)]
plot_psychometric(PSYCHO_FITS, PSYCHO_TABLE)
plt.show()
if PSYCHO_FITS is not None and not PSYCHO_FITS.empty:
    PSYCHO_FITS.to_csv("psychometric_fits.csv", index=False)
    PSYCHO_TABLE.to_csv("psychometric_counts.csv", index=False)
print(f"\nBoundaries carried forward to the chronometric cell: "
      f"{[round(b, 3) for b in BOUNDARIES] if BOUNDARIES else 'none (fit unavailable)'}")

### 1.6. Chronometric functions

If the boundaries estimated above are real, they should show up in a variable
that was never used to estimate them: **time**. The classic prediction is that
responses slow down near a category boundary, where the evidence is most
ambiguous.

Difficulty here is the distance from each trial's bar length to the nearest
**fitted** boundary, so it reflects where this subject actually placed the
criterion rather than where the design assumed it. `TotalTime_s`,
`DecisionTime_s` and `ExecutionTime_s` are tested separately on purpose: the
decision half is where a difficulty effect belongs, and finding one in the
execution half instead would point at a motor confound rather than a
perceptual one.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

CHRONO_MEASURES = ["TotalTime_s", "DecisionTime_s", "ExecutionTime_s"]
NOMINAL_BOUNDARIES = [5.05, 6.40]
N_DISTANCE_BINS = 5


def boundary_distance(df, boundaries, stimulus_col=STIMULUS_COL):
    """Distance from each trial's bar length to the NEAREST category boundary.

    Small distance = an ambiguous stimulus sitting near the criterion; large
    distance = a prototypical one deep inside its category. This is the
    difficulty axis the chronometric and error analyses are built on, and it
    is derived from the fitted boundaries so it reflects where THIS subject
    actually put the criterion, not where the design assumed it.
    """
    x = pd.to_numeric(df[stimulus_col], errors="coerce").to_numpy(float)
    if not boundaries:
        return pd.Series(np.full(x.shape, np.nan), index=df.index)
    d = np.min(np.abs(x[:, None] - np.asarray(boundaries, float)[None, :]), axis=1)
    return pd.Series(d, index=df.index)


def chronometric_stats(df, measures=CHRONO_MEASURES, dist_col="boundary_dist"):
    rows = []
    for m in measures:
        if m not in df.columns:
            continue
        sub = df[[dist_col, m]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(sub) < 10 or sub[dist_col].nunique() < 3:
            rows.append({"measure": m, "n": len(sub), "spearman_rho": np.nan, "p": np.nan})
            continue
        rho, p = stats.spearmanr(sub[dist_col], sub[m])
        slope, intercept, r, p_lin, se = stats.linregress(sub[dist_col], sub[m])
        rows.append({"measure": m, "n": len(sub), "spearman_rho": rho, "p": p,
                     "slope_s_per_deg": slope, "slope_p": p_lin})
    return pd.DataFrame(rows)


def plot_chronometric(df, boundaries, measures=CHRONO_MEASURES,
                      dist_col="boundary_dist", stimulus_col=STIMULUS_COL,
                      n_bins=N_DISTANCE_BINS):
    measures = [m for m in measures if m in df.columns]
    if not measures:
        print("No timing columns available; chronometric plot skipped.")
        return None
    fig, axes = plt.subplots(2, len(measures), figsize=(4.8 * len(measures), 7.6), squeeze=False)
    d = df.copy()
    d["_bin"] = pd.qcut(d[dist_col], q=min(n_bins, d[dist_col].nunique()), duplicates="drop")
    for j, m in enumerate(measures):
        ax = axes[0][j]
        g = d.groupby(stimulus_col)[m].agg(["median", "count",
                                           lambda s: s.quantile(0.25),
                                           lambda s: s.quantile(0.75)])
        g.columns = ["median", "count", "q25", "q75"]
        g = g[g["count"] >= 3]
        ax.errorbar(g.index, g["median"],
                    yerr=[g["median"] - g["q25"], g["q75"] - g["median"]],
                    fmt="o-", ms=5, capsize=3, color="#31688e")
        for b in boundaries:
            ax.axvline(b, ls="--", lw=1.2, color="#fde725")
        ax.set_xlabel(f"{stimulus_col} (deg VA)")
        ax.set_ylabel(f"{m} (median, IQR)")
        ax.set_title(f"{m} by stimulus length\n(dashed = fitted boundary)", fontsize=9)

        ax = axes[1][j]
        gb = d.groupby("_bin", observed=True)[m].agg(["median", "count",
                                                      lambda s: s.quantile(0.25),
                                                      lambda s: s.quantile(0.75)])
        gb.columns = ["median", "count", "q25", "q75"]
        centres = [iv.mid for iv in gb.index]
        ax.errorbar(centres, gb["median"],
                    yerr=[gb["median"] - gb["q25"], gb["q75"] - gb["median"]],
                    fmt="s-", ms=6, capsize=3, color="#440154")
        ax.set_xlabel("distance to nearest boundary (deg VA)")
        ax.set_ylabel(f"{m} (median, IQR)")
        ax.set_title(f"{m} by difficulty\n(left = ambiguous, right = prototypical)", fontsize=9)
    fig.suptitle("Chronometric functions: does the subject slow down near the category boundary?", y=1.0)
    fig.tight_layout()
    fig.savefig("chronometric_functions.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


def plot_accuracy_by_difficulty(df, dist_col="boundary_dist", n_bins=N_DISTANCE_BINS):
    if "IsCorrect" not in df.columns:
        return None
    d = df[[dist_col, "IsCorrect"]].apply(pd.to_numeric, errors="coerce").dropna()
    if d.empty:
        return None
    d["_bin"] = pd.qcut(d[dist_col], q=min(n_bins, d[dist_col].nunique()), duplicates="drop")
    g = d.groupby("_bin", observed=True)["IsCorrect"].agg(["mean", "count"])
    se = np.sqrt(g["mean"] * (1 - g["mean"]) / g["count"])
    fig, ax = plt.subplots(figsize=(5.6, 4.0))
    ax.errorbar([iv.mid for iv in g.index], g["mean"], yerr=1.96 * se,
                fmt="o-", ms=6, capsize=3, color="#21918c")
    ax.axhline(1.0 / 3.0, color=GRAY_LIGHT, lw=1, label="chance (3 categories)")
    ax.set_xlabel("distance to nearest boundary (deg VA)")
    ax.set_ylabel("P(correct)")
    ax.set_ylim(0, 1.02)
    ax.set_title("Accuracy by difficulty")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("accuracy_by_difficulty.png", dpi=FIGURE_DPI, bbox_inches="tight")
    rho, p = stats.spearmanr(d[dist_col], d["IsCorrect"])
    print(f"\nAccuracy vs distance-to-boundary: Spearman rho = {rho:+.3f}, p = {p:.2g} "
          f"(positive = more accurate on prototypical lengths, as expected).")
    return fig


CHRONO_BOUNDARIES = BOUNDARIES if BOUNDARIES else NOMINAL_BOUNDARIES
if not BOUNDARIES:
    print(f"No fitted boundaries available; falling back to the DESIGN boundaries "
          f"{NOMINAL_BOUNDARIES} (midpoints of the 12-length category split). "
          f"Interpret the distance axis as nominal, not subject-specific.")

CHRONO_DF = (ANALYSIS_DF if "ANALYSIS_DF" in globals() else FEATURES_DF).copy()
CHRONO_DF["boundary_dist"] = boundary_distance(CHRONO_DF, CHRONO_BOUNDARIES)

CHRONO_STATS = chronometric_stats(CHRONO_DF)
print("Chronometric statistics (timing vs distance to the nearest boundary):\n")
print(CHRONO_STATS.round(4).to_string(index=False))
print("\n  A NEGATIVE rho is the classic result: the closer the stimulus sits to the")
print("  boundary, the LONGER the subject takes. A slope near zero says the extra")
print("  errors near the boundary are not bought with extra time.")

plot_chronometric(CHRONO_DF, CHRONO_BOUNDARIES)
plt.show()
plot_accuracy_by_difficulty(CHRONO_DF)
plt.show()
CHRONO_STATS.to_csv("chronometric_stats.csv", index=False)

### 1.7. Sequential effects

`PrevTrialCorrect` and `PrevTrialDirection` have been in `trial_data_*.csv` for
several versions without any analysis reading them. Three questions come free:

- **Post-error slowing** — does an error make the next response slower?
- **Post-error accuracy** — and does it make the next response better?
- **Direction perseveration** — win-stay / lose-shift on the response
  *position*, which is a motor bias with nothing to do with the category.

One caveat drives the implementation: `CenterOutTask.m` carries
`prevTrialCorrect` straight across block boundaries, so the first trial of a
block describes the last trial of the *previous* block. Those pairs are not
real trial-to-trial transitions, so the previous outcome is recomputed within
each block and the first trial of every block is dropped. The agreement rate
against the logged column is printed as a check on the CSV.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEQ_MEASURES = ["TotalTime_s", "DecisionTime_s", "ExecutionTime_s"]
TRIAL_ORDER_COLS = ["Block", "TrialNumInBlock", "Attempt"]


def add_previous_trial(df, order_cols=TRIAL_ORDER_COLS):
    """Recompute each trial's predecessor WITHIN its block.

    trial_data_*.csv already carries PrevTrialCorrect / PrevTrialDirection,
    but the task writes them as a running value that is never reset, so the
    first trial of a block inherits the last trial of the previous block.
    Recomputing here with a within-block shift drops those seams, and the
    agreement rate printed below is a check that the two definitions agree
    everywhere else (they should, on every non-first trial of a block).
    """
    d = df.sort_values(order_cols).copy()
    d["prev_correct"] = d.groupby("Block")["IsCorrect"].shift(1)
    if "DirectionChosen" in d.columns:
        d["prev_direction"] = d.groupby("Block")["DirectionChosen"].shift(1)
    if "ErrorType" in d.columns:
        d["prev_error_type"] = d.groupby("Block")["ErrorType"].shift(1)
    d["is_block_start"] = d["prev_correct"].isna()
    if "PrevTrialCorrect" in d.columns:
        both = d[~d["is_block_start"]]
        agree = (pd.to_numeric(both["PrevTrialCorrect"], errors="coerce")
                 == both["prev_correct"]).mean()
        print(f"Recomputed predecessor agrees with the logged PrevTrialCorrect on "
              f"{100 * agree:.1f}% of non-block-start trials "
              f"({int(d['is_block_start'].sum())} block-start trials excluded).")
    return d


def post_error_slowing(df, measures=SEQ_MEASURES):
    """Is the subject slower on the trial AFTER an error than after a success?"""
    rows = []
    for m in measures:
        if m not in df.columns:
            continue
        sub = df[df["prev_correct"].notna()][[m, "prev_correct"]].copy()
        sub[m] = pd.to_numeric(sub[m], errors="coerce")
        sub = sub.dropna()
        after_err = sub.loc[sub["prev_correct"] == 0, m].to_numpy()
        after_ok = sub.loc[sub["prev_correct"] == 1, m].to_numpy()
        if after_err.size < 5 or after_ok.size < 5:
            rows.append({"measure": m, "n_after_error": after_err.size,
                         "n_after_correct": after_ok.size, "U": np.nan, "p": np.nan})
            continue
        U, p = stats.mannwhitneyu(after_err, after_ok, alternative="two-sided")
        rank_biserial = 1.0 - 2.0 * U / (after_err.size * after_ok.size)
        rows.append({
            "measure": m,
            "median_after_error": float(np.median(after_err)),
            "median_after_correct": float(np.median(after_ok)),
            "slowing_s": float(np.median(after_err) - np.median(after_ok)),
            "n_after_error": after_err.size, "n_after_correct": after_ok.size,
            "U": float(U), "p": float(p), "rank_biserial": float(-rank_biserial)})
    return pd.DataFrame(rows)


def post_error_accuracy(df):
    sub = df[df["prev_correct"].notna()]
    tab = pd.crosstab(sub["prev_correct"], sub["IsCorrect"])
    if tab.shape != (2, 2):
        return tab, np.nan, np.nan
    chi2, p, _, _ = stats.chi2_contingency(tab)
    acc_after_err = tab.loc[0, 1] / tab.loc[0].sum()
    acc_after_ok = tab.loc[1, 1] / tab.loc[1].sum()
    print(f"\nAccuracy after an ERROR:   {acc_after_err:.3f}  (n={int(tab.loc[0].sum())})")
    print(f"Accuracy after a CORRECT:  {acc_after_ok:.3f}  (n={int(tab.loc[1].sum())})")
    print(f"chi2 = {chi2:.3f}, p = {p:.3g}")
    return tab, chi2, p


def win_stay_lose_shift(df):
    """Does the subject repeat the previous DIRECTION more after a win than a loss?

    Direction is orthogonal to the category rule here: the correct target
    position is randomised independently of bar length, so any dependence on
    the previous trial's direction is a genuine sequential bias rather than
    something the task rewards.
    """
    if "prev_direction" not in df.columns or "DirectionChosen" not in df.columns:
        return None
    sub = df[df["prev_direction"].notna() & df["DirectionChosen"].notna()].copy()
    sub = sub[(sub["DirectionChosen"] != "None") & (sub["prev_direction"] != "None")]
    if sub.empty:
        print("\nNo usable direction pairs for win-stay/lose-shift.")
        return None
    sub["repeated"] = (sub["DirectionChosen"] == sub["prev_direction"]).astype(int)
    tab = pd.crosstab(sub["prev_correct"], sub["repeated"])
    n_dirs = sub["DirectionChosen"].nunique()
    chance = 1.0 / n_dirs if n_dirs else np.nan
    print(f"\nDirection repetition (chance = {chance:.3f} with {n_dirs} positions):")
    for lvl, label in [(1.0, "after CORRECT"), (0.0, "after ERROR")]:
        s = sub[sub["prev_correct"] == lvl]["repeated"]
        if len(s):
            print(f"  P(repeat) {label:15s} = {s.mean():.3f}  (n={len(s)})")
    if tab.shape == (2, 2):
        chi2, p, _, _ = stats.chi2_contingency(tab)
        print(f"  win-stay/lose-shift difference: chi2 = {chi2:.3f}, p = {p:.3g}")
    return sub


def plot_sequential(df, pes, measures=SEQ_MEASURES):
    measures = [m for m in measures if m in df.columns]
    fig, axes = plt.subplots(1, len(measures) + 1, figsize=(4.4 * (len(measures) + 1), 4.0),
                             squeeze=False)
    axes = axes.ravel()
    for ax, m in zip(axes, measures):
        sub = df[df["prev_correct"].notna()].copy()
        sub[m] = pd.to_numeric(sub[m], errors="coerce")
        sub = sub.dropna(subset=[m])
        data = [sub.loc[sub["prev_correct"] == 1, m], sub.loc[sub["prev_correct"] == 0, m]]
        ax.boxplot(data, showfliers=False, medianprops=dict(color="#fde725", lw=2))
        ax.set_xticks([1, 2])
        ax.set_xticklabels(["after\ncorrect", "after\nerror"])
        row = pes[pes["measure"] == m]
        ttl = f"{m}"
        if not row.empty and np.isfinite(row["p"].iloc[0]):
            ttl += f"\nMWU p={row['p'].iloc[0]:.3g}, rb={row['rank_biserial'].iloc[0]:+.2f}"
        ax.set_title(ttl, fontsize=9)
        ax.set_ylabel(m, fontsize=8)
    ax = axes[len(measures)]
    sub = df[df["prev_correct"].notna()]
    acc = sub.groupby("prev_correct")["IsCorrect"].agg(["mean", "count"])
    se = np.sqrt(acc["mean"] * (1 - acc["mean"]) / acc["count"])
    ax.bar(["after\ncorrect", "after\nerror"],
           [acc["mean"].get(1.0, np.nan), acc["mean"].get(0.0, np.nan)],
           yerr=[1.96 * se.get(1.0, 0), 1.96 * se.get(0.0, 0)],
           color=["#21918c", "#440154"], capsize=4)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("P(correct)")
    ax.set_title("Accuracy by previous outcome", fontsize=9)
    fig.suptitle("Sequential effects: does the previous trial change this one?", y=1.02)
    fig.tight_layout()
    fig.savefig("sequential_effects.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


SEQ_DF = add_previous_trial(ANALYSIS_DF if "ANALYSIS_DF" in globals() else FEATURES_DF)
POST_ERROR = post_error_slowing(SEQ_DF)
print("\nPost-error slowing (Mann-Whitney U; positive slowing_s = slower after an error):\n")
print(POST_ERROR.round(4).to_string(index=False))
print("\n  rank_biserial is the effect size: +1 = always slower after an error,")
print("  0 = no difference, -1 = always faster.")
POST_ERROR_TAB, _, _ = post_error_accuracy(SEQ_DF)
WSLS_DF = win_stay_lose_shift(SEQ_DF)
plot_sequential(SEQ_DF, POST_ERROR)
plt.show()
POST_ERROR.to_csv("post_error_slowing.csv", index=False)

### 1.8. Error taxonomy and foil events

`IsCorrect` collapses three very different failures into one bit. The MATLAB
side already distinguishes them:

| `ErrorType` | meaning |
|---|---|
| 0 | correct |
| 1 | early exit / timeout, no flash |
| 2 | wrong target chosen |
| 3 | hold-break on the correct target (strict pre-training) |

Only **type 2** is a category error — a target was chosen and it was the wrong
one. Types 1 and 3 are motor or attention failures where no category was ever
committed to. The separation matters: perceptual errors should concentrate near
the boundary while motor errors should be flat against difficulty, and pooling
them is what makes a lapse look like a discrimination failure.

`foil_events_*.csv` is also read here for the first time. It logs **every**
distractor touch during forgiving pre-training, so the number of foil touches
before the correct target is a graded measure of uncertainty that a
one-row-per-trial file cannot express.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

ERROR_TYPE_LABELS = {
    0: "0 correct",
    1: "1 early exit / timeout (motor)",
    2: "2 wrong target (perceptual)",
    3: "3 hold break, strict (motor)",
}
PERCEPTUAL_ERRORS = {2}
MOTOR_ERRORS = {1, 3}


def error_taxonomy(df):
    """Split errors into PERCEPTUAL and MOTOR, which IsCorrect alone conflates.

    ErrorType 2 means the subject reached a target of the wrong category: a
    genuine categorisation error, and the only kind the psychometric curve is
    about. ErrorType 1 and 3 are failures to hold or to leave in time -- the
    subject may well have known the answer. Pooling them inflates the lapse
    rate and flattens the psychometric slope, which is why the fit in the
    psychometric cell carries an explicit lapse term.
    """
    d = df.copy()
    d["ErrorType"] = pd.to_numeric(d["ErrorType"], errors="coerce")
    d["error_class"] = np.where(d["ErrorType"] == 0, "correct",
                        np.where(d["ErrorType"].isin(PERCEPTUAL_ERRORS), "perceptual",
                        np.where(d["ErrorType"].isin(MOTOR_ERRORS), "motor", "other")))
    counts = d["ErrorType"].value_counts().sort_index()
    tab = pd.DataFrame({
        "ErrorType": counts.index.astype(int),
        "label": [ERROR_TYPE_LABELS.get(int(i), f"{int(i)} unknown") for i in counts.index],
        "n": counts.values,
        "pct": (100.0 * counts.values / counts.sum()).round(2)})
    print("Trial outcomes by ErrorType:\n")
    print(tab.to_string(index=False))
    print("\nError class breakdown:")
    print(d["error_class"].value_counts().to_string())
    return d, tab


def confusion_by_category(df, truth_col="StimulusGroup", choice_col="ChosenTarget"):
    """True category x chosen category, restricted to trials with a real choice."""
    d = df[pd.to_numeric(df[choice_col], errors="coerce") >= 1].copy()
    if d.empty:
        return None
    d["_chosen"] = pd.to_numeric(d[choice_col], errors="coerce").astype(int).map(
        {i + 1: n for i, n in enumerate(CATEGORY_ORDER)})
    order = [c for c in CATEGORY_ORDER if c in set(d[truth_col]) | set(d["_chosen"].dropna())]
    cm = pd.crosstab(d[truth_col], d["_chosen"]).reindex(index=order, columns=order, fill_value=0)
    cmn = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0)
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="viridis", square=True, ax=axes[0],
                cbar_kws={"shrink": 0.8})
    axes[0].set_title("Confusion matrix (counts)")
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, square=True,
                ax=axes[1], cbar_kws={"shrink": 0.8})
    axes[1].set_title("Row-normalized: P(chosen | true)")
    for ax in axes:
        ax.set_xlabel("chosen category")
        ax.set_ylabel("true category")
        for tick in ax.get_xticklabels():
            tick.set_color(CATEGORY_COLORS.get(tick.get_text(), "#333333"))
        for tick in ax.get_yticklabels():
            tick.set_color(CATEGORY_COLORS.get(tick.get_text(), "#333333"))
    fig.suptitle("Where do the categorisation errors go?", y=1.02)
    fig.tight_layout()
    fig.savefig("category_confusion.png", dpi=FIGURE_DPI, bbox_inches="tight")
    off = cm.to_numpy().copy()
    np.fill_diagonal(off, 0)
    if off.sum():
        adjacent = sum(off[i, j] for i in range(off.shape[0]) for j in range(off.shape[1])
                       if abs(i - j) == 1)
        print(f"\nOf {int(off.sum())} categorisation errors, {adjacent} "
              f"({100.0 * adjacent / off.sum():.1f}%) land on an ADJACENT category. "
              f"A high share is the signature of a graded perceptual boundary rather "
              f"than random guessing.")
    return cm


def error_class_by_difficulty(df, dist_col="boundary_dist", n_bins=5):
    """Perceptual errors should concentrate near the boundary; motor ones should not."""
    if dist_col not in df.columns:
        print(f"'{dist_col}' missing; run the chronometric cell first.")
        return None
    d = df[[dist_col, "error_class"]].dropna()
    if d.empty or d[dist_col].nunique() < 3:
        return None
    d = d.copy()
    d["_bin"] = pd.qcut(d[dist_col], q=min(n_bins, d[dist_col].nunique()), duplicates="drop")
    rate = (d.groupby(["_bin", "error_class"], observed=True).size()
              .unstack(fill_value=0))
    rate = rate.div(rate.sum(axis=1), axis=0)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    centres = [iv.mid for iv in rate.index]
    for cls, colour in [("perceptual", "#440154"), ("motor", "#21918c")]:
        if cls in rate.columns:
            ax.plot(centres, rate[cls], "o-", ms=6, color=colour, label=cls)
    ax.set_xlabel("distance to nearest boundary (deg VA)")
    ax.set_ylabel("share of trials")
    ax.set_title("Error class by difficulty\n(perceptual should fall with distance; motor should be flat)",
                 fontsize=9)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("error_class_by_difficulty.png", dpi=FIGURE_DPI, bbox_inches="tight")
    for cls in ("perceptual", "motor"):
        sub = df[["error_class", dist_col]].dropna()
        if cls in set(sub["error_class"]):
            y = (sub["error_class"] == cls).astype(int)
            rho, p = stats.spearmanr(sub[dist_col], y)
            print(f"  {cls:11s} vs distance-to-boundary: rho = {rho:+.3f}, p = {p:.3g}")
    return rate


def foil_analysis(df, foil_path=None, id_cols=("Block", "TrialNumInBlock", "Attempt")):
    """Foil touches per trial: a graded measure of uncertainty.

    foil_events_*.csv is only written when the forgiving pre-training mode is
    on, where touching a distractor neither aborts the trial nor counts as an
    error. The number of distractors touched before reaching the correct
    target therefore grades hesitancy on a scale the binary IsCorrect cannot.
    """
    foil_path = foil_path or globals().get("FOIL_EVENTS_PATH")
    if not foil_path or not Path(foil_path).exists():
        print("\nNo foil_events_*.csv for this session (forgiving pre-training only); "
              "foil analysis skipped.")
        return None
    fo = pd.read_csv(foil_path)
    fo = fo.dropna(subset=[c for c in id_cols if c in fo.columns])
    if fo.empty:
        print("\nfoil_events file is empty; no distractor touches were logged.")
        return None
    id_cols = [c for c in id_cols if c in fo.columns and c in df.columns]
    for c in id_cols:
        fo[c] = pd.to_numeric(fo[c], errors="coerce")
    per_trial = (fo.groupby(id_cols).agg(n_foils=("FoilDirection", "size"),
                                         first_foil_s=("TimeSinceTargetOnset_s", "min"))
                   .reset_index())
    merged = df.merge(per_trial, on=id_cols, how="left")
    merged["n_foils"] = merged["n_foils"].fillna(0)
    print(f"\nFoil touches: {int(fo.shape[0])} events over "
          f"{per_trial.shape[0]} trials ({100.0 * per_trial.shape[0] / len(df):.1f}% of trials "
          f"had at least one).")
    print(f"Mean foils per trial: {merged['n_foils'].mean():.3f}")
    if "boundary_dist" in merged.columns:
        sub = merged[["boundary_dist", "n_foils"]].dropna()
        if len(sub) > 10 and sub["boundary_dist"].nunique() > 2:
            rho, p = stats.spearmanr(sub["boundary_dist"], sub["n_foils"])
            print(f"Foils vs distance-to-boundary: rho = {rho:+.3f}, p = {p:.3g} "
                  f"(negative = more hesitation on ambiguous lengths).")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
    axes[0].hist(merged["n_foils"], bins=np.arange(-0.5, merged["n_foils"].max() + 1.5),
                 color="#31688e", edgecolor="white")
    axes[0].set_xlabel("distractor touches in a trial")
    axes[0].set_ylabel("trials")
    axes[0].set_title("Distribution of foil touches")
    if "FoilDirection" in fo.columns:
        fo["FoilDirection"].value_counts().plot(kind="bar", ax=axes[1], color="#440154")
        axes[1].set_title("Which distractor position gets touched")
        axes[1].set_ylabel("events")
        axes[1].tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig("foil_events.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return merged


ERROR_SOURCE = (FEATURES_DF if "FEATURES_DF" in globals() else ANALYSIS_DF).copy()
ERROR_SOURCE["boundary_dist"] = boundary_distance(
    ERROR_SOURCE, CHRONO_BOUNDARIES if "CHRONO_BOUNDARIES" in globals() else NOMINAL_BOUNDARIES)
print("Running on the UNFILTERED trial table: drop_early_exits() removes exactly "
      "the ErrorType == 1 trials (no target entered), which are the motor errors "
      "this cell exists to separate out. Every other cell keeps using ANALYSIS_DF.\n")

ERROR_DF, ERROR_TABLE = error_taxonomy(ERROR_SOURCE)
CONFUSION = confusion_by_category(ERROR_DF)
plt.show()
print("\nError class vs difficulty:")
ERROR_BY_DIFFICULTY = error_class_by_difficulty(ERROR_DF)
plt.show()
FOIL_DF = foil_analysis(ERROR_DF)
plt.show()
ERROR_TABLE.to_csv("error_taxonomy.csv", index=False)

### 1.9. Trajectory-based changes of mind

The notebook already computes clean, filtered trajectories, then reduces them
to summary scalars. This cell uses their *shape*.

For each trial it takes the cursor's initial heading over the first 120 ms
after the movement takeoff (`move_takeoff_ms`: walking back from the speed
peak, the first sample still above 5 % of that peak), finds which of the four
targets that heading points at,
and compares it against the target finally chosen. When they differ, the reach
set off toward one category and ended at another — a decision that was still
being made after the movement began.

The takeoff is the anchor, rather than target onset or the exit from the
centre circle. Before takeoff the cursor is at rest in the centre, where the
direction of the residual jitter means nothing. The exit from the centre
circle comes too late: the cursor is already 100 px out, and with an
execution time near 170 ms (median of the 31 August control session) a 120 ms
window starting there covers most of the
path to the final target, so the measured heading would mostly reflect the
final choice. `straightness` serves as an independent cross-check, since a curved
path is exactly what a mid-flight correction should produce.

This is the analysis that most justifies recording trajectories at all: the
final choice alone throws this information away.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

INITIAL_HEADING_MS = 120.0
MIN_HEADING_PX = 12.0
COM_EPOCHS = (DECISION_EPOCH, MOVEMENT_EPOCH)
DIRECTION_UNIT = {"Right_0": (1.0, 0.0), "Up_90": (0.0, -1.0),
                  "Left_180": (-1.0, 0.0), "Down_270": (0.0, 1.0)}


def initial_heading(result, window_ms=INITIAL_HEADING_MS, min_px=MIN_HEADING_PX):
    """Unit vector of the FIRST part of the reach, measured from movement takeoff.

    Screen coordinates, so y grows downward: Up_90 sits at -y, which is why
    DIRECTION_UNIT encodes it that way and matches estimate_screen_layout's
    offsets. Returns (None, nan) when the cursor has not travelled far enough
    in the window for the direction to mean anything.
    """
    gt, gxy = result.grid_time_ms, result.grid_xy
    if gt.size < 3:
        return None, np.nan
    t0 = result.move_takeoff_ms if np.isfinite(result.move_takeoff_ms) else gt[0]
    mask = (gt >= t0) & (gt <= t0 + window_ms)
    if mask.sum() < 2:
        return None, np.nan
    seg = gxy[mask]
    v = seg[-1] - seg[0]
    dist = float(np.hypot(v[0], v[1]))
    if dist < min_px:
        return None, dist
    return (v / dist), dist


def nearest_direction(unit):
    """Which of the four target positions the initial heading points at."""
    if unit is None:
        return None, np.nan
    best, best_dot = None, -np.inf
    for name, u in DIRECTION_UNIT.items():
        dot = float(unit[0] * u[0] + unit[1] * u[1])
        if dot > best_dot:
            best, best_dot = name, dot
    angle = float(np.degrees(np.arccos(np.clip(best_dot, -1, 1))))
    return best, angle


def build_change_of_mind(traj_path, trial_data_path, kinematics_path=None,
                         move_epochs=COM_EPOCHS, window_ms=INITIAL_HEADING_MS):
    """Per-trial initial heading vs the target actually chosen.

    The logic is the mouse-tracking one: if the hand commits to a direction
    before the categorisation has settled, an ambiguous stimulus should show
    up as a reach that STARTS toward one target and ends at another. That
    makes a change of mind a directly observable event rather than something
    inferred from response time alone.
    """
    dataset = TrajectoryDataset(traj_path, move_epochs=move_epochs)
    processor = TrajectoryProcessor.for_profile(dataset.profile, grid_dt_s=GRID_DT_S,
                                                cutoff_hz=CUTOFF_HZ, cm_per_px=CM_PER_PX)
    info_table = TrialInfoTable(trial_data_path, kinematics_path=kinematics_path)
    recs = []
    for tr in dataset.trials():
        r = processor.process(tr.time_ms, tr.x, tr.y, tr.move_window_ms, tr.hold_start_ms)
        unit, dist = initial_heading(r, window_ms=window_ms)
        init_dir, angle = nearest_direction(unit)
        info = info_table.lookup(tr.block, tr.trial, tr.attempt)
        recs.append({"Block": tr.block, "TrialNumInBlock": tr.trial, "Attempt": tr.attempt,
                     "initial_direction": init_dir, "initial_heading_px": dist,
                     "initial_offaxis_deg": angle})
    out = pd.DataFrame.from_records(recs)
    print(f"Initial heading resolved on {out['initial_direction'].notna().sum()}/{len(out)} trials "
          f"(the rest moved less than {MIN_HEADING_PX:g} px in the first {window_ms:g} ms).")
    return out


def annotate_change_of_mind(df, com):
    d = df.merge(com, on=[c for c in ID_COLS if c in com.columns], how="left")
    have = d["initial_direction"].notna() & d["DirectionChosen"].notna() & (d["DirectionChosen"] != "None")
    d["changed_mind"] = np.where(have, (d["initial_direction"] != d["DirectionChosen"]).astype(float), np.nan)
    n = int(d["changed_mind"].notna().sum())
    rate = d["changed_mind"].mean()
    print(f"Change-of-mind rate: {rate:.3f} over {n} trials with both an initial "
          f"heading and a chosen target.")
    return d


def change_of_mind_stats(d, dist_col="boundary_dist", n_bins=5):
    rows = []
    sub = d[["changed_mind", dist_col]].dropna() if dist_col in d.columns else pd.DataFrame()
    if len(sub) > 10 and sub[dist_col].nunique() > 2:
        rho, p = stats.spearmanr(sub[dist_col], sub["changed_mind"])
        print(f"\nChange of mind vs distance-to-boundary: rho = {rho:+.3f}, p = {p:.3g}")
        print("  A NEGATIVE rho is the prediction: ambiguous lengths (near the boundary)")
        print("  produce more reaches that start toward one target and end at another.")
        rows.append({"test": "changed_mind ~ boundary_dist", "rho": rho, "p": p, "n": len(sub)})
    for col in ("straightness", "TotalTime_s", "DecisionTime_s"):
        if col in d.columns:
            s = d[["changed_mind", col]].apply(pd.to_numeric, errors="coerce").dropna()
            if len(s) > 10:
                a = s.loc[s["changed_mind"] == 1, col]
                b = s.loc[s["changed_mind"] == 0, col]
                if len(a) >= 5 and len(b) >= 5:
                    U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
                    print(f"  {col:16s} changed={a.median():.3f} vs kept={b.median():.3f}  "
                          f"(MWU p={p:.3g})")
                    rows.append({"test": f"{col} | changed vs kept", "rho": np.nan,
                                 "p": p, "n": len(s)})
    if "IsCorrect" in d.columns:
        s = d[["changed_mind", "IsCorrect"]].dropna()
        if len(s) > 10:
            acc_c = s.loc[s["changed_mind"] == 1, "IsCorrect"].mean()
            acc_k = s.loc[s["changed_mind"] == 0, "IsCorrect"].mean()
            print(f"  accuracy: changed = {acc_c:.3f}, kept = {acc_k:.3f}")
    return pd.DataFrame(rows)


def plot_change_of_mind(d, dist_col="boundary_dist", n_bins=5):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    ax = axes[0]
    if dist_col in d.columns:
        s = d[["changed_mind", dist_col]].dropna()
        if len(s) and s[dist_col].nunique() > 2:
            s = s.copy()
            s["_bin"] = pd.qcut(s[dist_col], q=min(n_bins, s[dist_col].nunique()), duplicates="drop")
            g = s.groupby("_bin", observed=True)["changed_mind"].agg(["mean", "count"])
            se = np.sqrt(g["mean"] * (1 - g["mean"]) / g["count"])
            ax.errorbar([iv.mid for iv in g.index], g["mean"], yerr=1.96 * se,
                        fmt="o-", ms=6, capsize=3, color="#440154")
    ax.set_xlabel("distance to nearest boundary (deg VA)")
    ax.set_ylabel("P(change of mind)")
    ax.set_title("Changes of mind by difficulty", fontsize=9)

    ax = axes[1]
    s = d[["initial_offaxis_deg", "changed_mind"]].dropna()
    if len(s):
        ax.hist([s.loc[s["changed_mind"] == 0, "initial_offaxis_deg"],
                 s.loc[s["changed_mind"] == 1, "initial_offaxis_deg"]],
                bins=18, stacked=True, color=["#21918c", "#440154"],
                label=["kept", "changed"], edgecolor="white")
        ax.legend(fontsize=8)
    ax.set_xlabel("angle between initial heading and nearest target (deg)")
    ax.set_ylabel("trials")
    ax.set_title("How committed is the initial reach?", fontsize=9)

    ax = axes[2]
    if "straightness" in d.columns:
        s = d[["straightness", "changed_mind"]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(s):
            ax.boxplot([s.loc[s["changed_mind"] == 0, "straightness"],
                        s.loc[s["changed_mind"] == 1, "straightness"]], showfliers=False,
                       medianprops=dict(color="#fde725", lw=2))
            ax.set_xticks([1, 2])
            ax.set_xticklabels(["kept", "changed"])
    ax.set_ylabel("straightness")
    ax.set_title("Path straightness by change of mind", fontsize=9)
    fig.suptitle("Trajectory as a window on the decision: reaches that start one way and end another",
                 y=1.02)
    fig.tight_layout()
    fig.savefig("change_of_mind.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


COM_TRAJ = globals().get("TRAJ_FULL_PATH") or globals().get("TRAJ_PATH")
COM_RAW = build_change_of_mind(COM_TRAJ, TRIAL_DATA_PATH,
                               kinematics_path=globals().get("KINEMATICS_PATH"))
COM_DF = annotate_change_of_mind(CHRONO_DF if "CHRONO_DF" in globals() else ANALYSIS_DF, COM_RAW)
COM_STATS = change_of_mind_stats(COM_DF)
plot_change_of_mind(COM_DF)
plt.show()
COM_DF.to_csv("trial_features_change_of_mind.csv", index=False)

### 1.10. Supervised decoding

The K-means cell above asks whether the kinematics fall into natural clusters,
then checks after the fact whether those clusters line up with the category.
That is a weak test — an unsupervised split can be driven by anything.

This asks the question directly: **given the movement, can the category be
predicted?** Logistic regression and LDA, stratified cross-validation, balanced
accuracy, and a permutation test that establishes chance level for *this*
sample size and class balance rather than assuming `1/n_classes`.

`BarSizeVA_deg` is held out when decoding `StimulusGroup`, since the category
is a deterministic function of the bar length and leaving it in would score a
lookup table instead of the question being asked. The adjusted Rand index of
the K-means solution is printed alongside for comparison.

A score sitting on the permutation null is a real result too, and one the
cross-tabulation above cannot establish in either direction.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import (StratifiedKFold, cross_val_predict,
                                     permutation_test_score)
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.inspection import permutation_importance
from sklearn.metrics import adjusted_rand_score

DECODE_TARGETS = ["StimulusGroup", "IsCorrect"]
DECODE_PREFERRED = ["peak_vel_cm", "mean_vel_cm", "median_vel_cm",
                    "peak_accel_cm", "mean_accel_cm", "median_accel_cm",
                    "path_length_cm", "straightness", "move_takeoff_ms",
                    "n_samples", "fs_hz", "hold_dur_ms", "hold_max_speed_cm",
                    "hold_excursion_px", "initial_heading_px", "initial_offaxis_deg"]
DECODE_EXCLUDE = ["BarSizeVA_deg", "DecisionTime_s", "ExecutionTime_s", "TotalTime_s",
                  "boundary_dist", "ChosenTarget", "ErrorType"]
N_SPLITS = 5
N_PERMUTATIONS = 200
DECODE_SEED = 0


def decode_feature_columns(df, exclude=DECODE_EXCLUDE):
    """Kinematic columns only.

    BarSizeVA_deg is excluded because it IS the stimulus: decoding the category
    from the length the category is defined by would be circular. The timing
    columns are excluded too, so a significant score means the MOVEMENT itself
    carries the information, not the latency. Drop them from the list below if
    you want the easier, less interesting question answered instead.
    """
    base = [c for c in DECODE_PREFERRED if c in df.columns]
    if not base:
        base = globals().get("PCA_FEATURES") or globals().get("KINEMATIC_FEATURES") or []
    cols = [c for c in base if c in df.columns and c not in exclude
            and pd.to_numeric(df[c], errors="coerce").nunique(dropna=True) > 1]
    if not cols:
        num = df.select_dtypes("number")
        cols = [c for c in num.columns
                if c not in exclude and c not in ID_COLS and num[c].nunique(dropna=True) > 1]
    return cols


def decode_target(df, target, features, n_splits=N_SPLITS, n_perm=N_PERMUTATIONS,
                  seed=DECODE_SEED):
    """Cross-validated decoding with a permutation test for significance.

    Balanced accuracy, so an unbalanced session cannot look good by always
    answering with the majority class. The permutation test rebuilds the null
    by shuffling the labels, which is the honest reference here: with this
    many features and this few trials, chance is not exactly 1/n_classes.
    """
    d = df[features + [target]].dropna()
    if d.empty or d[target].nunique() < 2:
        print(f"  {target}: not enough data or only one class; skipped.")
        return None
    counts = d[target].value_counts()
    if counts.min() < n_splits:
        n_splits = max(2, int(counts.min()))
        print(f"  {target}: smallest class has {counts.min()} trials; using {n_splits}-fold CV.")
    X = d[features].to_numpy(float)
    y = d[target].to_numpy()
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = {"target": target, "n": len(d), "n_classes": int(d[target].nunique()),
           "n_features": len(features)}
    for name, model in [("logistic", LogisticRegression(max_iter=5000)),
                        ("lda", LinearDiscriminantAnalysis())]:
        pipe = make_pipeline(StandardScaler(), model)
        score, perm_scores, p = permutation_test_score(
            pipe, X, y, scoring="balanced_accuracy", cv=cv,
            n_permutations=n_perm, random_state=seed, n_jobs=1)
        out[f"{name}_balanced_acc"] = float(score)
        out[f"{name}_null_mean"] = float(np.mean(perm_scores))
        out[f"{name}_p"] = float(p)
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    y_pred = cross_val_predict(pipe, X, y, cv=cv)
    out["_confusion"] = confusion_matrix(y, y_pred, labels=sorted(pd.unique(y)))
    out["_labels"] = sorted(pd.unique(y))
    out["_y"], out["_y_pred"] = y, y_pred
    pipe.fit(X, y)
    imp = permutation_importance(pipe, X, y, n_repeats=20, random_state=seed,
                                 scoring="balanced_accuracy")
    out["_importance"] = pd.DataFrame({"feature": features,
                                       "importance": imp.importances_mean,
                                       "sd": imp.importances_std}
                                      ).sort_values("importance", ascending=False)
    return out


def plot_decoding(results):
    live = [r for r in results if r]
    if not live:
        return None
    fig, axes = plt.subplots(2, len(live), figsize=(5.6 * len(live), 8.4), squeeze=False)
    for j, r in enumerate(live):
        ax = axes[0][j]
        cm = r["_confusion"]
        cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        sns.heatmap(cmn, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1,
                    xticklabels=r["_labels"], yticklabels=r["_labels"], square=True, ax=ax,
                    cbar_kws={"shrink": 0.8})
        ax.set_title(f"{r['target']}: cross-validated confusion\n"
                     f"balanced acc = {r['logistic_balanced_acc']:.3f} "
                     f"(null {r['logistic_null_mean']:.3f}, p = {r['logistic_p']:.3g})",
                     fontsize=9)
        ax.set_xlabel("predicted")
        ax.set_ylabel("true")
        ax = axes[1][j]
        imp = r["_importance"].head(12).iloc[::-1]
        ax.barh(imp["feature"], imp["importance"], xerr=imp["sd"], color="#31688e")
        ax.axvline(0, color="grey", lw=0.8)
        ax.set_xlabel("permutation importance (drop in balanced accuracy)")
        ax.set_title(f"{r['target']}: which kinematics carry it", fontsize=9)
        ax.tick_params(labelsize=7)
    fig.suptitle("Supervised decoding from movement kinematics alone", y=1.0)
    fig.tight_layout()
    fig.savefig("decoding.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


DECODE_DF = COM_DF if "COM_DF" in globals() else (ANALYSIS_DF if "ANALYSIS_DF" in globals() else FEATURES_DF)
DECODE_FEATURES = decode_feature_columns(DECODE_DF)
print(f"Decoding from {len(DECODE_FEATURES)} kinematic features: {DECODE_FEATURES}\n")
print(f"Stimulus and timing columns are held out on purpose (see the docstring): "
      f"{[c for c in DECODE_EXCLUDE if c in DECODE_DF.columns]}\n")

DECODE_RESULTS = []
for _t in DECODE_TARGETS:
    if _t in DECODE_DF.columns:
        print(f"Decoding {_t} ...")
        DECODE_RESULTS.append(decode_target(DECODE_DF, _t, DECODE_FEATURES))

DECODE_SUMMARY = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")}
                               for r in DECODE_RESULTS if r])
if not DECODE_SUMMARY.empty:
    print("\nDecoding summary (balanced accuracy, permutation-tested):\n")
    print(DECODE_SUMMARY.round(4).to_string(index=False))
    print("\n  A score close to null_mean with a large p means the kinematics do NOT")
    print("  carry that variable -- which is itself a result worth reporting, and one")
    print("  the K-means cross-tab above cannot establish either way.")
    DECODE_SUMMARY.to_csv("decoding_summary.csv", index=False)

if "CLUSTERED_DF" in globals() and "cluster" in CLUSTERED_DF.columns:
    _c = CLUSTERED_DF.dropna(subset=["cluster", "StimulusGroup"])
    if len(_c):
        ari = adjusted_rand_score(_c["StimulusGroup"], _c["cluster"])
        print(f"\nUnsupervised reference: K-means clusters vs StimulusGroup, "
              f"adjusted Rand index = {ari:.4f} (0 = chance agreement). Compare against "
              f"the supervised balanced accuracy above.")

plot_decoding(DECODE_RESULTS)
plt.show()